In [ ]:
VERSION        = "2.5+FIX-K1..K13"
PROTOCOL_LABEL = "P11.13-Rev7.4.0"
MODEL_LABEL    = "V50.6.6"
# ====================================================================
# EC-CODE v2.5 + FIX-K1..K13
# MODEL_LABEL = V50.6.6 | PROTOCOL = P11.13-Rev7.4.0
# ====================================================================

import os, json, re, warnings, hashlib, time, math
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import brentq, minimize_scalar
from scipy.integrate import solve_ivp

try:
    import statsmodels.api as sm
    HAS_SM = True
except ImportError:
    HAS_SM = False
    sm = None

try:
    from statsmodels.formula.api import ols as smf_ols
    HAS_SMF = True
except (ImportError, AttributeError):
    HAS_SMF = False
    smf_ols = None

# ============= VERSION + LABELS =============
    VERSION        = "2.5+FIX-K1..K13"
PROTOCOL_LABEL = "P11.13-Rev7.4.0"
MODEL_LABEL    = "V50.6.6"
# ============================================

MODE           = "DEMO"
USE_E_TN_EFF   = False
CURRENT_CONDS  = ("B", "C2")
ALL_CONDS      = ("A", "B", "C", "C2", "D", "E")
EPS_DYNAMIC_MODE = False
EPS_FIXED_VALUE  = 0.15

TOLERANCE = {
    "ec0": (0.03, ""), "t0": (0.002, ""), "ecf": (0.03, ""),
    "tf": (0.002, ""), "dpd": (0.001, ""), "dpd_pre": (0.001, ""),
    "V_cell_mean": (0.005, ""), "I_mean_mA": (0.05, ""),
    "T_b_mean": (0.01, ""), "pH_meas": (0.01, ""),
    "T_sample": (0.05, ""), "V_L_measured": (0.0001, ""),
    "charge_C": (0.1, ""), "dilution": (0.01, ""),
}

DEMO_CITED_ERROR = "لا يمكن الاستشهاد بنتائج DEMO."

print("B1 - imports + labels + TOLERANCE (v2.5)")

def assert_not_demo_for_citation(summary=None):
    if MODE == "DEMO": raise ValueError(DEMO_CITED_ERROR)
    if summary is not None and summary.get("reference_type") == "DEMO":
        raise ValueError(DEMO_CITED_ERROR)
    return True

def assert_mode_is_valid():
    if MODE not in ("DEMO", "REAL"):
        raise ValueError(f"MODE={MODE} غير صالح.")
    return True

# ============= ثوابت فيزيائية =============
F = 96485.33212
M_CL2 = 70.906
V_L_DEFAULT = 0.100
E_TN_25 = 1.481
E_TN_CL_25 = 1.732
dETN_O2_dT = -0.165e-3
dETN_CL_dT = +1.248e-3
R_GAS = 8.314
M_W_KG = 18.015e-3

D_CL = 2.03e-9  # m²/s  (= 2.03e-5 cm²/s)   [FIX-K1: كانت 2.03e-5 m²/s = خطأ 1e4]
C_CL = 6.74      # mol/m³ (= 5.6 mmol/L)     [FIX-K1: كانت 5.6e-3 mol/L]
T_CL = 0.51
F_EDL = 1.15
A_E_cm2 = 2.0
A_E = A_E_cm2 * 1e-4  # m²

C_WATER = 418.0
C_GLASS = 55.0
C_TOTAL = C_WATER + C_GLASS
A_S = 1.9635e-3
A_WALL = 8.0e-3
U_WALL = 14.0
U_CONV = 1.3
EPS_IR = 0.95
SIGMA_SB = 5.67e-8
K_DRY_FIXED = (U_WALL*A_WALL + U_CONV*A_S + EPS_IR*SIGMA_SB*A_S*4*298.15**3)

# K25 موحّد مع K25_opt المُقاس
K25_EFF_DEFAULT = 1.6573e-3  # m/s
K25_OPT_V22     = 1.657276e-3  # m/s

K1 = 4.45e-7
K2 = 4.69e-11
KW = 1e-14

def pKa_HOCl_at_T(T_C):
    """Morris (1966) — J. Phys. Chem., 70(12), 3798-3805."""
    return 7.54 - 0.05 * (T_C - 25)

PKA_HOCL = 7.54

# ============= FIX-I6: Outliers =============
GRUBBS_ALPHA = 0.05
DIXON_ALPHA  = 0.05
# =============================================

print("B2 - physical constants (v2.5)")

X1_GRID_MIN = 1e-4
X1_GRID_MAX = 0.35
X1_GRID_N = 1500
EPS_GRID_N = 80
X1_THRESHOLD = 0.005
REL_SYST_DEFAULT = 0.10
REL_MEAS_DEFAULT = 0.03   # FIX-K11: مكوّن نسبي (تخفيف/ماصّة/معايرة DPD) فوق LOD/3.3
X1_PHYSICAL_MIN = 1e-5
X1_PHYSICAL_MAX = 0.5
RUNS_COLUMNS_MANDATORY = ["run","cond","ec0","t0","ecf","tf","dpd","dilution"]
ACCEPT = {"I_mA": (49.5, 50.5), "V_cell_V": (8.0, 15.0),
    "kappa25_0": (785.0, 815.0), "V_L": (0.0990, 0.1010),
    "T_air_std": (0.0, 0.5), "RH_std": (0.0, 5.0),
    "max_abs_err_K": (0.0, 0.1)}
R5_RANGE = (1.8, 2.2)
R8_EC_DRIFT_MAX = 0.10
R8_TB_RANGE = (15.0, 40.0)

# FIX-K1: مدى معقول لـ delta_eff (cm) — فحص منطقي وليس معيار تصديق (مشتق من x1)
DELTA_RANGE_CM = (0.003, 0.05)

# FIX-K2: مقاطع E (بداية، نهاية، جهد اسمي) — القراءة من آخر 60 ث في كل مقطع
E_SEGMENTS = [(240.0, 300.0, 2.0), (540.0, 600.0, 3.0), (840.0, 900.0, 4.0)]

# FIX-K3: الحقول الإلزامية لكل شرط (يجب أن تكون غير فارغة في runs.csv)
_REQ_BASE = ["ec0", "t0", "ecf", "tf", "dpd"]
_REQ_CURRENT = _REQ_BASE + ["I_mean_mA", "V_cell_mean", "T_b_mean",
                            "V_L_measured", "T_air_std", "RH_std"]
REQUIRED_BY_COND = {
    "A": _REQ_BASE + ["T_b_mean"],
    "B": _REQ_CURRENT,
    "C": _REQ_BASE + ["dpd_pre"],
    "C2": _REQ_CURRENT + ["dpd_pre"],
    "D": ["ec0", "t0", "ecf", "tf"],
    "E": ["ec0", "t0", "ecf", "tf"],
}

print("B3 - grids + columns + accept + segments + required fields")

def _trapz(y, x): return float(np.trapezoid(y, x))

def _scalar(x):
    a = np.asarray(x)
    if a.ndim == 0: return float(a)
    if a.size == 1: return float(a.reshape(-1)[0])
    raise ValueError(f"Expected scalar, got shape {a.shape}")

def P_sat(T_C):
    T_C = _scalar(T_C)
    return 611.2 * np.exp(17.67 * T_C / (T_C + 243.5))

def Q_evap_abs(T_b_C, T_air_C, RH_pct, K25_eff):
    T_b_C = _scalar(T_b_C); T_air_C = _scalar(T_air_C)
    RH_pct = _scalar(RH_pct); K25_eff = _scalar(K25_eff)
    T_b_K = T_b_C + 273.15; T_air_K = T_air_C + 273.15
    T_film_K = 0.5 * (T_b_K + T_air_K)
    dP = 0.9997 * P_sat(T_b_C) - (RH_pct / 100.0) * P_sat(T_air_C)
    if dP <= 0: return 0.0
    K_m = K25_eff * (T_b_K / 298.15) ** 1.75
    m_dot = A_S * K_m * (M_W_KG / (R_GAS * T_film_K)) * dP
    L_v = 2.501e6 - 2.37e3 * T_b_C
    return float(m_dot * L_v)

def Q_evap_delta(T_b_C, T_air_C, RH_pct, K25_eff):
    return (Q_evap_abs(T_b_C, T_air_C, RH_pct, K25_eff)
            - Q_evap_abs(T_air_C, T_air_C, RH_pct, K25_eff))

def E_tn_at(T_b_C, x1=None):
    T = np.asarray(T_b_C, dtype=float)
    if x1 is not None and USE_E_TN_EFF:
        E_O2 = E_TN_25 + dETN_O2_dT * (T - 25.0)
        E_Cl = E_TN_CL_25 + dETN_CL_dT * (T - 25.0)
        val = x1 * E_Cl + (1 - x1) * E_O2
    else:
        val = E_TN_25 + dETN_O2_dT * (T - 25.0)
    return float(val) if val.ndim == 0 else val

def kappa25(ec, T, alpha): return ec / (1.0 + alpha * (T - 25.0))

# ============= FIX-J2: دالة موحّدة للطاقة =============
def thermal_model_energy_balance(t_now, T_b, t_pos, I_pos, V_pos,
                                   T_air, RH, K25, K_dry, C_tot):
    """FIX-J2: دالة موحّدة لـgeneration وcalibration."""
    T_b = float(np.asarray(T_b).item())
    if t_pos is not None and I_pos is not None and V_pos is not None:
        I_t = float(np.interp(t_now, t_pos, I_pos))
        V_t = float(np.interp(t_now, t_pos, V_pos))
        Q_gen = I_t * (V_t - E_tn_at(T_b))
    else:
        Q_gen = 0.0
    Q_dry = K_dry * (T_b - T_air)
    Q_ev = Q_evap_delta(T_b, T_air, RH, K25)
    return (Q_gen - Q_dry - Q_ev) / C_tot
# ========================================================

print("B4 - helpers + thermal_model_energy_balance (v2.5)")

def charge_from_trace(path):
    d = pd.read_csv(path, comment='#')
    t = d["t_s"].to_numpy(float); I = d["I_mA"].to_numpy(float) / 1000.0
    m = t >= 0
    return _trapz(I[m], t[m])

def q_gen_from_trace(path):
    d = pd.read_csv(path, comment='#')
    t = d["t_s"].to_numpy(float); I = d["I_mA"].to_numpy(float) / 1000.0
    V = d["V_V"].to_numpy(float)
    T = d.get("T_b_C", pd.Series(np.full_like(t, 25.0))).to_numpy(float)
    Etn = E_tn_at(T); m = t >= 0
    return _trapz(I[m] * (V[m] - Etn[m]), t[m])

def compute_I_lim(trace_path, t_start=540.0, t_end=600.0,
                  min_points=10, max_rel_std=0.1, near_zero_threshold=1e-6):
    d = pd.read_csv(trace_path, comment='#')
    t = d["t_s"].to_numpy(float); I = d["I_mA"].to_numpy(float)
    mask = (t >= t_start) & (t <= t_end)
    if mask.sum() == 0: return None, None, 0, "insufficient_points"
    I_window = I[mask]
    valid_mask = ~np.isnan(I_window)
    n_valid = int(np.sum(valid_mask))
    if n_valid < min_points:
        return None, None, n_valid, "insufficient_points"
    I_valid = I_window[valid_mask]
    I_lim = float(np.mean(I_valid))
    I_lim_std = float(np.std(I_valid))
    warnings_list = []
    if abs(I_lim) < near_zero_threshold:
        warnings_list.append("I_lim_near_zero")
    else:
        if I_lim < 0: warnings_list.append("negative_I_lim")
        rel_std = I_lim_std / abs(I_lim)
        if rel_std > max_rel_std:
            warnings_list.append(f"high_rel_std={rel_std:.3f}")
    warning = ";".join(warnings_list) if warnings_list else None
    return I_lim, I_lim_std, n_valid, warning

def compute_I_lim_segments(trace_path, segments=None, plateau_tol=0.05,
                           v_tol=0.05, v_nom_tol=0.2, min_points=10):
    """FIX-K2: I_lim من آخر 60 ث في كل مقطع جهد (يتجنب العابر بعد كل خطوة)،
    ثم اختبار هضبة: |I(V_أعلى) - I(V_السابق)| / I <= plateau_tol."""
    if segments is None: segments = E_SEGMENTS
    d = pd.read_csv(trace_path, comment='#')
    t = d["t_s"].to_numpy(float); I = d["I_mA"].to_numpy(float)
    V = d["V_V"].to_numpy(float) if "V_V" in d.columns else np.full_like(t, np.nan)
    seg_res = []
    for a, b, v_nom in segments:
        m = (t >= a) & (t < b)
        Iw = I[m]; Iw = Iw[~np.isnan(Iw)]
        if len(Iw) < min_points:
            seg_res.append({"V_nom": v_nom, "error": "insufficient_points", "n": int(len(Iw))})
            continue
        Vw = V[m]; Vw = Vw[~np.isnan(Vw)]
        v_ok = bool(len(Vw) > 0 and (np.max(Vw) - np.min(Vw)) <= v_tol
                    and abs(float(np.mean(Vw)) - v_nom) <= v_nom_tol)
        seg_res.append({"V_nom": v_nom, "I_mean_mA": float(np.mean(Iw)),
                        "I_std_mA": float(np.std(Iw)), "n": int(len(Iw)),
                        "V_mean": (float(np.mean(Vw)) if len(Vw) else None),
                        "V_constant": v_ok})
    valid = [x for x in seg_res if "I_mean_mA" in x]
    if len(valid) < 2:
        return {"error": "need >=2 valid segments", "segments": seg_res}
    warns = []
    for x in valid:
        if not x["V_constant"]:
            warns.append(f"V_not_at_nominal@{x['V_nom']}V")
    last, prev = valid[-1], valid[-2]
    if abs(prev["I_mean_mA"]) < 1e-6:
        rel = float("nan"); plateau = False
        warns.append("prev_segment_near_zero")
    else:
        rel = (last["I_mean_mA"] - prev["I_mean_mA"]) / abs(prev["I_mean_mA"])
        plateau = bool(abs(rel) <= plateau_tol)
    if not plateau:
        warns.append(f"no_plateau: I changes {rel*100:.1f}% between "
                     f"{prev['V_nom']}V and {last['V_nom']}V (I_lim not reached)")
    I_lim = float(np.mean([prev["I_mean_mA"], last["I_mean_mA"]])) if plateau else None
    I_lim_std = float(max(prev["I_std_mA"], last["I_std_mA"])) if plateau else None
    return {"I_lim_mA": I_lim, "I_lim_std_mA": I_lim_std, "plateau": plateau,
            "rel_change_last_two": (None if np.isnan(rel) else float(rel)),
            "plateau_tol": plateau_tol, "segments": seg_res,
            "warning": ";".join(warns) if warns else None}

def dpd_sigma(lod, dilution, dpd_obs, rel=None):
    """FIX-K11: sigma لقياس DPD بعد التخفيف: sqrt((LOD/3.3*dil)^2 + (rel*dpd_obs)^2)."""
    if rel is None: rel = REL_MEAS_DEFAULT
    dil = np.asarray(dilution, float); obs = np.asarray(dpd_obs, float)
    return np.sqrt(((lod/3.3)*dil)**2 + (rel*obs)**2)

def c2_consistency_check(df, x1r, recC, VL=V_L_DEFAULT, lod=0.05, z_flag=2.0):
    """FIX-K13: C2 = محلول مُضاف إليه Cl2 + تيار. المتوقع = r_C × (dpd_pre + Cl2 المنتَج بـx1 المقدَّر).
    إن كان المقاس أقل بوضوح => فقد إضافي تحت التيار (ε_B > ε_C) => x1 مُقدَّر بأقل من حقيقته.
    يُرجع أيضًا x1 البديل لو استُخدمت ε الفعّالة من C2 (حساسية، وليس تقديرًا معتمدًا)."""
    out = {"available": False}
    if recC is None or x1r.get("median") is None or "dpd_pre" not in df.columns:
        return out
    sub = df[df["cond"] == "C2"]
    if len(sub) == 0 or not sub["dpd_pre"].notna().all():
        return out
    r = float(recC["mean"]); sr = max(float(recC.get("std", 0.1) or 0.1), 0.05)
    if r <= 0: return out
    x1 = float(x1r["median"])
    lo = x1r.get("CI95_lo_final", x1r.get("CI95_lo")); hi = x1r.get("CI95_hi_final", x1r.get("CI95_hi"))
    x1_rel = ((hi - lo) / (2 * 1.96) / x1) if (lo is not None and hi is not None and x1 > 0) else 0.10
    rows = []
    for _, row in sub.iterrows():
        Q = float(row["Q"]); pre = float(row["dpd_pre"]); dil = float(row["dilution"])
        obs = float(row["dpd"]) * dil
        prod = x1 * (Q / F) / 2.0 * M_CL2 * 1000.0 / VL
        pred = r * (pre + prod)
        s_obs = float(dpd_sigma(lod, dil, obs))
        s_pred = float(np.sqrt((pred * sr / r) ** 2 + (r * prod * x1_rel) ** 2))
        s_tot = float(np.sqrt(s_obs ** 2 + s_pred ** 2))
        eps_eff = float(np.clip(1.0 - obs / (pre + prod), 0.0, 0.99))
        rows.append({"run": str(row["run"]), "obs": obs, "pred": float(pred),
                     "ratio": float(obs / pred), "z": float((obs - pred) / s_tot),
                     "sigma": s_tot, "eps_effective": eps_eff,
                     "min_detectable_rel": float(2.0 * s_tot / pred)})
    mean_z = float(np.mean([x["z"] for x in rows]))
    mean_eps = float(np.mean([x["eps_effective"] for x in rows]))
    x1_alt = float(x1 * r / (1.0 - mean_eps))
    out.update({"available": True, "runs": rows, "mean_z": mean_z,
                "mean_ratio": float(np.mean([x["ratio"] for x in rows])),
                "consistent": bool(abs(mean_z) <= z_flag), "z_flag": z_flag,
                "eps_C": float(1.0 - r), "eps_C2_effective": mean_eps,
                "x1_if_eps_C2": x1_alt, "x1_shift_pct": float((x1_alt / x1 - 1.0) * 100.0),
                "min_detectable_deficit": float(np.mean([x["min_detectable_rel"] for x in rows])),
                "note": "z<0 بوضوح = فقد إضافي تحت التيار؛ z>0 = كسب غير متوقع (تلوث/خطأ dpd_pre). "
                        "الأخطاء المشتركة بين التشغيلتين لا تُقسَم على sqrt(n)."})
    return out

def dpd_expected(x1, eps, Q, VL=0.1):
    n = x1 * (Q / F) / 2.0
    return n * (1.0 - eps) * M_CL2 * 1000.0 / VL

def integrate_thermal_ode(trace_path, T_air=25.0, RH_pct=50.0,
                          K25_eff=K25_EFF_DEFAULT, K_dry=K_DRY_FIXED, C_tot=C_TOTAL):
    d = pd.read_csv(trace_path, comment='#')
    t = d["t_s"].to_numpy(float); I = d["I_mA"].to_numpy(float)/1000.0
    V = d["V_V"].to_numpy(float)
    T_b_meas = d.get("T_b_C", pd.Series(np.full_like(t,25.0))).to_numpy(float)
    m = t >= 0
    if not m.any(): return {"error": "no current data"}
    t_pos = t[m] - t[m][0]; I_pos = I[m]; V_pos = V[m]
    T0 = float(T_b_meas[m][0]); T_b_meas_pos = T_b_meas[m]
    def dTb(t_now, T_b):
        return thermal_model_energy_balance(
            t_now, T_b, t_pos, I_pos, V_pos,
            T_air, RH_pct, K25_eff, K_dry, C_tot
        )
    try:
        sol = solve_ivp(dTb, [t_pos[0], t_pos[-1]], [T0], t_eval=t_pos,
                        method='RK45', rtol=1e-9, atol=1e-11)
        max_abs_err = float(np.max(np.abs(sol.y[0] - T_b_meas_pos)))
    except Exception as e: return {"error": str(e)}
    return {"T_b_meas_final": float(T_b_meas_pos[-1]),
            "T_b_pred_final": float(sol.y[0][-1]),
            "max_abs_err_K": max_abs_err, "K_dry_W_per_K": float(K_dry),
            "K25_eff_used": float(K25_eff), "T_air_used": float(T_air),
            "RH_used": float(RH_pct), "T0_used": float(T0)}

print("B5 - traces + thermal ODE + compute_I_lim")

def _integrate_D_curve(d, K25, T_air, RH, K_dry=None, C_tot=None, metric="max"):
    """FIX-J2: تستخدم thermal_model_energy_balance الموحّدة."""
    if K_dry is None: K_dry = K_DRY_FIXED
    if C_tot is None: C_tot = C_TOTAL
    t = d["t_s"].to_numpy(float); T_meas = d["T_b_C"].to_numpy(float)
    if len(t) < 2: return 1e10
    def dTb(t_now, T_b):
        return thermal_model_energy_balance(
            t_now, T_b, None, None, None,
            T_air, RH, K25, K_dry, C_tot
        )
    try:
        sol = solve_ivp(dTb, [t[0], t[-1]], [T_meas[0]], t_eval=t,
                        method='RK45', rtol=1e-10, atol=1e-12)
        err = sol.y[0] - T_meas
        if metric == "rms": return float(np.sqrt(np.mean(err**2)))
        return float(np.max(np.abs(err)))
    except Exception: return 1e10

def read_environment(trace_path, strict=None):
    T_air, RH = None, None
    with open(trace_path) as f:
        for line in f:
            if not line.startswith("#"): break
            if "T_air_actual" in line:
                T_air = float(line.split("=")[1].strip().split()[0])
            elif "RH_actual" in line:
                RH = float(line.split("=")[1].strip().split()[0])
    if strict is None: strict = (MODE == "REAL")
    if T_air is None:
        if strict: raise ValueError("Trace header missing T_air_actual.")
        T_air = 24.8
    if RH is None:
        if strict: raise ValueError("Trace header missing RH_actual.")
        RH = 51.2
    return T_air, RH

def calibrate_K25_from_D_refined(trace_path, K25_coarse_range=(1e-4,1e-2),
                                  K25_coarse_n=20, max_err_tol=0.1,
                                  n_iterations=5, min_range_width=1e-6):
    d = pd.read_csv(trace_path, comment='#')
    try: T_air, RH = read_environment(str(trace_path), strict=False)
    except ValueError: T_air, RH = 25.0, 50.0
    def loss(K25): return _integrate_D_curve(d, K25, T_air, RH, metric="rms")  # FIX-K6: RMS بدل max (أقل حساسية للضوضاء)
    lo, hi = K25_coarse_range
    best_K25 = None; best_err = np.inf
    for iteration in range(n_iterations):
        if (hi - lo) < min_range_width: break
        K25_vals = np.linspace(lo, hi, K25_coarse_n)
        errs = np.array([loss(k) for k in K25_vals])
        idx_best = int(np.argmin(errs))
        lo_local = K25_vals[max(0, idx_best-1)]
        hi_local = K25_vals[min(len(K25_vals)-1, idx_best+1)]
        try:
            r = minimize_scalar(loss, bounds=(lo_local, hi_local),
                                method='bounded', options={'xatol':1e-10,'maxiter':300})
            K25_iter = float(r.x); err_iter = float(r.fun)
        except Exception:
            K25_iter = float(K25_vals[idx_best]); err_iter = float(errs[idx_best])
        if err_iter < best_err:
            best_K25 = K25_iter; best_err = err_iter
        width = max((hi-lo)*0.1, min_range_width)
        lo = max(K25_coarse_range[0], K25_iter - width)
        hi = min(K25_coarse_range[1], K25_iter + width)
    if best_K25 is None: return None, float("inf"), False
    err_max = _integrate_D_curve(d, best_K25, T_air, RH, metric="max")
    return best_K25, err_max, bool(err_max < max_err_tol)

def calibrate_K25_multi(d_paths, spread_max=0.20):
    """FIX-K6: معايرة K25 من كل تشغيلات D (الوسيط) مع تقرير التشتت."""
    fits = []
    for p in d_paths:
        try:
            k, e, cv = calibrate_K25_from_D_refined(str(p))
            fits.append({"run": Path(p).stem, "K25": (float(k) if k is not None else None),
                         "err_max_K": float(e), "converged": bool(cv)})
        except Exception as ex:
            fits.append({"run": Path(p).stem, "error": str(ex)[:80], "converged": False})
    good = [f["K25"] for f in fits if f.get("converged") and f.get("K25") is not None]
    out = {"fits": fits, "n_runs": len(fits), "n_converged": len(good),
           "K25_median": None, "spread_rel": None, "spread_max": spread_max}
    if good:
        med = float(np.median(good)); out["K25_median"] = med
        if len(good) >= 2: out["spread_rel"] = float((max(good) - min(good)) / med)
    return out

print("B6 - D curve + K25 calibration")

# ============= استكمال الخلية 1 — الجزء 2 =============

def solve_charge_balance(x1, Q, VL, pH_0, pH_f_meas):
    if x1 is None or pH_f_meas is None: return None
    n_e = Q / F; C_added = x1 * n_e / (2 * VL)
    h0 = 10**(-pH_0); OH0 = KW/h0
    D0 = h0**2 + K1*h0 + K1*K2
    a1_0 = K1*h0/D0; a2_0 = K1*K2/D0
    def eq(pH, CT):
        h = 10**(-pH); OH = KW/h
        pka = pKa_HOCl_at_T(25.0)
        alpha_HOCl = 1.0/(1.0 + 10**(pka-pH))
        D = h**2 + K1*h + K1*K2
        a1 = K1*h/D; a2 = K1*K2/D
        return (C_added*(1-alpha_HOCl) - CT*(a1-a1_0+2*a2-2*a2_0)
                - (OH-OH0) + (h-h0))
    def find_CT(CT):
        try: return brentq(lambda ph: eq(ph, CT), 4.0, 12.0) - pH_f_meas
        except Exception: return -1.0
    try: return float(brentq(find_CT, 1e-5, 5e-2))
    except Exception: return None

def compute_gas_volumes(x1, Q, T_K=298.15, P_atm=101325.0):
    if x1 is None: return None
    n_e = Q/F
    V_H2 = (n_e/2.0)*R_GAS*T_K/P_atm*1e6
    V_O2 = ((1.0-x1)*n_e/4.0)*R_GAS*T_K/P_atm*1e6
    return {"V_H2_mL": float(V_H2), "V_O2_mL": float(V_O2),
            "V_total_mL": float(V_H2+V_O2)}

def back_calculate_delta(x1_median, I_avg_A, warn=True):
    """FIX-K1: وحدات SI متسقة (D بـ m²/s، C بـ mol/m³) والناتج بالـ cm."""
    if x1_median is None or x1_median <= 0 or I_avg_A <= 0: return None
    delta_m = (F*D_CL*C_CL*A_E*F_EDL)/(x1_median*I_avg_A*(1-T_CL))
    delta = float(delta_m * 100.0)  # m -> cm
    if (delta < DELTA_RANGE_CM[0] or delta > DELTA_RANGE_CM[1]) and warn and MODE == "REAL":
        warnings.warn(f"delta_eff={delta:.4f} cm خارج {DELTA_RANGE_CM}", stacklevel=2)
    return delta

def beta_from_rec(r, sr, srm=0.20, sr_min=0.05):
    if r is None or sr is None or sr > srm: return (1.2, 4.9)
    if r > 1.0: raise ValueError(f"rec={r:.4f} > 1.0.")
    if r <= 0.0: raise ValueError(f"rec={r:.4f} <= 0.")
    e = 1.0 - r
    if e < 0.01: raise ValueError(f"eps={e:.4f} < 0.01.")
    if e > 0.99: raise ValueError(f"eps={e:.4f} > 0.99.")
    sr = max(float(sr), sr_min); s2 = sr**2
    if e*(1-e)/s2 <= 1: return (1.2, 4.9)
    a = e*(e*(1-e)/s2 - 1); b = (1-e)*(e*(1-e)/s2 - 1)
    return (max(a,0.01), max(b,0.01))

def eps_prior_from_C_or_C2(rec_C, rec_C2):
    if rec_C is not None:
        m = float(rec_C["mean"]); s = max(float(rec_C["std"]), 0.05)
        return beta_from_rec(m, s), "C"
    elif rec_C2 is not None and rec_C2["mean"] < 1.0:
        return beta_from_rec(rec_C2["mean"], rec_C2["std"]), "C2"
    return beta_from_rec(None, None), "default"

print("B7 - chemistry + gas + delta + beta")

# ============= FIX-I2: GUM =============
def gum_uncertainty(x1_median, x1_std_A, rel_syst_B=0.10):
    """GUM (JCGM 100:2008) — Combined uncertainty."""
    u_A = float(x1_std_A) if x1_std_A else 0.0
    u_B = float(rel_syst_B) * float(x1_median)
    u_c = float(np.sqrt(u_A**2 + u_B**2))
    U = 2.0 * u_c
    return {
        "u_A": u_A,
        "u_B": u_B,
        "u_c": u_c,
        "U_expanded": U,
        "k": 2.0,
        "CI95_GUM": [float(x1_median - U), float(x1_median + U)],
    }

# ============= FIX-I3: Monte Carlo =============
def error_propagation_monte_carlo(dpd_obs, Q_list, sig_list, VL=0.1, eb=(1.2, 4.9),
                                  n_samples=10000, seed=42):
    """FIX-K7: Monte Carlo مبني على البيانات الفعلية:
    dpd ~ N(dpd_i, sig_i)، eps ~ Beta(eb)، Q ~ N(Q_i, 1%)، V_L ~ N(VL, 0.00015).
    x1_i = dpd_i*VL / ((1-eps)*M*1000*Q_i/(2F))، ثم متوسط التشغيلات."""
    rng = np.random.default_rng(seed=seed)
    dpd = np.asarray(dpd_obs, float); Q = np.asarray(Q_list, float)
    sg = np.asarray(sig_list, float); n = len(dpd)
    eps = rng.beta(eb[0], eb[1], n_samples)
    VLs = rng.normal(VL, 0.00015, n_samples)
    Qs = Q[None, :] * (1.0 + rng.normal(0.0, 0.01, (n_samples, n)))
    d = dpd[None, :] + rng.normal(0.0, 1.0, (n_samples, n)) * sg[None, :]
    x1_runs = d * VLs[:, None] / ((1.0 - eps[:, None]) * M_CL2 * 1000.0 * Qs / (2.0 * F))
    x1 = x1_runs.mean(axis=1)
    return {"n_samples": int(n_samples),
            "x1_mean": float(np.mean(x1)), "x1_std": float(np.std(x1)),
            "x1_median": float(np.median(x1)),
            "CI95_lo": float(np.percentile(x1, 2.5)),
            "CI95_hi": float(np.percentile(x1, 97.5)),
            "method": "Monte Carlo (data-driven; cross-check فقط)"}

def sensitivity_elasticities(eps):
    """FIX-K7: مرونات تحليلية d ln x1 / d ln p للنموذج
    x1 = dpd*V_L / ((1-eps)*M*1000*Q/(2F)). K25 لا يدخل في x1."""
    return {"method": "analytic elasticities (d ln x1 / d ln p)",
            "elasticities": {"dpd": 1.0, "Q": -1.0, "V_L": 1.0,
                             "eps": float(eps/(1.0-eps)), "K25": 0.0},
            "note": "x1 ∝ dpd·V_L/((1-eps)·Q). K25 لا يؤثر في x1."}

def detect_outliers_grubbs(data, alpha=0.05):
    """Grubbs test."""
    data = np.asarray(data, dtype=float)
    n = len(data)
    if n < 7:
        return {"method": "Grubbs", "outliers": [], "n_outliers": 0, "note": "n < 7"}
    mean = np.mean(data); std = np.std(data, ddof=1)
    if std == 0:
        return {"method": "Grubbs", "outliers": [], "n_outliers": 0, "note": "std=0"}
    G = np.abs(data - mean) / std
    from scipy.stats import t as t_dist
    t_crit = t_dist.ppf(1 - alpha/(2*n), n-2)
    G_crit = (n-1) * np.sqrt(t_crit**2 / (n * (n - 2 + t_crit**2)))
    outliers = np.where(G > G_crit)[0].tolist()
    return {
        "method": "Grubbs", "n": n,
        "G_max": float(np.max(G)), "G_crit": float(G_crit),
        "outliers_idx": outliers, "n_outliers": len(outliers),
    }

def detect_outliers_dixon(data, alpha=0.05):
    """Dixon Q test."""
    data = np.sort(np.asarray(data, dtype=float))
    n = len(data)
    if n < 3 or n > 10:
        return {"method": "Dixon", "outliers": [], "n_outliers": 0, "note": "n out of [3,10]"}
    rng_data = data[-1] - data[0]
    if rng_data == 0:
        return {"method": "Dixon", "outliers": [], "n_outliers": 0, "note": "range=0"}
    Q_low = (data[1] - data[0]) / rng_data
    Q_high = (data[-1] - data[-2]) / rng_data
    Q_crit = {3:0.941, 4:0.765, 5:0.642, 6:0.560,
              7:0.507, 8:0.468, 9:0.437, 10:0.412}.get(n, 1.0)
    outliers = []
    if Q_low > Q_crit: outliers.append(0)
    if Q_high > Q_crit: outliers.append(n-1)
    return {
        "method": "Dixon Q", "n": n,
        "Q_low": float(Q_low), "Q_high": float(Q_high),
        "Q_crit": float(Q_crit), "outliers_idx": outliers, "n_outliers": len(outliers),
    }

# ============= FIX-I7: Reproducibility =============
def reproducibility_check(df_list, value_col="x1_median"):
    """CV across multiple runs."""
    if not df_list or len(df_list) < 2:
        return {"method": "CV", "n_runs": len(df_list) if df_list else 0, "note": "need >= 2"}
    values = [float(d.get(value_col, np.nan)) for d in df_list if d.get(value_col) is not None]
    values = [v for v in values if not np.isnan(v)]
    if len(values) < 2:
        return {"method": "CV", "n_runs": len(values), "note": "need >= 2 valid"}
    mean = np.mean(values); std = np.std(values, ddof=1)
    cv = std / mean if mean != 0 else float('inf')
    return {
        "method": "CV", "n_runs": len(values),
        "mean": float(mean), "std": float(std),
        "CV": float(cv), "CV_pct": float(cv * 100),
        "acceptable": bool(cv < 0.10),
    }

print("B7b - GUM + MC + elasticities + Outliers + Repro")

def _infer_x1_grid(dpd_list, Q_list, sig_list, VL, eb, sigma_x=0.0):
    x1g = np.logspace(np.log10(X1_GRID_MIN), np.log10(X1_GRID_MAX), X1_GRID_N)
    eg = np.linspace(0.001, 0.60, EPS_GRID_N)
    ep = stats.beta.pdf(eg, *eb); ep = ep/ep.sum()
    log_eps_prior = np.log(np.maximum(ep, 1e-300))
    dl = np.asarray(dpd_list, dtype=float); Ql = np.asarray(Q_list, dtype=float)
    sl = np.asarray(sig_list, dtype=float)
    x1_col = x1g[:, None, None]; eps_row = eg[None, :, None]
    Q_row = Ql[None, None, :]; sig_row = sl[None, None, :]
    n_cl2 = x1_col*(Q_row/F)/2.0
    if sigma_x > 0:
        mu_c = n_cl2*(1.0-eps_row)*M_CL2*1000.0/VL
        sig_eff = np.sqrt(sig_row**2 + ((mu_c/np.maximum(x1_col,1e-20))*sigma_x)**2)
        resid = (dl[None,None,:] - mu_c)/sig_eff
        log_lik = -0.5*resid**2 - np.log(sig_eff)
    else:
        mu = n_cl2*(1.0-eps_row)*M_CL2*1000.0/VL
        resid = (dl[None,None,:] - mu)/sig_row
        log_lik = -0.5*resid**2 - np.log(sig_row)
    log_lik_total = log_lik.sum(axis=2)
    log_post_joint = log_lik_total + log_eps_prior[None, :]
    log_post_x1 = np.logaddexp.reduce(log_post_joint, axis=1)
    log_post_x1 -= log_post_x1.max()
    p = np.exp(log_post_x1); p /= p.sum(); c = np.cumsum(p)
    return {"median": float(x1g[np.searchsorted(c,0.50)]),
            "CI95_lo": float(x1g[np.searchsorted(c,0.025)]),
            "CI95_hi": float(x1g[np.searchsorted(c,0.975)]),
            "P_x1_gt_0.005": float(p[x1g>X1_THRESHOLD].sum())}

def infer_x1(dpd_list, Q_list, sig_list, VL=0.1, eb=(1.2,4.9)):
    if EPS_DYNAMIC_MODE:
        a, b = eb; eps_used = float(a/(a+b))
        eps_source = f"dynamic_from_beta(a={a:.2f}, b={b:.2f})"
    else:
        eps_used = EPS_FIXED_VALUE
        eps_source = f"fixed_at_{EPS_FIXED_VALUE}"
    res1 = _infer_x1_grid(dpd_list, Q_list, sig_list, VL, eb, sigma_x=0.0)
    x1_1 = res1["median"]
    preds = np.array([dpd_expected(x1_1, eps_used, Q, VL) for Q in Q_list])
    resid = np.asarray(dpd_list, float) - preds
    sig_obs = np.asarray(sig_list, float)
    sigma_x_est = 0.0; chi2_dof_pass1 = None
    if len(resid) >= 3:
        chi2 = np.sum((resid/sig_obs)**2)
        dof = len(resid)-1; chi2_dof_pass1 = chi2/dof
        if chi2_dof_pass1 > 1.2:
            s_res_sq = float(np.var(resid, ddof=1))
            s_obs_sq = float(np.mean(sig_obs**2))
            excess = max(0.0, s_res_sq - s_obs_sq)
            mu_mean = float(np.mean(preds))
            if mu_mean > 0 and x1_1 > 0:
                sigma_x_est = np.sqrt(excess)*x1_1/mu_mean
            sigma_x_est = min(sigma_x_est, 0.2*x1_1)
    if sigma_x_est > 0:
        res2 = _infer_x1_grid(dpd_list, Q_list, sig_list, VL, eb, sigma_x=sigma_x_est)
        preds2 = np.array([dpd_expected(res2["median"], eps_used, Q, VL) for Q in Q_list])
        sl_eff2 = np.sqrt(sig_obs**2 + ((preds2/res2["median"])*sigma_x_est)**2)
        resid2 = np.asarray(dpd_list, float) - preds2
        chi2_post = np.sum((resid2/sl_eff2)**2)
        dof_post = max(1, len(resid2)-1)
        res2["sigma_x"] = float(sigma_x_est); res2["hierarchical"] = True
        res2["chi2_dof"] = float(chi2_post/dof_post)
        res2["chi2_dof_pass1"] = float(chi2_dof_pass1)
        res2["eps_fixed_at"] = eps_used; res2["eps_source"] = eps_source
        res2["eps_dynamic_mode"] = EPS_DYNAMIC_MODE
        return res2
    else:
        res1["sigma_x"] = 0.0; res1["hierarchical"] = False
        res1["chi2_dof"] = (float(chi2_dof_pass1) if chi2_dof_pass1 is not None else None)
        res1["chi2_dof_pass1"] = res1["chi2_dof"]
        res1["eps_fixed_at"] = eps_used; res1["eps_source"] = eps_source
        res1["eps_dynamic_mode"] = EPS_DYNAMIC_MODE
        return res1

def add_systematic_asymmetric(median, ci_lo_bayes, ci_hi_bayes, rel_syst=0.10):
    half_lo = median - ci_lo_bayes; half_hi = ci_hi_bayes - median
    syst = rel_syst*median
    return (float(median - np.sqrt(half_lo**2+syst**2)),
            float(median + np.sqrt(half_hi**2+syst**2)))

def assert_x1_physical(x1_median, mode=None):
    if mode is None: mode = MODE
    if mode != "REAL": return True
    if x1_median is None: return True
    if not (X1_PHYSICAL_MIN <= x1_median <= X1_PHYSICAL_MAX):
        raise ValueError(f"x1_median={x1_median:.6e} خارج النطاق.")
    return True

print("B8 - Bayesian grid + infer_x1")

def welch_blocked(df_ab, dv="dk_res", group="cond", block="day"):
    df_use = df_ab[df_ab[group].isin(["A","B"])].copy()
    A = df_use[df_use[group]=="A"][dv].to_numpy()
    B = df_use[df_use[group]=="B"][dv].to_numpy()
    out = {"n_A": int(len(A)), "n_B": int(len(B))}
    if len(A) >= 2 and len(B) >= 2:
        t_naive, p_naive = stats.ttest_ind(B, A, equal_var=False)
        out["welch_t_naive"] = float(t_naive); out["welch_p_naive"] = float(p_naive)
        out["method"] = "welch_naive"
    else:
        out["method"] = "insufficient_n"; return out
    if block not in df_use.columns or not HAS_SM or len(df_use) < 4:
        out["p_reported"] = out["welch_p_naive"]; return out
    days_both = sum(1 for d in df_use[block].unique()
                    if (df_use[df_use[block]==d][group]=="A").sum() >= 1
                    and (df_use[df_use[block]==d][group]=="B").sum() >= 1)
    if days_both < 2:
        out["p_reported"] = out["welch_p_naive"]; return out
    df_use["cond_c"] = df_use[group].astype("category")
    df_use["day_c"] = df_use[block].astype("category")
    if HAS_SMF:
        try:
            model = smf_ols(dv + " ~ C(cond_c) + C(day_c)", data=df_use).fit()
            anova = sm.stats.anova_lm(model, typ=2)
            cond_rows = [r for r in anova.index if 'cond_c' in str(r)]
            day_rows = [r for r in anova.index if 'day_c' in str(r)]
            if cond_rows: out["blocked_p_cond"] = float(anova.loc[cond_rows[0], "PR(>F)"])
            if day_rows: out["blocked_p_day"] = float(anova.loc[day_rows[0], "PR(>F)"])
            out["method"] = "two_way_anova"
            out["p_reported"] = out.get("blocked_p_cond", out["welch_p_naive"])
        except Exception as e:
            out["p_reported"] = out["welch_p_naive"]; out["note"] = str(e)
    else:
        out["p_reported"] = out["welch_p_naive"]
        out["method"] = "welch_only_no_smf"
    return out

def pH_correct(ph, Tb, T_sample=None, TRT=22.0):
    if T_sample is not None: return ph - 0.008*(Tb-T_sample)
    return ph - 0.008*(Tb-TRT)

def validate_runs(df, VL=V_L_DEFAULT, strict_mode=False):
    warns = []
    if "V_cell_mean" in df.columns:
        lo, hi = ACCEPT["V_cell_V"]
        for _, r in df.iterrows():
            if r["cond"] in CURRENT_CONDS:
                v = r.get("V_cell_mean", np.nan)
                if pd.notna(v) and not (lo <= v <= hi):
                    warns.append(f"[{r['run']}] V_cell={v:.2f}")
    if "I_mean_mA" in df.columns:
        lo, hi = ACCEPT["I_mA"]
        for _, r in df.iterrows():
            if r["cond"] in CURRENT_CONDS:
                i = r.get("I_mean_mA", np.nan)
                if pd.notna(i) and not (lo <= i <= hi):
                    warns.append(f"[{r['run']}] I={i:.2f}")
    if "k0" in df.columns:
        lo, hi = ACCEPT["kappa25_0"]
        for _, r in df.iterrows():
            if r["cond"] in ("D","C2"): continue
            k0 = r.get("k0", np.nan)
            if pd.notna(k0) and not (lo <= k0 <= hi):
                warns.append(f"[{r['run']}] kappa25_0={k0:.1f}")
    for col in ["T_air_std", "RH_std"]:
        if col in df.columns:
            lo, hi = ACCEPT[col]
            for _, r in df.iterrows():
                s = r.get(col, np.nan)
                if pd.notna(s) and s > hi:
                    warns.append(f"[{r['run']}] {col}={s:.3f}")
    if "V_L_measured" in df.columns:
        lo, hi = ACCEPT["V_L"]
        for _, r in df.iterrows():
            v = r.get("V_L_measured", np.nan)
            if pd.notna(v) and not (lo <= v <= hi):
                warns.append(f"[{r['run']}] V_L={v:.4f}")
    if "dpd_pre" in df.columns:
        for _, r in df.iterrows():
            if r["cond"] in ("C","C2"):
                dp = r.get("dpd_pre", np.nan)
                if pd.isna(dp):
                    if strict_mode: warns.append(f"[{r['run']}] dpd_pre=NaN (C-9)")
                elif dp <= 0:
                    warns.append(f"[{r['run']}] dpd_pre={dp:.4f} <= 0 (C-9)")
                elif not (R5_RANGE[0] <= dp <= R5_RANGE[1]):
                    warns.append(f"[{r['run']}] dpd_pre={dp:.4f} خارج [{R5_RANGE[0]}, {R5_RANGE[1]}] (R-5)")
    if "cond" in df.columns and "T_b_mean" in df.columns:
        c2 = df[df["cond"]=="C2"]
        if len(c2) >= 2:
            std_tb = float(c2["T_b_mean"].std(ddof=1))
            if std_tb > 0.3:
                warns.append(f"C2 T_b std={std_tb:.3f} > 0.3 (R-6)")
    if strict_mode:
        optional_cols = ["V_L_measured","T_b_mean","T_air_std","RH_std",
                         "I_mean_mA","V_cell_mean"]  # FIX-K3
        for col in optional_cols:
            if col in df.columns:
                for _, r in df.iterrows():
                    if r["cond"] in CURRENT_CONDS:
                        if pd.isna(r.get(col)):
                            warns.append(f"[{r['run']}] {col}=NaN (R-7)")
    if "ec0" in df.columns and "ecf" in df.columns:
        for _, r in df.iterrows():
            ec0 = r.get("ec0", np.nan); ecf = r.get("ecf", np.nan)
            if pd.notna(ec0) and pd.notna(ecf) and ec0 > 0:
                rel_ec = abs(ecf - ec0) / ec0
                if rel_ec > R8_EC_DRIFT_MAX:
                    warns.append(f"[{r['run']}] Δec/ec0={rel_ec*100:.1f}% > 10% (R-8a)")
    if "T_b_mean" in df.columns:
        for _, r in df.iterrows():
            tb = r.get("T_b_mean", np.nan)
            if pd.notna(tb):
                if tb > R8_TB_RANGE[1]:
                    warns.append(f"[{r['run']}] T_b={tb:.2f} > {R8_TB_RANGE[1]} (R-8b)")
                elif tb < R8_TB_RANGE[0]:
                    warns.append(f"[{r['run']}] T_b={tb:.2f} < {R8_TB_RANGE[0]} (R-8c)")
    return warns

def make_runs_template():
    """FIX-K3: قالب بلا قيم مُعبَّأة مسبقًا للقياسات (dpd_pre وcharge_C وV_L_measured فارغة)."""
    n = 18
    return pd.DataFrame({
        "run": ([f"A_{i}" for i in range(1, 6)] + [f"B_{i}" for i in range(1, 6)]
                + [f"C_{i}" for i in range(1, 3)] + [f"C2_{i}" for i in range(1, 3)]
                + [f"D_{i}" for i in range(1, 4)] + ["E_1"]),
        "cond": (["A"]*5 + ["B"]*5 + ["C"]*2 + ["C2"]*2 + ["D"]*3 + ["E"]),
        "day": ([1,1,1,2,2] + [1,1,1,2,2] + [2,2] + [2,2] + [1,2,2] + [2]),
        "ec0": [None]*n, "t0": [None]*n, "ecf": [None]*n, "tf": [None]*n,
        "dpd": [None]*n,
        "dilution": [1]*5 + [3]*5 + [1]*2 + [5]*2 + [1]*3 + [1],
        "charge_C": [0]*5 + [None]*5 + [0]*2 + [None]*2 + [0]*3 + [0],
        "dpd_pre": [None]*n,
        "V_cell_mean": [0]*5 + [None]*5 + [0]*2 + [None]*2 + [0]*3 + [None],
        "I_mean_mA": [None]*n, "T_b_mean": [None]*n, "pH_meas": [None]*n,
        "T_sample": [None]*n, "ATC_state": ["OFF"]*n,
        "T_air_std": [None]*n, "RH_std": [None]*n,
        "V_L_measured": [None]*n,
    })

def missing_required_fields(df):
    """FIX-K3: قائمة الحقول الإلزامية الفارغة حسب الشرط."""
    out = []; absent = set()
    for _, r in df.iterrows():
        for col in REQUIRED_BY_COND.get(r.get("cond"), []):
            if col not in df.columns:
                if col not in absent:
                    absent.add(col); out.append(f"عمود مفقود: {col}")
            elif pd.isna(r[col]):
                out.append(f"{r.get('run')}: {col}")
    return out

def load_cal(p):
    """FIX-K12: تجاهل البنود غير الرقمية (batch_date) والـuncertainty الفارغة."""
    p = Path(p)
    if not p.exists(): return {}
    df = pd.read_csv(p); out = {}
    for _, r in df.iterrows():
        try: v = float(r["value"])
        except (TypeError, ValueError): continue
        try: u = float(r.get("uncertainty", 0.0))
        except (TypeError, ValueError): u = 0.0
        if np.isnan(u): u = 0.0
        out[r["item"]] = (v, u)
    return out

def verify_files(runs_csv="runs.csv", cal_csv="calibration.csv",
                 traces_dir="traces", verbose=True):
    issues = []; checks_passed = 0; checks_total = 0; details = {}
    if not Path(runs_csv).exists():
        issues.append(f"{runs_csv} مفقود"); checks_total += 1
    else:
        df = pd.read_csv(runs_csv)
        dist = df["cond"].value_counts().to_dict()
        expected_dist = {"A":5, "B":5, "C":2, "C2":2, "D":3, "E":1}
        if all(dist.get(k,0) == v for k, v in expected_dist.items()):
            if verbose: print(f"  [OK] توزيع runs.csv مطابق: {dist}")
            checks_passed += 1
        else:
            issues.append(f"توزيع الشروط: {dist}")
        checks_total += 1; details["runs_dist"] = dist
    if not Path(cal_csv).exists():
        issues.append(f"{cal_csv} مفقود"); checks_total += 1
    else:
        cal_df = pd.read_csv(cal_csv)
        details["cal_n_items"] = len(cal_df)
        checks_passed += 1; checks_total += 1
    if not Path(traces_dir).exists():
        issues.append(f"{traces_dir} مفقود"); checks_total += 1
    else:
        traces = sorted(Path(traces_dir).glob("*.csv"))
        details["n_traces"] = len(traces)
        if len(traces) >= 4:
            if verbose: print(f"  [OK] traces: {len(traces)} ملف")
            checks_passed += 1
        else:
            issues.append(f"traces: {len(traces)}")
        checks_total += 1
        PRODUCED_DIRS = {"traces_demo", "traces_real"}
        total_fill = 0; total_nan = 0
        nan_in_produced_traces = 0; n_templates_excluded = 0
        for tp in traces:
            in_produced_dir = tp.parent.name.lower() in PRODUCED_DIRS
            is_template_name = ("template" in tp.name.lower()
                                or "blank" in tp.name.lower())
            content = tp.read_text(encoding='utf-8')
            total_fill += content.count("FILL")
            nan_count = content.count("NaN") + content.count("nan")
            total_nan += nan_count
            is_produced = False
            if in_produced_dir and not is_template_name:
                is_produced = True
            if not is_produced:
                for line in content.splitlines():
                    low = line.strip().lower()
                    if low.startswith("#") and ("trace" in low or "produced" in low or "mode" in low):
                        is_produced = True; break
            if not is_produced and not is_template_name:
                try:
                    d_tmp = pd.read_csv(tp, comment='#', nrows=10)
                    cols = set(d_tmp.columns)
                    if {"T_b_C", "EC_raw_uScm"}.issubset(cols): is_produced = True
                except Exception: pass
            if is_produced: nan_in_produced_traces += nan_count
            else:
                if is_template_name: n_templates_excluded += 1
        details["total_fill"] = total_fill; details["total_nan"] = total_nan
        details["nan_in_produced_traces"] = nan_in_produced_traces
        details["n_templates_excluded"] = n_templates_excluded
        if total_fill > 0:
            issues.append(f"traces: FILL={total_fill}")
        if nan_in_produced_traces > 0:
            issues.append(f"traces: NaN={nan_in_produced_traces}")
        if total_fill == 0 and nan_in_produced_traces == 0:
            checks_passed += 1
        checks_total += 1
        if Path(runs_csv).exists():
            df_runs = pd.read_csv(runs_csv)
            missing_traces = [r for r in df_runs["run"]
                              if not (Path(traces_dir)/(r+".csv")).exists()]
            details["missing_traces"] = missing_traces
            if len(missing_traces) == 0: checks_passed += 1
            checks_total += 1
    if Path(runs_csv).exists() and Path("runs_demo.csv").exists():
        h_real = hashlib.md5(Path(runs_csv).read_bytes()).hexdigest()
        h_demo = hashlib.md5(Path("runs_demo.csv").read_bytes()).hexdigest()
        if h_real != h_demo: checks_passed += 1
        else: issues.append("hash مطابق")
        checks_total += 1
    if Path(runs_csv).exists() and Path("runs_template.csv").exists():
        try:
            tpl = pd.read_csv("runs_template.csv")
            df_cur = pd.read_csv(runs_csv)
            measured_cols = ["V_cell_mean", "I_mean_mA", "ec0", "t0",
                             "ecf", "tf", "dpd", "T_b_mean"]
            n_matches = 0
            for col in measured_cols:
                if col in df_cur.columns and col in tpl.columns:
                    cur_v = pd.to_numeric(df_cur[col], errors='coerce').fillna(-999).values
                    tpl_v = pd.to_numeric(tpl[col], errors='coerce').fillna(-999).values
                    if len(cur_v) == len(tpl_v) and (cur_v == tpl_v).all():
                        issues.append(f"{col}: كل القيم = القالب (FILL)")
                        n_matches += 1
            details["template_match_count"] = n_matches
            if n_matches == 0: checks_passed += 1
            checks_total += 1
        except Exception as e:
            issues.append(f"template check error: {str(e)[:50]}")
            checks_total += 1
    if Path(runs_csv).exists():
        try:
            df_req = pd.read_csv(runs_csv)
            miss_req = missing_required_fields(df_req)
            details["missing_required"] = miss_req
            if miss_req:
                issues.append(f"حقول إلزامية فارغة ({len(miss_req)}): "
                              + "; ".join(miss_req[:6]) + (" ..." if len(miss_req) > 6 else ""))
            else:
                checks_passed += 1
            checks_total += 1
            if "dpd_pre" in df_req.columns and "cond" in df_req.columns:
                dp = pd.to_numeric(df_req.loc[df_req["cond"].isin(["C", "C2"]), "dpd_pre"],
                                   errors="coerce")
                if len(dp) > 0 and dp.notna().all() and (dp == 2.00).all():
                    issues.append("dpd_pre = 2.00 تمامًا لكل صفوف C/C2: تأكد أنه قياس DPD فعلي "
                                  "قبل الكهرلة وليس قيمة القالب القديم")
        except Exception as e:
            issues.append(f"required-fields check error: {str(e)[:50]}"); checks_total += 1
    return {"checks_passed": checks_passed, "checks_total": checks_total,
            "issues": issues, "details": details}

print("B9b - verify_files (v2.5)")

def run_analysis(runs_csv, traces_dir=None, cal_csv=None,
                 alpha=None, lod=None, VL=V_L_DEFAULT,
                 T_air=25.0, RH_pct=50.0,
                 K25_eff=K25_EFF_DEFAULT, K_dry=K_DRY_FIXED,
                 pH_0=None, rel_syst=REL_SYST_DEFAULT,
                 out="analysis_summary.json", strict=True, verbose=True):
    assert_mode_is_valid()
    runs_csv = Path(runs_csv)
    if not runs_csv.exists(): raise FileNotFoundError(str(runs_csv))
    df = pd.read_csv(runs_csv)
    miss = [c for c in RUNS_COLUMNS_MANDATORY if c not in df.columns]
    if miss: raise ValueError(f"Missing columns: {miss}")

    if runs_csv.name == "runs.csv":
        demo_path = Path("runs_demo.csv")
        if demo_path.exists():
            h_real = hashlib.md5(runs_csv.read_bytes()).hexdigest()
            h_demo = hashlib.md5(demo_path.read_bytes()).hexdigest()
            if h_real == h_demo:
                raise ValueError("runs.csv مطابق لـruns_demo.csv (hash)!")
            try:
                df_real = pd.read_csv(runs_csv); df_demo = pd.read_csv(demo_path)
                if len(df_real) == len(df_demo):
                    common_cols = [c for c in df_real.columns
                                   if c in df_demo.columns
                                   and c not in ["run", "cond", "day", "ATC_state"]]
                    numeric_cols = [c for c in common_cols
                                    if pd.api.types.is_numeric_dtype(df_real[c])]
                    n_diff_total = 0; n_checked = 0
                    for c in numeric_cols:
                        try:
                            a = pd.to_numeric(df_real[c], errors='coerce').values
                            b = pd.to_numeric(df_demo[c], errors='coerce').values
                            mask = ~(np.isnan(a) | np.isnan(b))
                            if mask.sum() == 0: continue
                            n_checked += 1
                            tol = TOLERANCE.get(c, (1e-6, "default"))[0]
                            n_diff_total += int((np.abs(a[mask] - b[mask]) > tol).sum())
                        except Exception: continue
                    if n_checked > 0 and n_diff_total == 0:
                        raise ValueError("N-12: كل القيم متطابقة (per-row diff = 0)!")
            except (pd.errors.ParserError, pd.errors.EmptyDataError,
                    FileNotFoundError, PermissionError, UnicodeDecodeError):
                pass

    cal = load_cal(cal_csv) if cal_csv else {}
    if alpha is None: alpha = cal.get("alpha", (0.0209, 0.0005))[0]
    if lod is None: lod = cal.get("LOD_DPD", (0.05, 0.01))[0]
    T_RT = cal.get("T_RT", (22.0, 0.5))[0]
    if pH_0 is None:
        pH_0 = cal.get("pH_0", (7.0, 0.1))[0]
    K25_eff_from_cal = cal.get("K25_eff", None)
    K25_eff_source = "default"
    if K25_eff_from_cal is not None:
        K25_eff = K25_eff_from_cal[0]
        K25_eff_source = "from_calibration_csv"
    Qs, Qgs = [], []; miss_tr, nom = [], []
    for r in df["run"]:
        cond = df.loc[df["run"]==r, "cond"].iloc[0]
        tp = Path(traces_dir)/(r+".csv") if traces_dir else None
        if tp and tp.exists():
            Qs.append(charge_from_trace(tp)); Qgs.append(q_gen_from_trace(tp))
        else:
            Qs.append(30.0 if cond in CURRENT_CONDS else 0.0); Qgs.append(np.nan)
            if cond in CURRENT_CONDS:
                nom.append(r)
                if traces_dir and tp and not tp.exists(): miss_tr.append(r)
    if MODE == "REAL" and nom:   # FIX-K10: لا Q اسمي في REAL
        raise ValueError(f"traces مفقودة لتشغيلات تيار {nom}: لا يُسمح بـ Q اسمي (30 C) في REAL.")
    df["Q"] = Qs; df["Q_gen"] = Qgs
    df["k0"] = kappa25(df["ec0"], df["t0"], alpha)
    df["kf"] = kappa25(df["ecf"], df["tf"], alpha)
    df["dk_raw"] = df["kf"] - df["k0"]
    ctrl = df[df["cond"]=="A"]
    drift = float(np.median(ctrl["dk_raw"])) if len(ctrl) >= 2 else 0.0
    df["dk_res"] = df["dk_raw"] - drift
    welch_res = welch_blocked(df, dv="dk_res", group="cond", block="day")
    recC, recC2 = None, None
    if "dpd_pre" in df.columns:
        for c in ["C","C2"]:
            sub = df[df["cond"]==c]
            if len(sub) > 0 and sub["dpd_pre"].notna().all():
                r = (sub["dpd"]*sub["dilution"])/sub["dpd_pre"]
                rec = {"mean": float(r.mean()),
                       "std": float(r.std(ddof=1)) if len(r) > 1 else 0.10,
                       "n": int(len(r))}
                if c == "C": recC = rec
                else: recC2 = rec
    eb, eps_src = eps_prior_from_C_or_C2(recC, recC2)
    B = df[df["cond"]=="B"]
    if len(B) > 0:
        dpd_obs = (B["dpd"]*B["dilution"]).to_numpy()
        sig_dpd = dpd_sigma(lod, B["dilution"].to_numpy(), dpd_obs)
        x1r = infer_x1(dpd_obs, B["Q"].to_numpy(), sig_dpd, VL, eb)
    else:
        dpd_obs = np.array([])
        x1r = {"median": None, "CI95_lo": None, "CI95_hi": None,
               "P_x1_gt_0.005": None, "sigma_x": None,
               "hierarchical": False, "chi2_dof": None, "chi2_dof_pass1": None,
               "eps_fixed_at": EPS_FIXED_VALUE,
               "eps_source": f"fixed_at_{EPS_FIXED_VALUE}",
               "eps_dynamic_mode": EPS_DYNAMIC_MODE}
    if x1r["median"] is not None:
        assert_x1_physical(x1r["median"], mode=MODE)
        ci_lo_f, ci_hi_f = add_systematic_asymmetric(
            x1r["median"], x1r["CI95_lo"], x1r["CI95_hi"], rel_syst)
        x1r["CI95_lo_final"] = ci_lo_f; x1r["CI95_hi_final"] = ci_hi_f
        x1r["rel_syst"] = float(rel_syst); x1r["asymmetric_CI"] = True
    I_avg_A = 0.050
    if len(B) > 0 and "I_mean_mA" in df.columns:
        I_avg_A = float(B["I_mean_mA"].mean())/1000.0
    delta_eff = back_calculate_delta(x1r["median"], I_avg_A)
    _ctx = dict(df=df, B=B, dpd_obs=dpd_obs, x1r=x1r, welch_res=welch_res,
                recC=recC, recC2=recC2, eb=eb, eps_src=eps_src,
                drift=drift, delta_eff=delta_eff, I_avg_A=I_avg_A,
                K25_eff=K25_eff, K25_eff_source=K25_eff_source,
                K_dry=K_dry, T_air=T_air, RH_pct=RH_pct, T_RT=T_RT,
                VL=VL, lod=lod, alpha=alpha, pH_0=pH_0,
                rel_syst=rel_syst, out=out, strict=strict,
                verbose=verbose, miss_tr=miss_tr, nom=nom,
                traces_dir=traces_dir)
    return _run_analysis_part2(_ctx)

print("B10a - run_analysis (part 1) v2.5")

def _run_analysis_part2(c):
    df = c["df"]; B = c["B"]; dpd_obs = c["dpd_obs"]; x1r = c["x1r"]
    welch_res = c["welch_res"]; recC = c["recC"]; recC2 = c["recC2"]
    eb = c["eb"]; eps_src = c["eps_src"]; drift = c["drift"]
    delta_eff = c["delta_eff"]
    K25_eff = c["K25_eff"]; K25_eff_source = c["K25_eff_source"]
    K_dry = c["K_dry"]; T_air = c["T_air"]; RH_pct = c["RH_pct"]
    T_RT = c["T_RT"]; VL = c["VL"]; lod = c["lod"]; alpha = c["alpha"]
    pH_0 = c["pH_0"]; rel_syst = c["rel_syst"]; out = c["out"]
    strict = c["strict"]; verbose = c["verbose"]
    miss_tr = c["miss_tr"]; nom = c["nom"]; traces_dir = c["traces_dir"]
    if K25_eff is None: K25_eff = K25_EFF_DEFAULT
    K25_calibration_note = None; K25_calibration_error = None
    K25_calibration_converged = None; K25_calibration_optimal = None
    K25_fits = []; K25_spread = None
    if traces_dir:
        d_traces = sorted([p for p in Path(traces_dir).glob("D*.csv")
                           if re.match(r"^D_\d+\.csv$", p.name)])
        if len(d_traces) >= 1:
            try:
                cal_multi = calibrate_K25_multi([str(p) for p in d_traces])
                K25_fits = cal_multi["fits"]; K25_spread = cal_multi["spread_rel"]
                if cal_multi["n_converged"] >= 1:
                    K25_eff = cal_multi["K25_median"]
                    K25_calibration_optimal = K25_eff
                    K25_eff_source = "calibrated_from_D_refined"
                    K25_calibration_converged = True
                    sp = "n/a" if K25_spread is None else f"{K25_spread*100:.1f}%"
                    K25_calibration_note = (f"K25_eff={K25_eff:.6e} (median of "
                        f"{cal_multi['n_converged']}/{cal_multi['n_runs']} D runs, spread={sp}).")
                else:
                    K25_calibration_converged = False
                    K25_calibration_error = "K25 did NOT converge in any D run (max err >= 0.1 K)."
            except Exception as e:
                K25_calibration_error = f"D fit error: {e}"
    thermal_ode = None; T_air_used, RH_used = T_air, RH_pct
    env_source = "default"
    K25_eff_used_in_ode = K25_eff if K25_eff is not None else K25_EFF_DEFAULT
    K25_used_in_ode_source = "default"
    if traces_dir and len(B) > 0:
        first_b = B["run"].iloc[0]
        tp = Path(traces_dir)/(first_b+".csv")
        if not tp.exists():
            for b_run in B["run"]:
                tp_try = Path(traces_dir)/(b_run+".csv")
                if tp_try.exists():
                    tp = tp_try; break
        if tp.exists():
            try:
                T_air_trace, RH_trace = read_environment(str(tp), strict=(MODE=="REAL"))
                T_air_used, RH_used = T_air_trace, RH_trace
                env_source = "trace_header"
            except ValueError as e:
                env_source = f"ERROR ({str(e)[:50]})"
                if MODE == "REAL": raise
            K25_for_ode = K25_eff if MODE == "REAL" else K25_EFF_DEFAULT
            K25_eff_used_in_ode = K25_for_ode
            K25_used_in_ode_source = ("calibrated" if MODE=="REAL" else "default")
            thermal_ode = integrate_thermal_ode(str(tp), T_air=T_air_used,
                RH_pct=RH_used, K25_eff=K25_for_ode, K_dry=K_dry)
    max_abs_err_flag = None
    if thermal_ode and "max_abs_err_K" in thermal_ode:
        lo, hi = ACCEPT["max_abs_err_K"]
        if not (lo <= thermal_ode["max_abs_err_K"] <= hi):
            msg = f"max_abs_err_K={thermal_ode['max_abs_err_K']:.4f} خارج [{lo}, {hi}]"
            max_abs_err_flag = msg
            # FIX-K5: تحذير فقط — الحرارة لا تدخل في x1
            warnings.warn(f"{msg} ({MODE})", stacklevel=2)
            print(f"[!] {msg} ({MODE}) — تحذير: فحص الحرارة لا يدخل في x1")
    I_lim_result = None
    if traces_dir:
        e_trace = Path(traces_dir)/"E_1.csv"
        if e_trace.exists():
            try:
                I_lim_result = compute_I_lim_segments(str(e_trace))   # FIX-K2
            except Exception as e:
                I_lim_result = {"error": str(e)}
    CT_eff = None
    if len(B) > 0 and "pH_meas" in df.columns:
        for _, r in B.iterrows():
            if pd.notna(r.get("pH_meas")):
                pH_f_corr = r["pH_meas"]
                if "T_sample" in df.columns and pd.notna(r.get("T_sample")):
                    Tb_r = r["T_b_mean"] if pd.notna(r.get("T_b_mean")) else 25.0
                    pH_f_corr = pH_correct(r["pH_meas"], Tb_r, T_sample=r["T_sample"])
                elif "T_b_mean" in df.columns and pd.notna(r.get("T_b_mean")):
                    pH_f_corr = pH_correct(r["pH_meas"], r["T_b_mean"], T_RT)
                Q_use = r["Q"] if pd.notna(r["Q"]) else 30.0
                CT_eff = solve_charge_balance(x1r["median"], Q_use, VL, pH_0, pH_f_corr)
                break
    Q_mean_traces = float(B["Q"].mean()) if len(B) > 0 else 30.0
    gas_vol = compute_gas_volumes(x1r["median"], Q_mean_traces)
    gas = gas_vol if gas_vol else {"V_H2_mL": None, "V_O2_mL": None, "V_total_mL": None}
    verdict = "inconclusive"
    if x1r["CI95_lo"] is not None:
        if x1r["CI95_lo"] > X1_THRESHOLD:
            verdict = "pathway_Cl_confirmed"
        elif len(dpd_obs) > 0 and np.all(dpd_obs < lod):
            if recC2 and recC2["mean"] >= 0.60:
                verdict = "x1_below_upper95"
            elif recC2 and recC2["mean"] < 0.40:
                verdict = "inconclusive_measurement"
            else:
                verdict = "inconclusive"
    warns = validate_runs(df, VL, strict_mode=(MODE=="REAL"))
    if strict and MODE == "REAL" and warns:
        raise ValueError(f"Acceptance criteria violated: {warns}")
    CT_eff_warning = None
    if CT_eff is not None and MODE == "DEMO":
        CT_eff_warning = "قيد 24: CT_eff غير مستقر بين تشغيلات DEMO."

    # FIX-K7: GUM + MC (data-driven) + مرونات + Outliers  (cross-checks؛ CI المعتمد = x1.CI95_*_final)
    gum_result = None; mc_result = None; sens_result = None; outliers_result = None
    if x1r["median"] is not None:
        try:
            x1_std_approx = (x1r["CI95_hi"] - x1r["CI95_lo"]) / (2 * 1.96)
            gum_result = gum_uncertainty(x1r["median"], x1_std_approx, rel_syst)
        except Exception as e:
            gum_result = {"error": str(e)}
        try:
            sig_B = dpd_sigma(lod, B["dilution"].to_numpy(dtype=float), dpd_obs)
            mc_result = error_propagation_monte_carlo(dpd_obs, B["Q"].to_numpy(dtype=float),
                                                      sig_B, VL, eb, n_samples=10000)
        except Exception as e:
            mc_result = {"error": str(e)}
        try:
            a_, b_ = eb
            sens_result = sensitivity_elasticities(a_/(a_+b_))
        except Exception as e:
            sens_result = {"error": str(e)}
    if len(B) > 0 and "dk_res" in B.columns:
        try:
            dk_vals = B["dk_res"].dropna().to_numpy()
            if len(dk_vals) > 0:
                outliers_result = {"grubbs": detect_outliers_grubbs(dk_vals),
                                   "dixon": detect_outliers_dixon(dk_vals)}
        except Exception as e:
            outliers_result = {"error": str(e)}

    try:
        c2_check = c2_consistency_check(df, x1r, recC, VL, lod)
    except Exception as e:
        c2_check = {"available": False, "error": str(e)[:80]}

    # FIX-J4: K25 bias في diagnostics
    K25_bias_pct = None
    if K25_calibration_optimal is not None:
        K25_bias_pct = float(abs(K25_calibration_optimal - K25_EFF_DEFAULT) / K25_EFF_DEFAULT * 100)

    summ = {
        "protocol": PROTOCOL_LABEL, "model": MODEL_LABEL,
        "code_version": VERSION, "mode": MODE,
        "reference_type": "REAL" if MODE=="REAL" else "DEMO",
        "alpha_per_K": alpha, "LOD_DPD": lod,
        "V_L": VL, "T_RT": T_RT, "pH_0_used": pH_0,
        "K_dry_used_W_per_K": float(K_dry),
        "K25_eff_used_m_s": float(K25_eff),
        "K25_eff_used_in_ode_m_s": float(K25_eff_used_in_ode),
        "K25_used_in_ode_source": K25_used_in_ode_source,
        "K25_eff_source": K25_eff_source, "environment_source": env_source,
        "K25_bias_pct": K25_bias_pct,
        "T_air_used": float(T_air_used), "RH_used": float(RH_used),
        "n_A": welch_res.get("n_A", 0), "n_B": welch_res.get("n_B", 0),
        "drift_control": drift, "welch": welch_res,
        "recovery_C": recC, "recovery_C2": recC2,
        "recovery_C2_note": "خام وغير مصحَّح للـCl2 الناتج كهربائيًا؛ استخدم C2_consistency",
        "C2_consistency": c2_check,
        "eps_source": eps_src, "eps_beta_prior": {"a": eb[0], "b": eb[1]},
        "x1": x1r,
        "six_events": {
            "1_electrochemistry": {"x1_median": x1r["median"],
                "x1_CI95_bayes": [x1r["CI95_lo"], x1r["CI95_hi"]],
                "x1_CI95_final": [x1r.get("CI95_lo_final"), x1r.get("CI95_hi_final")],
                "Q_mean_C": Q_mean_traces},
            "2_mass_transfer": {"delta_eff_cm": delta_eff,
                "note": "derived from x1 (not independent)"},
            "3_thermal": thermal_ode,
            "4_conductivity": {
                "dk_res_B_mean": float(B["dk_res"].mean()) if len(B) > 0 else None,
                "welch_p": welch_res.get("p_reported")},
            "5_acid_base": {"CT_eff_M": CT_eff, "warning": CT_eff_warning},
            "6_gas": {"V_H2_mL": gas["V_H2_mL"], "V_O2_mL": gas["V_O2_mL"],
                "V_total_mL": gas["V_total_mL"], "Q_used_C": Q_mean_traces},
            "7_I_lim": I_lim_result},
        "analysis_I": {
            "note": "cross-checks فقط؛ CI المعتمد = x1.CI95_lo_final / CI95_hi_final",
            "I2_GUM": gum_result,
            "I3_MonteCarlo": mc_result,
            "I5_Sensitivity": sens_result,
            "I6_Outliers": outliers_result,
        },
        "diagnostics": {
            "missing_traces": miss_tr, "used_nominal_Q_for": nom,
            "validation_warnings": warns,
            "statsmodels_available": HAS_SM,
            "statsmodels_formula_available": HAS_SMF,
            "strict_mode": strict,
            "K25_calibration_note": K25_calibration_note,
            "K25_calibration_error": K25_calibration_error,
            "K25_calibration_converged": K25_calibration_converged,
            "K25_calibration_optimal": K25_calibration_optimal,
            "K25_fits": K25_fits, "K25_spread_rel": K25_spread,
            "K25_calibration_method": "RMS fit لكل D ثم الوسيط؛ التقارب: max err < 0.1 K",
            "K25_in_ode_note": "REAL يستخدم K25 المُعايَر. DEMO يستخدم K25_EFF_DEFAULT.",
            "K25_note": "K25 لا يدخل في x1؛ يؤثر فقط في فحص الحرارة، وغير قابل للتعريف عند خطأ K_dry.",
            "max_abs_err_flag": max_abs_err_flag,
            "R5_range": list(R5_RANGE),
            "R8_ec_drift_max": R8_EC_DRIFT_MAX,
            "R8_tb_range": list(R8_TB_RANGE),
            "checks": {
                "I_lim_plateau_ok": (bool(I_lim_result.get("plateau"))
                                     if (I_lim_result and "error" not in I_lim_result) else None),
                "K25_converged": K25_calibration_converged,
                "K25_spread_ok": ((K25_spread < 0.20) if K25_spread is not None else None),
                "thermal_check_ok": ((max_abs_err_flag is None) if thermal_ode else None),
                "recovery_C_present": recC is not None,
                "all_current_traces_present": (len(nom) == 0),
                "validation_warnings_none": (len(warns) == 0),
                "x1_chi2_ok": (x1r.get("chi2_dof") is None or x1r["chi2_dof"] <= 2.0),
                "C2_consistent": (c2_check.get("consistent") if c2_check.get("available") else None),
            },
            "limitations_open": [
                "LIM-1: eps ثابتة (0.15) في تشخيص chi2 فقط",
                "LIM-4: x1 من B فقط (n=5)",
                "LIM-5: delta_eff مشتق من x1 (غير مستقل)",
                "LIM-6: K25 غير قابل للتعريف دون K_dry",
                "LIM-7: I_lim من 3 جهود فقط (2/3/4 V)",
                "LIM-8: استرجاع C2 الخام غير مصحَّح؛ يُستعمل فحص الاتساق K13 (قدرته محدودة بـ n=2)"]},
        "verdict": verdict}
    Path(out).write_text(json.dumps(summ, indent=2, ensure_ascii=False, default=str))
    print(f"[OK] Saved: {Path(out).resolve()}")
    if verbose: print(json.dumps(summ, indent=2, ensure_ascii=False, default=str))
    return summ

print("B10b - run_analysis (part 2) v2.5")

def make_demo_data(x1_true=0.051, seed=42, n_A=5, n_B=5, n_C=2, n_C2=2, n_D=3):
    np.random.seed(seed); rows = []
    alpha_k = 0.0209; dk_res_raw_target = -2.84
    def _ecf(ec0, t0, tf, dk):
        k0 = ec0/(1+alpha_k*(t0-25)); kf = k0+dk
        return kf*(1+alpha_k*(tf-25))
    T_b_base = 25.35; T_sample_base = 23.5
    pH_in_situ_true = 8.5 + np.random.normal(0, 0.05)
    for i in range(1, n_A+1):
        rows.append({"run": f"A_{i}", "cond": "A", "day": 1 if i<=3 else 2,
            "ec0": 799.6+np.random.normal(0,0.3), "t0": 25.03+np.random.normal(0,0.02),
            "ecf": 799.4+np.random.normal(0,0.3), "tf": 25.05+np.random.normal(0,0.02),
            "dpd": max(0, np.random.normal(0.02,0.01)), "dilution": 1,
            "charge_C": 0, "dpd_pre": np.nan, "V_cell_mean": 0, "I_mean_mA": 0,
            "T_b_mean": 25.03+np.random.normal(0,0.05), "pH_meas": np.nan,
            "T_sample": np.nan, "ATC_state": "OFF", "T_air_std": 0.14,
            "RH_std": 1.8, "V_L_measured": 0.100+np.random.normal(0,0.0001)})
    for i in range(1, n_B+1):
        sig_true = dpd_expected(x1_true, 0.15, 30.0)
        sig_obs = sig_true + np.random.normal(0, sig_true*0.02)
        ec0 = 799.5+np.random.normal(0,0.3); t0 = 25.02+np.random.normal(0,0.02)
        tf = 25.55+np.random.normal(0,0.02)
        T_b_mean_val = T_b_base+np.random.normal(0,0.05)
        T_sample_val = T_sample_base+np.random.normal(0,0.1)
        pH_meas_val = (pH_in_situ_true + 0.008*(T_b_mean_val-T_sample_val)
                       + np.random.normal(0,0.005))
        rows.append({"run": f"B_{i}", "cond": "B", "day": 1 if i<=3 else 2,
            "ec0": ec0, "t0": t0, "ecf": _ecf(ec0,t0,tf,dk_res_raw_target), "tf": tf,
            "dpd": sig_obs/3.0, "dilution": 3,
            "charge_C": 30.0+np.random.normal(0,0.05), "dpd_pre": np.nan,
            "V_cell_mean": 12.13+np.random.normal(0,0.05),
            "I_mean_mA": 50.02+np.random.normal(0,0.05),
            "T_b_mean": T_b_mean_val, "pH_meas": pH_meas_val,
            "T_sample": T_sample_val, "ATC_state": "OFF", "T_air_std": 0.14,
            "RH_std": 1.8, "V_L_measured": 0.100+np.random.normal(0,0.0001)})
    for i in range(1, n_C+1):
        dpd_c_val = 2.0*(0.855+np.random.normal(0,0.02))
        rec_c = dpd_c_val/2.0
        if not (0.10 < rec_c < 0.99):
            rec_c = float(np.clip(rec_c, 0.10, 0.99)); dpd_c_val = rec_c*2.0
        rows.append({"run": f"C_{i}", "cond": "C", "day": 2,
            "ec0": 799.7, "t0": 25.01, "ecf": 799.7, "tf": 25.02,
            "dpd": dpd_c_val, "dilution": 1, "charge_C": 0, "dpd_pre": 2.00,
            "V_cell_mean": 0, "I_mean_mA": 0, "T_b_mean": 25.01, "pH_meas": np.nan,
            "T_sample": np.nan, "ATC_state": "OFF", "T_air_std": 0.14,
            "RH_std": 1.8, "V_L_measured": 0.100})
    for i in range(1, n_C2+1):
        # FIX-K4: لا قصّ عند 0.99 — استرجاع ~0.90 مع x1 مهمل في C2
        sig_true_c2 = 0.855*(2.0 + dpd_expected(x1_true, 0.0, 30.0))   # FIX-K13: متسق مع x1_true
        sig_obs = sig_true_c2 + np.random.normal(0, 0.03)
        ec0 = 79.97+np.random.normal(0,0.05); t0 = 25.03+np.random.normal(0,0.02)
        tf = 25.55+np.random.normal(0,0.02)
        dpd_c2_val = sig_obs/5.0
        rows.append({"run": f"C2_{i}", "cond": "C2", "day": 2,
            "ec0": ec0, "t0": t0, "ecf": _ecf(ec0,t0,tf,dk_res_raw_target/10), "tf": tf,
            "dpd": dpd_c2_val, "dilution": 5,
            "charge_C": 30.0+np.random.normal(0,0.05), "dpd_pre": 2.00,
            "V_cell_mean": 12.13, "I_mean_mA": 50.01, "T_b_mean": 25.35,
            "pH_meas": np.nan, "T_sample": np.nan, "ATC_state": "OFF",
            "T_air_std": 0.14, "RH_std": 1.8, "V_L_measured": 0.100})
    for i in range(1, n_D+1):
        rows.append({"run": f"D_{i}", "cond": "D", "day": 1 if i==1 else 2,
            "ec0": 799.6, "t0": 35.01, "ecf": 799.6, "tf": 25.10, "dpd": 0.02,
            "dilution": 1, "charge_C": 0, "dpd_pre": np.nan, "V_cell_mean": 0,
            "I_mean_mA": 0, "T_b_mean": 30.00, "pH_meas": np.nan,
            "T_sample": np.nan, "ATC_state": "OFF", "T_air_std": 0.14,
            "RH_std": 1.8, "V_L_measured": 0.100})
    rows.append({"run": "E_1", "cond": "E", "day": 2,
        "ec0": 799.6, "t0": 25.02, "ecf": 799.5, "tf": 25.55, "dpd": 0.02,
        "dilution": 1, "charge_C": 0, "dpd_pre": np.nan, "V_cell_mean": 3.0,
        "I_mean_mA": np.nan, "T_b_mean": 25.35, "pH_meas": np.nan,
        "T_sample": np.nan, "ATC_state": "OFF", "T_air_std": 0.14,
        "RH_std": 1.8, "V_L_measured": 0.100})
    return pd.DataFrame(rows)

print("B11 - make_demo_data")

def _generate_demo_traces(traces_dir, demo_df, seed=42):
    """FIX-J2: تستخدم thermal_model_energy_balance الموحّدة."""
    np.random.seed(seed+1)
    t = np.linspace(-60, 660, 721)
    T_air_demo, RH_demo = 24.8, 51.2
    K25_demo = K25_EFF_DEFAULT
    for _, row in demo_df.iterrows():
        cond = row["cond"]
        if cond in ("B","C2"):
            I_mA = np.where((t>=0)&(t<=600), 50.0, 0.0)
            V_V = np.where(I_mA>0, 12.13, 0.0)
            t_pos = t - t[0]; I_pos = I_mA/1000.0; V_pos = V_V
            def _dTb(t_now, T_b):
                return thermal_model_energy_balance(
                    t_now, T_b, t_pos, I_pos, V_pos,
                    T_air_demo, RH_demo, K25_demo, K_DRY_FIXED, C_TOTAL
                )
            T0 = float(row["t0"])
            # FIX-K4: الزمن النسبي t_pos (كان t الحقيقي => إزاحة 60 ث = 0.067 K)
            sol = solve_ivp(_dTb, [t_pos[0], t_pos[-1]], [T0], t_eval=t_pos,
                            method='RK45', rtol=1e-9, atol=1e-11)
            T_b = sol.y[0] + np.random.normal(0, 0.005, len(t))
            EC = np.where(I_mA>0, 799.5-0.0047*np.maximum(t,0), 799.5)
            EC += np.random.normal(0, 0.05, len(t))
            header = "# T_air_actual = 24.8 C\n# RH_actual = 51.2 %\n"
            with open(traces_dir/f"{row['run']}.csv", "w") as f:
                f.write(header)
                pd.DataFrame({"t_s": t, "I_mA": I_mA, "V_V": V_V,
                              "T_b_C": T_b, "EC_raw_uScm": EC}).to_csv(f, index=False)
        elif cond == "D":
            t_D = np.linspace(0, 1800, 1801); T0 = 35.0
            def _dTb_demo(t_now, T_b):
                return thermal_model_energy_balance(
                    t_now, T_b, None, None, None,
                    T_air_demo, RH_demo, K25_demo, K_DRY_FIXED, C_TOTAL
                )
            sol = solve_ivp(_dTb_demo, [0,1800], [T0], t_eval=t_D,
                            method='RK45', rtol=1e-10, atol=1e-12)
            T_b_D = sol.y[0] + np.random.normal(0, 0.02, len(t_D))
            header = "# T_air_actual = 24.8 C\n# RH_actual = 51.2 %\n"
            with open(traces_dir/f"{row['run']}.csv", "w") as f:
                f.write(header)
                pd.DataFrame({"t_s": t_D, "I_mA": np.zeros_like(t_D),
                              "V_V": np.zeros_like(t_D), "T_b_C": T_b_D,
                              "EC_raw_uScm": 799.6+np.random.normal(0,0.05,len(t_D))
                              }).to_csv(f, index=False)
        elif cond == "E":
            t_E = np.linspace(-60, 960, 1021)
            # FIX-K4/K2: منحنى واقعي — عابر بعد كل خطوة جهد (tau=40 s) وهضبة عند ~15 mA
            I_mA = np.zeros_like(t_E); _prev = 0.0
            for _a, _b, _Ip in [(0.0, 300.0, 10.0), (300.0, 600.0, 15.0), (600.0, 900.0001, 15.1)]:
                _m = (t_E >= _a) & (t_E < _b)
                I_mA[_m] = _Ip + (_prev - _Ip) * np.exp(-(t_E[_m] - _a) / 40.0)
                _prev = _Ip
            I_mA = I_mA + np.random.normal(0, 0.15, len(t_E))
            I_mA = np.where((t_E<0) | (t_E>900), 0.0, I_mA)
            V_V = np.where((t_E>=0)&(t_E<300), 2.0,
                    np.where((t_E>=300)&(t_E<600), 3.0,
                    np.where((t_E>=600)&(t_E<=900), 4.0, 0.0)))
            T_b = np.linspace(25.02, 25.30, 1021)+np.random.normal(0,0.005,1021)
            EC = np.where(I_mA>0, 799.5, 799.6)+np.random.normal(0,0.05,1021)
            header = "# T_air_actual = 24.8 C\n# RH_actual = 51.2 %\n"
            with open(traces_dir/f"{row['run']}.csv", "w") as f:
                f.write(header)
                pd.DataFrame({"t_s": t_E, "I_mA": I_mA, "V_V": V_V,
                              "T_b_C": T_b, "EC_raw_uScm": EC}).to_csv(f, index=False)
        elif cond == "A":
            t_A = np.linspace(-60, 660, 721)
            I_mA_A = np.zeros_like(t_A); V_V_A = np.zeros_like(t_A)
            T_b_A = np.full_like(t_A, 25.0) + np.random.normal(0, 0.005, len(t_A))
            EC_A = np.full_like(t_A, 799.6) + np.random.normal(0, 0.05, len(t_A))
            header = "# T_air_actual = 24.8 C\n# RH_actual = 51.2 %\n"
            with open(traces_dir/f"{row['run']}.csv", "w") as f:
                f.write(header)
                pd.DataFrame({"t_s": t_A, "I_mA": I_mA_A, "V_V": V_V_A,
                              "T_b_C": T_b_A, "EC_raw_uScm": EC_A}).to_csv(f, index=False)
        elif cond == "C":
            t_C = np.linspace(0, 300, 301)
            I_mA_C = np.zeros_like(t_C); V_V_C = np.zeros_like(t_C)
            T_b_C = np.full_like(t_C, 25.0) + np.random.normal(0, 0.005, len(t_C))
            EC_C = np.full_like(t_C, 799.6) + np.random.normal(0, 0.05, len(t_C))
            header = "# T_air_actual = 24.8 C\n# RH_actual = 51.2 %\n"
            with open(traces_dir/f"{row['run']}.csv", "w") as f:
                f.write(header)
                pd.DataFrame({"t_s": t_C, "I_mA": I_mA_C, "V_V": V_V_C,
                              "T_b_C": T_b_C, "EC_raw_uScm": EC_C}).to_csv(f, index=False)

def save_demo_ref(seed=42):
    global MODE
    MODE = "DEMO"
    demo = make_demo_data(seed=seed)
    demo.to_csv("runs_demo.csv", index=False)
    print(f"[OK] Created runs_demo.csv ({len(demo)} rows)")
    traces_dir = Path("traces_demo"); traces_dir.mkdir(exist_ok=True)
    _generate_demo_traces(traces_dir, demo, seed=seed)
    n_traces = len(list(traces_dir.glob("*.csv")))
    print(f"[OK] Created {n_traces} traces")
    result = run_analysis("runs_demo.csv", traces_dir=str(traces_dir),
                          cal_csv=None, out="reference_demo_summary.json",
                          strict=False, verbose=False)
    result["reference_type"] = "DEMO"
    Path("reference_demo_summary.json").write_text(
        json.dumps(result, indent=2, ensure_ascii=False, default=str))
    return result

print("B12 - traces + save_demo_ref (v2.5)")

# ============= استكمال الخلية 1 — الجزء 4 (Stress tests + main) =============

def _test_K25_direct():
    from pathlib import Path; import tempfile
    t = np.linspace(0, 1800, 1801); T_air_true = 24.8; RH_true = 51.2
    K25_true = K25_EFF_DEFAULT; T0 = 35.0
    def _dTb(t_now, T_b):
        return thermal_model_energy_balance(
            t_now, T_b, None, None, None,
            T_air_true, RH_true, K25_true, K_DRY_FIXED, C_TOTAL
        )
    sol = solve_ivp(_dTb, [0,1800], [T0], t_eval=t, method='RK45', rtol=1e-10, atol=1e-12)
    with tempfile.TemporaryDirectory() as tmp:
        p = Path(tmp)/"D_test.csv"
        with open(p, "w") as f:
            f.write(f"# T_air_actual = {T_air_true} C\n# RH_actual = {RH_true} %\n")
            pd.DataFrame({"t_s": t, "I_mA": np.zeros_like(t), "V_V": np.zeros_like(t),
                          "T_b_C": sol.y[0], "EC_raw_uScm": np.full_like(t,799.6)}).to_csv(f, index=False)
        K25_fit, err, conv = calibrate_K25_from_D_refined(str(p))
        if not conv: return [("K25 direct", "[X] not converged", f"err={err:.4f}")]
        rel_err = abs(K25_fit-K25_true)/K25_true
        if rel_err < 0.05:
            return [("K25 direct", f"[OK] recovered {K25_fit:.6e}", f"rel_err={rel_err*100:.2f}%")]
        return [("K25 direct", f"[!] rel_err={rel_err*100:.2f}%", "")]

def _test_K25_realistic():
    """معلوماتي: K25 غير قابل للتعريف عندما يكون K_dry خاطئًا 15% (LIM-6)."""
    from pathlib import Path; import tempfile
    rng = np.random.default_rng(seed=123)
    t = np.linspace(0, 1800, 1801); T_air_true = 24.8; RH_true = 51.2
    K25_true = K25_EFF_DEFAULT; K_dry_sim = K_DRY_FIXED*1.15
    C_tot_sim = C_TOTAL*1.05; T0 = 35.0
    def _dTb_sim(t_now, T_b):
        return thermal_model_energy_balance(
            t_now, T_b, None, None, None,
            T_air_true, RH_true, K25_true, K_dry_sim, C_tot_sim
        )
    sol = solve_ivp(_dTb_sim, [0,1800], [T0], t_eval=t, method='RK45', rtol=1e-10, atol=1e-12)
    T_b_noisy = sol.y[0] + rng.normal(0, 0.02, len(t))
    with tempfile.TemporaryDirectory() as tmp:
        p = Path(tmp)/"D_realistic.csv"
        with open(p, "w") as f:
            f.write(f"# T_air_actual = {T_air_true} C\n# RH_actual = {RH_true} %\n")
            pd.DataFrame({"t_s": t, "I_mA": np.zeros_like(t), "V_V": np.zeros_like(t),
                          "T_b_C": T_b_noisy, "EC_raw_uScm": np.full_like(t,799.6)}).to_csv(f, index=False)
        K25_fit, err, conv = calibrate_K25_from_D_refined(str(p))
        if K25_fit is None: return [("K25 realistic", "[i] no fit", "LIM-6")]
        rel_err = abs(K25_fit-K25_true)/K25_true
        return [("K25 realistic", f"[i] rel_err={rel_err*100:.1f}% (K_dry+15%)", "LIM-6: K25 غير قابل للتعريف")]

def _test_demo_circular_behavior():
    global MODE
    old_mode = MODE; results = []; MODE = "DEMO"
    try:
        try: assert_not_demo_for_citation(); results.append(("DEMO guard", "[X]", "expected"))
        except ValueError: results.append(("DEMO guard", "[OK] رفع", "correct"))
        try: assert_not_demo_for_citation(summary={"reference_type":"DEMO"})
        except ValueError: results.append(("DEMO guard summary", "[OK] رفع", "correct"))
        MODE = "REAL"
        try:
            assert_not_demo_for_citation(summary={"reference_type":"REAL"})
            results.append(("REAL guard", "[OK]", "correct"))
        except ValueError: results.append(("REAL guard", "[X]", "unexpected"))
    finally: MODE = old_mode
    return results

def _test_verify_files_integration():
    from pathlib import Path; import tempfile
    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        pd.DataFrame({"cond":["A"]*5+["B"]*5+["C"]*2+["C2"]*2+["D"]*3+["E"]*1,
                      "run":[f"r{i}" for i in range(18)]}).to_csv(tmp/"runs.csv", index=False)
        pd.DataFrame({"item":["alpha"],"value":[0.0209]}).to_csv(tmp/"calibration.csv", index=False)
        traces = tmp/"traces"; traces.mkdir()
        for i in range(7):
            (traces/f"trace_{i}.csv").write_text("t_s,I_mA\n"+("0,0\n"*200)+"FILL,FILL\n"+("0,0\n"*100))
        pd.DataFrame({"cond":["A"]*2,"run":["x","y"]}).to_csv(tmp/"runs_demo.csv", index=False)
        import os; old_cwd = os.getcwd(); os.chdir(tmp)
        try:
            res = verify_files(runs_csv="runs.csv", cal_csv="calibration.csv",
                               traces_dir="traces", verbose=False)
        finally: os.chdir(old_cwd)
        fill_detected = res["details"].get("total_fill", 0)
        if fill_detected == 14:
            return [("verify_files FIX-D", f"[OK] FILL={fill_detected}", "correct")]
        return [("verify_files FIX-D", f"[!] FILL={fill_detected}", "check")]

def _test_FIX_D_template_exclusion():
    from pathlib import Path
    import tempfile, os
    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        pd.DataFrame({"cond":["A"]*5+["B"]*5+["C"]*2+["C2"]*2+["D"]*3+["E"]*1,
                      "run":[f"r{i}" for i in range(18)]}).to_csv(tmp/"runs.csv", index=False)
        pd.DataFrame({"item":["alpha"],"value":[0.0209]}).to_csv(tmp/"calibration.csv", index=False)
        traces = tmp/"traces"; traces.mkdir()
        for i in range(7):
            (traces/f"trace_{i}.csv").write_text("t_s,I_mA\n0,0\n")
        (traces/"E_template.csv").write_text(
            "# template\n"+"t_s,I_mA,T_b_C,EC_raw_uScm\n"+
            ",".join(["NaN"]*4)+"\n"*50)
        pd.DataFrame({"cond":["A"]*2,"run":["x","y"]}).to_csv(tmp/"runs_demo.csv", index=False)
        old_cwd = os.getcwd(); os.chdir(tmp)
        try:
            res = verify_files(runs_csv="runs.csv", cal_csv="calibration.csv",
                               traces_dir="traces", verbose=False)
        finally: os.chdir(old_cwd)
        if res["details"].get("n_templates_excluded", 0) >= 1 and res["details"].get("nan_in_produced_traces", 0) == 0:
            return [("FIX-D template", f"[OK]", "correct")]
        return [("FIX-D template", f"[!]", "check")]

def _test_FIX_D2_template_check():
    import tempfile, os
    from pathlib import Path
    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        tpl = pd.DataFrame({
            "run":["A_1"],"cond":["A"],"day":[1],
            "ec0":[None],"t0":[None],"ecf":[None],"tf":[None],
            "dpd":[None],"dilution":[1],"charge_C":[0],"dpd_pre":[None],
            "V_cell_mean":[0],"I_mean_mA":[None],"T_b_mean":[None],
            "pH_meas":[None],"T_sample":[None],"ATC_state":["OFF"],
            "T_air_std":[None],"RH_std":[None],"V_L_measured":[0.100]})
        tpl.to_csv(tmp/"runs_template.csv", index=False)
        tpl.to_csv(tmp/"runs.csv", index=False)
        pd.DataFrame({"item":["alpha"],"value":[0.0209]}).to_csv(tmp/"calibration.csv", index=False)
        traces = tmp/"traces"; traces.mkdir()
        (traces/"A_1.csv").write_text("# trace\n# produced\nt_s,I_mA,V_V,T_b_C,EC_raw_uScm\n0,0,0,25,799\n")
        pd.DataFrame({"run":["A_1"],"cond":["A"]}).to_csv(tmp/"runs_demo.csv", index=False)
        old_cwd = os.getcwd(); os.chdir(tmp)
        try:
            res = verify_files(runs_csv="runs.csv", cal_csv="calibration.csv",
                               traces_dir="traces", verbose=False)
        finally: os.chdir(old_cwd)
        if any("FILL" in str(i) for i in res["issues"]):
            return [("FIX-D2 template match", "[OK] رفض", "correct")]
        return [("FIX-D2 template match", f"[!]", "check")]

def _test_compute_I_lim_full():
    from pathlib import Path; import tempfile
    results = []
    with tempfile.TemporaryDirectory() as tmp:
        p = Path(tmp)/"E_test.csv"
        t = np.linspace(0, 600, 601)
        rng = np.random.default_rng(seed=99)
        I_mA = np.where((t>=0)&(t<=600), 15.0, 0.0) + rng.normal(0, 0.1, 601)
        with open(p, "w") as f:
            f.write("# T_air_actual = 24.8 C\n# RH_actual = 51.2 %\n")
            pd.DataFrame({"t_s": t, "I_mA": I_mA, "V_V": np.full_like(t, 3.0),
                          "T_b_C": np.full_like(t, 25.0),
                          "EC_raw_uScm": np.full_like(t, 799.6)}).to_csv(f, index=False)
        I_lim, _, _, warn = compute_I_lim(str(p), t_start=300.0, t_end=600.0)
        if I_lim is not None and 14.5 <= I_lim <= 15.5:
            results.append(("compute_I_lim نظيف", f"[OK] {I_lim:.2f}", "correct"))
        else:
            results.append(("compute_I_lim نظيف", f"[X]", "unexpected"))
    with tempfile.TemporaryDirectory() as tmp:
        p = Path(tmp)/"E_all_nan.csv"
        t = np.linspace(0, 600, 601)
        with open(p, "w") as f:
            f.write("# T_air_actual = 24.8 C\n# RH_actual = 51.2 %\n")
            pd.DataFrame({"t_s": t, "I_mA": np.full_like(t, np.nan),
                          "V_V": np.full_like(t, 3.0),
                          "T_b_C": np.full_like(t, 25.0),
                          "EC_raw_uScm": np.full_like(t, 799.6)}).to_csv(f, index=False)
        I_lim, _, _, warn = compute_I_lim(str(p))
        if I_lim is None:
            results.append(("compute_I_lim all-NaN", "[OK] None", "correct"))
        else:
            results.append(("compute_I_lim all-NaN", f"[X]", "unexpected"))
    with tempfile.TemporaryDirectory() as tmp:
        p = Path(tmp)/"E_neg.csv"
        t = np.linspace(0, 600, 601)
        I_mA = np.where((t>=0)&(t<=600), -5.0, 0.0)
        with open(p, "w") as f:
            f.write("# T_air_actual = 24.8 C\n# RH_actual = 51.2 %\n")
            pd.DataFrame({"t_s": t, "I_mA": I_mA, "V_V": np.full_like(t, 3.0),
                          "T_b_C": np.full_like(t, 25.0),
                          "EC_raw_uScm": np.full_like(t, 799.6)}).to_csv(f, index=False)
        _, _, _, warn = compute_I_lim(str(p))
        if warn and "negative_I_lim" in warn:
            results.append(("compute_I_lim negative", "[OK]", "correct"))
        else:
            results.append(("compute_I_lim negative", f"[X]", "unexpected"))
    with tempfile.TemporaryDirectory() as tmp:
        p = Path(tmp)/"E_zero.csv"
        t = np.linspace(0, 600, 601)
        I_mA = np.where((t>=0)&(t<=600), 1e-9, 0.0)
        with open(p, "w") as f:
            f.write("# T_air_actual = 24.8 C\n# RH_actual = 51.2 %\n")
            pd.DataFrame({"t_s": t, "I_mA": I_mA, "V_V": np.full_like(t, 3.0),
                          "T_b_C": np.full_like(t, 25.0),
                          "EC_raw_uScm": np.full_like(t, 799.6)}).to_csv(f, index=False)
        _, _, _, warn = compute_I_lim(str(p))
        if warn and "I_lim_near_zero" in warn:
            results.append(("compute_I_lim near_zero", "[OK]", "correct"))
        else:
            results.append(("compute_I_lim near_zero", f"[X]", "unexpected"))
    with tempfile.TemporaryDirectory() as tmp:
        p = Path(tmp)/"E_noisy.csv"
        t = np.linspace(0, 600, 601)
        rng = np.random.default_rng(seed=11)
        I_mA = np.where((t>=0)&(t<=600), 15.0+rng.normal(0,5.0,601), 0.0)
        with open(p, "w") as f:
            f.write("# T_air_actual = 24.8 C\n# RH_actual = 51.2 %\n")
            pd.DataFrame({"t_s": t, "I_mA": I_mA, "V_V": np.full_like(t, 3.0),
                          "T_b_C": np.full_like(t, 25.0),
                          "EC_raw_uScm": np.full_like(t, 799.6)}).to_csv(f, index=False)
        _, _, _, warn = compute_I_lim(str(p))
        if warn and "high_rel_std" in warn:
            results.append(("compute_I_lim high_rel_std", "[OK]", "correct"))
        else:
            results.append(("compute_I_lim high_rel_std", f"[!]", "check"))
    return results

def _test_beta_from_rec_raises():
    results = []
    for r_val, name in [(1.5, "r>1"), (-0.1, "r<=0"),
                        (0.995, "e<0.01"), (0.005, "e>0.99")]:
        try:
            beta_from_rec(r=r_val, sr=0.10)
            results.append((f"beta_from_rec {name}", "[X]", "expected"))
        except ValueError:
            results.append((f"beta_from_rec {name}", "[OK] رفع", "correct"))
    return results

def _test_N12_per_row_diff_behavioral():
    import tempfile, os
    from pathlib import Path
    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        demo = pd.DataFrame({"run":["B_1"],"cond":["B"],"day":[1],
            "ec0":[799.5],"t0":[25.02],"ecf":[796.7],"tf":[25.55],
            "dpd":[1.71],"dilution":[3],"dpd_pre":[np.nan],
            "charge_C":[30.0],"V_cell_mean":[12.13],"I_mean_mA":[50.0],
            "T_b_mean":[25.35],"pH_meas":[np.nan],"T_sample":[np.nan],
            "ATC_state":["OFF"],"T_air_std":[0.14],"RH_std":[1.8],
            "V_L_measured":[0.100]})
        demo.to_csv(tmp/"runs_demo.csv", index=False)
        real = demo.copy(); real.loc[0,"ec0"] = 804.5
        real.to_csv(tmp/"runs.csv", index=False)
        traces = tmp/"traces"; traces.mkdir()
        t = np.linspace(0, 600, 601)
        with open(traces/"B_1.csv", "w") as f:
            f.write("# T_air_actual = 24.8 C\n# RH_actual = 51.2 %\n")
            pd.DataFrame({"t_s": t, "I_mA": np.full_like(t, 50.0),
                          "V_V": np.full_like(t, 12.13),
                          "T_b_C": np.linspace(25.0, 25.5, 601),
                          "EC_raw_uScm": np.full_like(t, 799.5)}).to_csv(f, index=False)
        old_cwd = os.getcwd(); os.chdir(tmp)
        try:
            try:
                run_analysis("runs.csv", traces_dir="traces", out="test.json",
                             strict=False, verbose=False)
                return [("N-12 per-row diff", "[OK] مرّ", "correct")]
            except ValueError as e:
                if "N-12" in str(e):
                    return [("N-12 per-row diff", "[X] رفض", "unexpected")]
                return [("N-12 per-row diff", f"[!]", "check")]
        finally: os.chdir(old_cwd)

def _test_N12_full_match_raises():
    import tempfile, os
    from pathlib import Path
    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        demo = pd.DataFrame({"run":["B_1"],"cond":["B"],"day":[1],
            "ec0":[799.5],"t0":[25.02],"ecf":[796.7],"tf":[25.55],
            "dpd":[1.71],"dilution":[3],"dpd_pre":[np.nan],
            "charge_C":[30.0],"V_cell_mean":[12.13],"I_mean_mA":[50.0],
            "T_b_mean":[25.35],"pH_meas":[np.nan],"T_sample":[np.nan],
            "ATC_state":["OFF"],"T_air_std":[0.14],"RH_std":[1.8],
            "V_L_measured":[0.100]})
        demo.to_csv(tmp/"runs_demo.csv", index=False)
        demo.to_csv(tmp/"runs.csv", index=False, lineterminator='\r\n')
        traces = tmp/"traces"; traces.mkdir()
        t = np.linspace(0, 600, 601)
        with open(traces/"B_1.csv", "w") as f:
            f.write("# T_air_actual = 24.8 C\n# RH_actual = 51.2 %\n")
            pd.DataFrame({"t_s": t, "I_mA": np.full_like(t, 50.0),
                          "V_V": np.full_like(t, 12.13),
                          "T_b_C": np.linspace(25.0, 25.5, 601),
                          "EC_raw_uScm": np.full_like(t, 799.5)}).to_csv(f, index=False)
        old_cwd = os.getcwd(); os.chdir(tmp)
        try:
            try:
                run_analysis("runs.csv", traces_dir="traces", out="test.json",
                             strict=False, verbose=False)
                return [("N-12 full match", "[X] لم يرفض", "expected raise")]
            except ValueError as e:
                if "N-12" in str(e) or "مطابق" in str(e):
                    return [("N-12 full match", "[OK] رفض", "correct")]
                return [("N-12 full match", f"[!]", "check")]
        finally: os.chdir(old_cwd)

def _test_max_abs_err_demo_warn():
    global MODE
    from pathlib import Path
    import tempfile, os
    old_mode = MODE
    try:
        with tempfile.TemporaryDirectory() as tmp:
            tmp = Path(tmp)
            pd.DataFrame({"run":["B_1"],"cond":["B"],"day":[1],
                "ec0":[799.5],"t0":[25.02],"ecf":[796.7],"tf":[25.55],
                "dpd":[1.71],"dilution":[3],"charge_C":[30.0],"dpd_pre":[np.nan],
                "V_cell_mean":[12.13],"I_mean_mA":[50.0],"T_b_mean":[25.35],
                "pH_meas":[np.nan],"T_sample":[np.nan],"ATC_state":["OFF"],
                "T_air_std":[0.14],"RH_std":[1.8],"V_L_measured":[0.100]}).to_csv(tmp/"runs.csv", index=False)
            t = np.linspace(0, 600, 601)
            traces_dir = tmp/"traces"; traces_dir.mkdir()
            with open(traces_dir/"B_1.csv", "w") as f:
                f.write("# T_air_actual = 24.8 C\n# RH_actual = 51.2 %\n")
                pd.DataFrame({"t_s": t, "I_mA": np.full_like(t, 50.0),
                              "V_V": np.full_like(t, 12.13),
                              "T_b_C": np.full_like(t, 35.0),
                              "EC_raw_uScm": np.full_like(t, 799.5)}).to_csv(f, index=False)
            old_cwd = os.getcwd(); os.chdir(tmp)
            try:
                MODE = "DEMO"
                try:
                    with warnings.catch_warnings(record=True) as w:
                        warnings.simplefilter("always")
                        run_analysis("runs.csv", traces_dir="traces",
                                     out="test_demo.json", strict=False, verbose=False)
                        if [x for x in w if "max_abs_err_K" in str(x.message)]:
                            return [("max_abs_err warn DEMO", "[OK] تحذير", "correct")]
                        return [("max_abs_err warn DEMO", "[i] لا تحذير", "check")]
                except ValueError as e:
                    return [("max_abs_err warn DEMO", f"[!] رفع", "check")]
            finally:
                os.chdir(old_cwd); MODE = old_mode
    finally: MODE = old_mode

def _test_x1_physical():
    global MODE
    old_mode = MODE; results = []; MODE = "REAL"
    try:
        try: assert_x1_physical(1.5, mode="REAL"); results.append(("x1 out", "[X]", "expected"))
        except ValueError: results.append(("x1 out", "[OK] رفع", "correct"))
        try: assert_x1_physical(0.05, mode="REAL"); results.append(("x1 in", "[OK]", "correct"))
        except ValueError: results.append(("x1 in", "[X]", "unexpected"))
        try: assert_x1_physical(1.5, mode="DEMO"); results.append(("x1 DEMO", "[OK]", "correct"))
        except ValueError: results.append(("x1 DEMO", "[X]", "unexpected"))
    finally: MODE = old_mode
    return results

def _test_validate_runs_C9_and_R5():
    results = []
    base = {"run":"C_1","cond":"C","day":2,"ec0":799.7,"t0":25.01,
            "ecf":799.7,"tf":25.02,"dpd":1.71,"dilution":1,"charge_C":0,
            "dpd_pre":2.00,"V_cell_mean":0,"I_mean_mA":0,"T_b_mean":25.01,
            "pH_meas":np.nan,"T_sample":np.nan,"ATC_state":"OFF",
            "T_air_std":0.14,"RH_std":1.8,"V_L_measured":0.100}
    df_nan = pd.DataFrame([{**base, "dpd_pre": np.nan}])
    if any("dpd_pre=NaN" in w for w in validate_runs(df_nan, strict_mode=True)):
        results.append(("C-9", "[OK] تحذير", "correct"))
    else: results.append(("C-9", "[X]", "expected"))
    df_ok = pd.DataFrame([{**base, "dpd_pre": 2.03}])
    if not any("R-5" in w for w in validate_runs(df_ok, strict_mode=True)):
        results.append(("R-5 2.03", "[OK] مقبول", "correct"))
    else: results.append(("R-5 2.03", "[X]", "unexpected"))
    df_bad = pd.DataFrame([{**base, "dpd_pre": 2.5}])
    if any("R-5" in w for w in validate_runs(df_bad, strict_mode=True)):
        results.append(("R-5 2.5", "[OK] تحذير", "correct"))
    else: results.append(("R-5 2.5", "[X]", "expected"))
    df_low = pd.DataFrame([{**base, "dpd_pre": 1.5}])
    if any("R-5" in w for w in validate_runs(df_low, strict_mode=True)):
        results.append(("R-5 1.5", "[OK] تحذير", "correct"))
    else: results.append(("R-5 1.5", "[X]", "expected"))
    return results

def _test_validate_runs_R8():
    results = []
    base = {"run":"A_1","cond":"A","day":1,"ec0":799.6,"t0":25.0,
            "ecf":799.4,"tf":25.1,"dpd":0.02,"dilution":1,"charge_C":0,
            "dpd_pre":np.nan,"V_cell_mean":0,"I_mean_mA":0,"T_b_mean":25.0,
            "pH_meas":np.nan,"T_sample":np.nan,"ATC_state":"OFF",
            "T_air_std":0.14,"RH_std":1.8,"V_L_measured":0.100}
    df1 = pd.DataFrame([{**base, "ecf": 680.0}])
    if any("R-8a" in w for w in validate_runs(df1, strict_mode=True)):
        results.append(("R-8a", "[OK]", "correct"))
    else: results.append(("R-8a", "[X]", "expected"))
    df2 = pd.DataFrame([{**base, "T_b_mean": 45.0}])
    if any("R-8b" in w for w in validate_runs(df2, strict_mode=True)):
        results.append(("R-8b", "[OK]", "correct"))
    else: results.append(("R-8b", "[X]", "expected"))
    df3 = pd.DataFrame([{**base, "T_b_mean": 10.0}])
    if any("R-8c" in w for w in validate_runs(df3, strict_mode=True)):
        results.append(("R-8c", "[OK]", "correct"))
    else: results.append(("R-8c", "[X]", "expected"))
    df4 = pd.DataFrame([base])
    if not any("R-8" in w for w in validate_runs(df4, strict_mode=True)):
        results.append(("R-8 pass", "[OK]", "correct"))
    else: results.append(("R-8 pass", "[X]", "unexpected"))
    return results

def _test_FIX_H1_H2_labels():
    results = []
    if MODEL_LABEL == "V50.6.6":
        results.append(("FIX-H1 MODEL_LABEL", "[OK] V50.6.6", "correct"))
    else:
        results.append(("FIX-H1 MODEL_LABEL", f"[X] {MODEL_LABEL}", "unexpected"))
    if PROTOCOL_LABEL == "P11.13-Rev7.4.0":
        results.append(("FIX-H2 PROTOCOL_LABEL", "[OK] Rev7.4.0", "correct"))
    else:
        results.append(("FIX-H2 PROTOCOL_LABEL", f"[X]", "unexpected"))
    return results

def _test_FIX_H3_K25_unified():
    results = []
    if abs(K25_EFF_DEFAULT - K25_OPT_V22) < 1e-4:
        results.append(("FIX-H3 K25 unified", f"[OK] {K25_EFF_DEFAULT:.6e}", "correct"))
    else:
        results.append(("FIX-H3 K25 unified", f"[X]", "unexpected"))
    return results

def _test_FIX_H4_A_C_traces():
    results = []
    traces_dir = Path("traces_demo")
    if not traces_dir.exists():
        return [("FIX-H4 A+C traces", "[X] traces_demo/ مفقود", "check")]
    a_count = len(list(traces_dir.glob("A_*.csv")))
    c_count = len(list(traces_dir.glob("C_*.csv")))
    total = len(list(traces_dir.glob("*.csv")))
    if a_count >= 5 and c_count >= 2 and total >= 18:
        results.append(("FIX-H4 A+C traces", f"[OK] A={a_count} C={c_count} total={total}", "correct"))
    else:
        results.append(("FIX-H4 A+C traces", f"[!] A={a_count} C={c_count} total={total}", "check"))
    return results

def _test_FIX_H5_I_lim_noise():
    results = []
    e_trace = Path("traces_demo")/"E_1.csv"
    if not e_trace.exists():
        return [("FIX-H5 I_lim noise", "[X] E_1.csv مفقود", "check")]
    try:
        I_lim, I_lim_std, n_pts, warn = compute_I_lim(str(e_trace))
        if I_lim_std > 0.001:
            results.append(("FIX-H5 I_lim noise", f"[OK] std={I_lim_std:.4f}", "correct"))
        else:
            results.append(("FIX-H5 I_lim noise", f"[!] std={I_lim_std:.4f}", "check"))
    except Exception as e:
        results.append(("FIX-H5 I_lim noise", f"[X]", "unexpected"))
    return results

def _test_FIX_H6_pKa_function():
    results = []
    try:
        pka_25 = pKa_HOCl_at_T(25.0); pka_35 = pKa_HOCl_at_T(35.0)
        if abs(pka_25 - 7.54) < 1e-6 and abs(pka_35 - 7.04) < 1e-6:
            results.append(("FIX-H6 pKa(T)", f"[OK] 25°C={pka_25}, 35°C={pka_35}", "correct"))
        else:
            results.append(("FIX-H6 pKa(T)", f"[X]", "unexpected"))
    except Exception as e:
        results.append(("FIX-H6 pKa(T)", f"[X]", "unexpected"))
    return results

def _test_FIX_H7_C_CL():
    if abs(C_CL - 5.6) < 1e-9 and abs(D_CL - 2.03e-9) < 1e-15:
        return [("FIX-H7 C_CL", f"[OK] C={C_CL} mol/m3, D={D_CL} m2/s", "correct")]
    return [("FIX-H7 C_CL", "[X]", "unexpected")]

def _test_FIX_H8_A_E():
    results = []
    if abs(A_E - 2.0e-4) < 1e-9:
        results.append(("FIX-H8 A_E", f"[OK] {A_E} m²", "correct"))
    else:
        results.append(("FIX-H8 A_E", f"[X]", "unexpected"))
    return results

def _test_FIX_I1_k25_updated():
    results = []
    if abs(K25_EFF_DEFAULT - 1.6573e-3) < 1e-9:
        results.append(("FIX-I1 K25 updated", f"[OK] {K25_EFF_DEFAULT:.6e}", "correct"))
    else:
        results.append(("FIX-I1 K25 updated", f"[X]", "unexpected"))
    return results

def _test_FIX_I2_GUM():
    results = []
    try:
        gum = gum_uncertainty(0.05, 0.005, 0.10)
        if "u_c" in gum and gum["u_c"] > 0 and abs(gum["U_expanded"] - 2.0*gum["u_c"]) < 1e-12:
            results.append(("FIX-I2 GUM", f"[OK] u_c={gum['u_c']:.4f}, U={gum['U_expanded']:.4f}", "correct"))
        else:
            results.append(("FIX-I2 GUM", f"[X]", "unexpected"))
    except Exception as e:
        results.append(("FIX-I2 GUM", f"[X]", "unexpected"))
    return results

def _test_FIX_I3_MonteCarlo():
    try:
        mc = error_propagation_monte_carlo([4.8]*5, [30.0]*5, [0.15]*5, 0.1, (1.2, 4.9), n_samples=1000)
        if "x1_mean" in mc and "CI95_lo" in mc and mc["n_samples"] == 1000:
            return [("FIX-I3 Monte Carlo", f"[OK] n={mc['n_samples']}, mean={mc['x1_mean']:.4f}", "correct")]
    except Exception:
        pass
    return [("FIX-I3 Monte Carlo", "[X]", "unexpected")]

def _test_FIX_I5_Sobol():
    try:
        s_ = sensitivity_elasticities(0.15)
        e = s_["elasticities"]
        ok = (abs(e["eps"] - 0.15/0.85) < 1e-12 and e["K25"] == 0.0
              and e["dpd"] == 1.0 and e["Q"] == -1.0)
        if ok: return [("FIX-I5 Sobol (OAT)", f"[OK] eps={e['eps']:.4f}, K25=0", "correct")]
    except Exception:
        pass
    return [("FIX-I5 Sobol (OAT)", "[X]", "unexpected")]

def _test_FIX_I6_Outliers():
    results = []
    try:
        data = [1.0, 1.1, 1.05, 0.95, 1.02, 1.08, 5.0]
        g = detect_outliers_grubbs(data)
        if g.get("n_outliers", 0) >= 1:
            results.append(("FIX-I6 Grubbs", f"[OK] {g['n_outliers']} outlier", "correct"))
        else:
            results.append(("FIX-I6 Grubbs", f"[!] no outlier detected", "check"))
        d = detect_outliers_dixon(data)
        if "Q_low" in d or "Q_high" in d or "note" in d:
            results.append(("FIX-I6 Dixon", f"[OK]", "correct"))
        else:
            results.append(("FIX-I6 Dixon", f"[X]", "unexpected"))
    except Exception as e:
        results.append(("FIX-I6 Outliers", f"[X] {str(e)[:30]}", "unexpected"))
    return results

def _test_FIX_I7_Repro():
    results = []
    try:
        r = reproducibility_check([
            {"x1_median": 0.050}, {"x1_median": 0.048}, {"x1_median": 0.052}
        ])
        if "CV" in r and r["n_runs"] == 3:
            results.append(("FIX-I7 Repro", f"[OK] CV={r['CV_pct']:.2f}%", "correct"))
        else:
            results.append(("FIX-I7 Repro", f"[X]", "unexpected"))
    except Exception as e:
        results.append(("FIX-I7 Repro", f"[X]", "unexpected"))
    return results

def _test_FIX_J3_K25_bias():
    """FIX-J3: يتحقق من أن K25 calibration bias < 1%."""
    from pathlib import Path; import tempfile
    K25_true = K25_EFF_DEFAULT
    T_air_true = 24.8; RH_true = 51.2; T0 = 35.0
    t = np.linspace(0, 1800, 1801)
    def _dTb(t_now, T_b):
        return thermal_model_energy_balance(
            t_now, T_b, None, None, None,
            T_air_true, RH_true, K25_true, K_DRY_FIXED, C_TOTAL
        )
    sol = solve_ivp(_dTb, [0, 1800], [T0], t_eval=t, method='RK45', rtol=1e-10, atol=1e-12)
    T_b_data = sol.y[0]
    with tempfile.TemporaryDirectory() as tmp:
        p = Path(tmp)/"D_test.csv"
        with open(p, "w") as f:
            f.write(f"# T_air_actual = {T_air_true} C\n# RH_actual = {RH_true} %\n")
            pd.DataFrame({
                "t_s": t, "I_mA": np.zeros_like(t), "V_V": np.zeros_like(t),
                "T_b_C": T_b_data, "EC_raw_uScm": np.full_like(t, 799.6)
            }).to_csv(f, index=False)
        K25_fit, err, conv = calibrate_K25_from_D_refined(str(p))
        if not conv:
            return [("FIX-J3 K25 bias", f"[X] not converged", "unexpected")]
        rel_err = abs(K25_fit - K25_true) / K25_true
        if rel_err < 0.01:
            return [("FIX-J3 K25 bias", f"[OK] rel_err={rel_err*100:.4f}% (< 1%)", "correct")]
        elif rel_err < 0.05:
            return [("FIX-J3 K25 bias", f"[i] rel_err={rel_err*100:.2f}% (1-5%)", "check")]
        else:
            return [("FIX-J3 K25 bias", f"[X] rel_err={rel_err*100:.2f}% (> 5%)", "unexpected")]

def _test_FIX_J5_GUM_extended():
    results = []
    cases = [
        (0.05, 0.005, 0.10, "typical"),
        (0.10, 0.010, 0.05, "small_rel_syst"),
        (0.01, 0.001, 0.20, "large_rel_syst"),
    ]
    for x1, u_A, rel_B, name in cases:
        try:
            gum = gum_uncertainty(x1, u_A, rel_B)
            if "u_c" in gum and gum["u_c"] > 0 and abs(gum["U_expanded"] - 2.0*gum["u_c"]) < 1e-12:
                results.append((f"FIX-J5 GUM {name}", f"[OK] u_c={gum['u_c']:.5f}", "correct"))
            else:
                results.append((f"FIX-J5 GUM {name}", "[X]", "unexpected"))
        except Exception as e:
            results.append((f"FIX-J5 GUM {name}", f"[X] {str(e)[:30]}", "unexpected"))
    return results

def _test_FIX_J6_MC_extended():
    results = []
    for n in [1000, 10000]:
        try:
            mc = error_propagation_monte_carlo([4.8]*5, [30.0]*5, [0.15]*5, 0.1, (1.2, 4.9), n_samples=n, seed=42)
            if mc.get("n_samples") == n and mc["CI95_lo"] <= mc["x1_median"] <= mc["CI95_hi"]:
                results.append((f"FIX-J6 MC n={n}", "[OK] median in CI95", "correct"))
            else:
                results.append((f"FIX-J6 MC n={n}", "[X]", "unexpected"))
        except Exception as e:
            results.append((f"FIX-J6 MC n={n}", f"[X] {str(e)[:30]}", "unexpected"))
    return results

def _write_trace(path, t, I, V, Tb=25.0, ec=799.6):
    with open(path, "w") as f:
        f.write("# T_air_actual = 24.8 C\n# RH_actual = 51.2 %\n")
        pd.DataFrame({"t_s": t, "I_mA": I, "V_V": V,
                      "T_b_C": np.full_like(t, Tb) if np.isscalar(Tb) else Tb,
                      "EC_raw_uScm": np.full_like(t, ec)}).to_csv(f, index=False)

def _make_E(plateaus, tau=40.0, noise=0.1, seed=5):
    t = np.linspace(-60, 960, 1021); rng = np.random.default_rng(seed)
    I = np.zeros_like(t); V = np.zeros_like(t); prev = 0.0
    for (a, b, v), Ip in zip([(0.0, 300.0, 2.0), (300.0, 600.0, 3.0), (600.0, 900.0001, 4.0)], plateaus):
        m = (t >= a) & (t < b)
        I[m] = Ip + (prev - Ip) * np.exp(-(t[m] - a) / tau); V[m] = v; prev = Ip
    I = np.where((t >= 0) & (t <= 900), I + rng.normal(0, noise, len(t)), 0.0)
    return t, I, V

def _test_K11_sigma():
    sg = dpd_sigma(0.05, [3], [4.8])[0]
    ref = np.sqrt((0.05/3.3*3)**2 + (0.03*4.8)**2)
    ok = abs(sg - ref) < 1e-12 and sg > (0.05/3.3*3)
    return [("K11 sigma proportional", f"[OK] sigma={sg:.4f} (LOD وحده {0.05/3.3*3:.4f})" if ok else f"[X] {sg}", "LOD + 3%")]

def _test_K12_load_cal_text_rows():
    import tempfile
    with tempfile.TemporaryDirectory() as tmp:
        p = Path(tmp)/"cal.csv"
        pd.DataFrame({"item": ["alpha", "pH_0", "batch_date"], "value": [0.0209, 7.0, "2026-10-11"],
                      "uncertainty": [0.0005, 0.1, None]}).to_csv(p, index=False)
        c = load_cal(p)
    ok = c.get("alpha") == (0.0209, 0.0005) and c.get("pH_0", (None,))[0] == 7.0 and "batch_date" not in c
    return [("K12 load_cal text rows", "[OK] batch_date متجاوَز" if ok else f"[X] {c}", "was crash")]

def _c2_case(factor, x1=0.05):
    r = 0.855; Q = 30.0; pre = 2.0
    prod = x1*(Q/F)/2.0*M_CL2*1000.0/0.1
    dpd = r*(pre+prod)*factor/5.0
    df_ = pd.DataFrame({"cond": ["C2", "C2"], "run": ["C2_1", "C2_2"], "Q": [Q, Q],
                        "dpd_pre": [pre, pre], "dilution": [5, 5], "dpd": [dpd, dpd*1.01]})
    x1r = {"median": x1, "CI95_lo_final": 0.044, "CI95_hi_final": 0.0598}
    return df_, x1r, {"mean": r, "std": 0.024, "n": 2}

def _test_K13_C2_consistency():
    out = []
    df_, x1r, rec = _c2_case(1.0)
    c = c2_consistency_check(df_, x1r, rec)
    ok = c["available"] and c["consistent"] and abs(c["mean_z"]) < 0.5
    out.append(("K13 C2 consistent", f"[OK] z={c['mean_z']:+.2f}" if ok else f"[X] {c}", "no extra loss"))
    df_, x1r, rec = _c2_case(0.6)
    c = c2_consistency_check(df_, x1r, rec)
    ok = c["available"] and (not c["consistent"]) and c["mean_z"] < -2 and c["x1_shift_pct"] > 10
    out.append(("K13 C2 extra loss flagged", (f"[OK] z={c['mean_z']:+.1f}, x1 البديل {c['x1_shift_pct']:+.0f}%" if ok else f"[X] {c}"), "40% loss"))
    c = c2_consistency_check(df_, x1r, None)
    out.append(("K13 C2 unavailable", "[OK] available=False بلا C" if c["available"] is False else f"[X] {c}", "needs C"))
    return out

def _test_K1_delta_units():
    d = back_calculate_delta(0.051, 0.05)
    ref = F*2.03e-9*5.6*2e-4*1.15/(0.051*0.05*(1-0.51))*100.0
    if d is not None and abs(d-ref)/ref < 1e-9 and 0.015 < d < 0.025 and DELTA_RANGE_CM[0] <= d <= DELTA_RANGE_CM[1]:
        return [("K1 delta units", f"[OK] delta={d:.4f} cm", "SI-consistent")]
    return [("K1 delta units", f"[X] {d}", "unexpected")]

def _test_K2_I_lim_segments():
    import tempfile
    out = []
    with tempfile.TemporaryDirectory() as tmp:
        p = Path(tmp)/"E_1.csv"
        t, I, V = _make_E([10.0, 15.0, 15.1]); _write_trace(p, t, I, V)
        r = compute_I_lim_segments(str(p))
        ok = (r.get("plateau") is True and r.get("I_lim_mA") is not None
              and abs(r["I_lim_mA"] - 15.05) < 0.15)
        old, *_ = compute_I_lim(str(p), t_start=300.0, t_end=600.0)
        out.append(("K2 I_lim segments", (f"[OK] I_lim={r['I_lim_mA']:.2f} (النافذة القديمة {old:.2f})" if ok else f"[X] {r}"),
                    "transient excluded"))
        t, I, V = _make_E([10.0, 15.0, 20.0]); _write_trace(p, t, I, V)
        r = compute_I_lim_segments(str(p))
        ok = (r.get("plateau") is False and r.get("I_lim_mA") is None
              and "no_plateau" in (r.get("warning") or ""))
        out.append(("K2 I_lim no plateau", "[OK] لا هضبة => I_lim=None" if ok else f"[X] {r}", "correct"))
        t, I, V = _make_E([10.0, 15.0, 15.1]); V = np.where(V > 0, 3.0, 0.0); _write_trace(p, t, I, V)
        r = compute_I_lim_segments(str(p))
        ok = "V_not_at_nominal" in (r.get("warning") or "")
        out.append(("K2 I_lim V mismatch", "[OK] تحذير الجهد" if ok else f"[X] {r}", "correct"))
    return out

def _test_K3_template():
    out = []
    tpl = make_runs_template()
    ok = bool(tpl["dpd_pre"].isna().all() and tpl["V_L_measured"].isna().all()
              and tpl.loc[tpl["cond"].isin(["B", "C2"]), "charge_C"].isna().all())
    out.append(("K3 template no prefill", "[OK] dpd_pre/V_L/charge_C فارغة" if ok else "[X]", "correct"))
    df = tpl.copy()
    miss0 = missing_required_fields(df)
    for c in ["ec0", "t0", "ecf", "tf", "dpd", "I_mean_mA", "T_b_mean", "T_air_std", "RH_std", "V_L_measured"]:
        df[c] = 1.0
    df.loc[df["cond"].isin(["B", "C2"]), "V_cell_mean"] = 12.0
    miss1 = missing_required_fields(df)      # dpd_pre لا تزال فارغة => يجب أن تظهر
    w = validate_runs(df.assign(T_b_mean=25.0, V_L_measured=0.1, I_mean_mA=50.0, ec0=799.6, ecf=799.5, t0=25.0, tf=25.1),
                      strict_mode=True)
    ok = (len(miss0) > 20 and len(miss1) == 4 and all("dpd_pre" in m for m in miss1)
          and any("C-9" in x for x in w))
    out.append(("K3 required fields", f"[OK] dpd_pre غير المقاس يُرفض (C-9 + {len(miss1)} حقل)" if ok else f"[X] {miss1} {w[:2]}", "correct"))
    return out

def _test_K4_demo_thermal():
    b = sorted(Path("traces_demo").glob("B_*.csv"))
    if not b: return [("K4 demo thermal aligned", "[X] traces_demo مفقود", "check")]
    r = integrate_thermal_ode(str(b[0]), T_air=24.8, RH_pct=51.2, K25_eff=K25_EFF_DEFAULT, K_dry=K_DRY_FIXED)
    e = r.get("max_abs_err_K", 9e9)
    return [("K4 demo thermal aligned", (f"[OK] max_err={e:.4f} K (<0.03)" if e < 0.03 else f"[!] max_err={e:.4f} K"), "was 0.098")]

def _test_K5_thermal_real_no_abort():
    global MODE
    import tempfile, os
    old_mode = MODE; res = []
    try:
        with tempfile.TemporaryDirectory() as tmp:
            tmp = Path(tmp)
            pd.DataFrame({"run":["B_1"],"cond":["B"],"day":[1],"ec0":[799.5],"t0":[25.02],"ecf":[796.7],
                "tf":[25.55],"dpd":[1.71],"dilution":[3],"charge_C":[30.0],"dpd_pre":[np.nan],
                "V_cell_mean":[12.13],"I_mean_mA":[50.0],"T_b_mean":[25.35],"pH_meas":[np.nan],
                "T_sample":[np.nan],"ATC_state":["OFF"],"T_air_std":[0.14],"RH_std":[1.8],
                "V_L_measured":[0.100]}).to_csv(tmp/"real_like.csv", index=False)
            tr = tmp/"traces"; tr.mkdir(); t = np.linspace(0, 600, 601)
            _write_trace(tr/"B_1.csv", t, np.full_like(t, 50.0), np.full_like(t, 12.13), Tb=35.0, ec=799.5)
            old_cwd = os.getcwd(); os.chdir(tmp)
            try:
                MODE = "REAL"
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    s_ = run_analysis("real_like.csv", traces_dir="traces", out="o.json", strict=False, verbose=False)
                flag = s_["diagnostics"]["max_abs_err_flag"]
                res.append(("K5 thermal REAL no abort", "[OK] لم يُرفض التحليل + العلم مرفوع" if flag else "[X] لا علم", "correct"))
            except ValueError as e:
                res.append(("K5 thermal REAL no abort", f"[X] رُفض: {str(e)[:40]}", "unexpected"))
            finally:
                os.chdir(old_cwd)
    finally:
        MODE = old_mode
    return res

def _test_K6_K25_multi():
    import tempfile
    t = np.linspace(0, 1800, 1801); rng = np.random.default_rng(7)
    sol = solve_ivp(lambda tn, T: thermal_model_energy_balance(tn, T, None, None, None, 24.8, 51.2,
                    K25_EFF_DEFAULT, K_DRY_FIXED, C_TOTAL), [0, 1800], [35.0], t_eval=t, rtol=1e-10, atol=1e-12)
    with tempfile.TemporaryDirectory() as tmp:
        paths = []
        for i in range(3):
            p = Path(tmp)/f"D_{i+1}.csv"
            _write_trace(p, t, np.zeros_like(t), np.zeros_like(t), Tb=sol.y[0] + rng.normal(0, 0.02, len(t)))
            paths.append(str(p))
        r = calibrate_K25_multi(paths)
    ok = (r["n_converged"] == 3 and abs(r["K25_median"] - K25_EFF_DEFAULT)/K25_EFF_DEFAULT < 0.06
          and r["spread_rel"] is not None and r["spread_rel"] < 0.10)
    return [("K6 K25 multi-D", (f"[OK] median={r['K25_median']:.4e} spread={r['spread_rel']*100:.2f}%" if ok else f"[X] {r}"), "RMS fit")]

def _test_K7_MC_data_driven():
    base = error_propagation_monte_carlo([4.8]*5, [30.0]*5, [0.15]*5, 0.1, (1.2, 4.9), 5000)
    wide = error_propagation_monte_carlo([4.8]*5, [30.0]*5, [1.5]*5, 0.1, (1.2, 4.9), 5000)
    central = 4.8*0.1/((1-1.2/6.1)*M_CL2*1000.0*30.0/(2*F))
    ok = (wide["x1_std"] > base["x1_std"] and base["CI95_lo"] < base["x1_median"] < base["CI95_hi"]
          and 0.5*central < base["x1_median"] < 1.5*central)
    return [("K7 MC data-driven", (f"[OK] std يتبع sigma ({base['x1_std']:.4f} -> {wide['x1_std']:.4f})" if ok else f"[X] {base} {wide}"), "uses data")]

def _test_K7_elasticities():
    e = sensitivity_elasticities(0.15)["elasticities"]
    ok = e["K25"] == 0.0 and abs(e["eps"] - 0.15/0.85) < 1e-12
    return [("K7 elasticities", "[OK] K25=0, eps=0.1765" if ok else "[X]", "analytic")]

def _test_K10_missing_trace_real():
    global MODE
    import tempfile, os
    old_mode = MODE; res = []
    try:
        with tempfile.TemporaryDirectory() as tmp:
            tmp = Path(tmp)
            pd.DataFrame({"run":["B_1"],"cond":["B"],"day":[1],"ec0":[799.5],"t0":[25.02],"ecf":[796.7],
                "tf":[25.55],"dpd":[1.71],"dilution":[3]}).to_csv(tmp/"real_like.csv", index=False)
            (tmp/"traces").mkdir()
            old_cwd = os.getcwd(); os.chdir(tmp)
            try:
                MODE = "REAL"
                try:
                    run_analysis("real_like.csv", traces_dir="traces", out="o.json", strict=False, verbose=False)
                    res.append(("K10 missing trace REAL", "[X] لم يرفض", "expected raise"))
                except ValueError as e:
                    res.append(("K10 missing trace REAL", "[OK] رفض Q الاسمي" if "traces" in str(e) else f"[!] {str(e)[:40]}", "correct"))
            finally:
                os.chdir(old_cwd)
    finally:
        MODE = old_mode
    return res

def run_stress_tests():
    """FIX-J1: استعادة + إضافة اختبارات v2.2 + v2.5"""
    all_results = []
    tests = [
        ("K25 direct", _test_K25_direct),
        ("K25 realistic", _test_K25_realistic),
        ("DEMO circular", _test_demo_circular_behavior),
        ("verify_files FIX-D", _test_verify_files_integration),
        ("FIX-D template", _test_FIX_D_template_exclusion),
        ("FIX-D2 template match", _test_FIX_D2_template_check),
        ("compute_I_lim", _test_compute_I_lim_full),
        ("beta_from_rec raises", _test_beta_from_rec_raises),
        ("N-12 per-row diff", _test_N12_per_row_diff_behavioral),
        ("N-12 full match", _test_N12_full_match_raises),
        ("max_abs_err warn DEMO", _test_max_abs_err_demo_warn),
        ("x1 physical bounds", _test_x1_physical),
        ("validate_runs C-9 + R-5", _test_validate_runs_C9_and_R5),
        ("validate_runs R-8", _test_validate_runs_R8),
        ("FIX-H1 H2 labels", _test_FIX_H1_H2_labels),
        ("FIX-H3 K25 unified", _test_FIX_H3_K25_unified),
        ("FIX-H4 A+C traces", _test_FIX_H4_A_C_traces),
        ("FIX-H5 I_lim noise", _test_FIX_H5_I_lim_noise),
        ("FIX-H6 pKa(T)", _test_FIX_H6_pKa_function),
        ("FIX-H7 C_CL", _test_FIX_H7_C_CL),
        ("FIX-H8 A_E", _test_FIX_H8_A_E),
        ("FIX-I1 K25 updated", _test_FIX_I1_k25_updated),
        ("FIX-I2 GUM", _test_FIX_I2_GUM),
        ("FIX-I3 Monte Carlo", _test_FIX_I3_MonteCarlo),
        ("FIX-I5 Sobol", _test_FIX_I5_Sobol),
        ("FIX-I6 Outliers", _test_FIX_I6_Outliers),
        ("FIX-I7 Repro", _test_FIX_I7_Repro),
        ("FIX-J3 K25 bias", _test_FIX_J3_K25_bias),
        ("FIX-J5 GUM extended", _test_FIX_J5_GUM_extended),
        ("FIX-J6 MC extended", _test_FIX_J6_MC_extended),
        ("K1 delta units", _test_K1_delta_units),
        ("K13 C2 consistency", _test_K13_C2_consistency),
        ("K12 load_cal", _test_K12_load_cal_text_rows),
        ("K11 sigma", _test_K11_sigma),
        ("K2 I_lim segments", _test_K2_I_lim_segments),
        ("K3 template", _test_K3_template),
        ("K4 demo thermal", _test_K4_demo_thermal),
        ("K5 thermal REAL", _test_K5_thermal_real_no_abort),
        ("K6 K25 multi-D", _test_K6_K25_multi),
        ("K7 MC data-driven", _test_K7_MC_data_driven),
        ("K7 elasticities", _test_K7_elasticities),
        ("K10 missing trace REAL", _test_K10_missing_trace_real),
    ]
    for test_name, fn in tests:
        try:
            import io, contextlib
            with contextlib.redirect_stdout(io.StringIO()):
                res = fn()
            all_results.extend(res)
        except Exception as e:
            all_results.append((f"{test_name} (EXCEPTION)", "[X]",
                                f"{type(e).__name__}: {str(e)[:80]}"))
    n_total = len(all_results)
    n_pass = sum(1 for _, st, _ in all_results if "[OK]" in st)
    n_fail = sum(1 for _, st, _ in all_results if "[X]" in st)
    n_check = n_total - n_pass - n_fail
    return {"total": n_total, "passed": n_pass, "failed": n_fail,
            "check": n_check, "results": all_results}

print("B13 - Stress tests (v2.5 مع FIX-K1..K13)")

def main():
    t_start = time.time()
    print("=" * 70)
    print(f"EC-CODE v{VERSION} - {MODEL_LABEL}")
    print("=" * 70)
    print("\n[1/6] save_demo_ref()...")
    t1 = time.time()
    result = save_demo_ref(seed=42)
    print(f"   t {time.time()-t1:.1f}s")
    Path("analysis_demo_summary.json").write_text(
        json.dumps(result, indent=2, ensure_ascii=False, default=str))
    print(f"[OK] Saved: analysis_demo_summary.json")

    print("\n[2/6] DEMO guard check...")
    try:
        assert_not_demo_for_citation(summary=result)
        print("  [!] لم يرفع")
    except ValueError:
        print("  [OK] DEMO guard يعمل")

    print("\n[3/6] تشخيص K25 bias (FIX-J3):")
    d_traces = sorted(Path("traces_demo").glob("D*.csv"))
    if d_traces:
        K25_opt, err_opt, conv = calibrate_K25_from_D_refined(str(d_traces[0]))
        if K25_opt:
            rel_bias = abs(K25_opt - K25_EFF_DEFAULT) / K25_EFF_DEFAULT
            print(f"    K25_EFF_DEFAULT = {K25_EFF_DEFAULT:.6e} m/s")
            print(f"    K25_opt (measured) = {K25_opt:.6e} m/s")
            print(f"    bias = {rel_bias*100:.4f}%")
            if rel_bias < 0.01:
                print(f"    [OK] bias < 1% (FIX-J2 ناجح)")
            elif rel_bias < 0.05:
                print(f"    [i] bias < 5% (مقبول)")
            else:
                print(f"    [X] bias > 5% (يحتاج مراجعة)")

    print("\n[4/6] تشخيص FIX-A - max_abs_err_K:")
    b_traces = sorted(Path("traces_demo").glob("B_*.csv"))
    if b_traces:
        T_air_d, RH_d = read_environment(str(b_traces[0]), strict=False)
        print(f"    read_environment -> T_air={T_air_d}, RH={RH_d}")
        ode_res = integrate_thermal_ode(str(b_traces[0]), T_air=T_air_d, RH_pct=RH_d,
                                        K25_eff=K25_EFF_DEFAULT,
                                        K_dry=K_DRY_FIXED, C_tot=C_TOTAL)  # FIX-K4: بيئة الـtrace
        if "error" not in ode_res:
            print(f"    max_abs_err_K = {ode_res['max_abs_err_K']:.6f}")

    print("\n[5/6] run_stress_tests()...")
    stress = run_stress_tests()
    print(f"\nSTRESS TESTS SUMMARY")
    print(f"  Total : {stress['total']}")
    print(f"  Pass  : {stress['passed']}")
    print(f"  Fail  : {stress['failed']}")
    print(f"  Check : {stress['check']}")

    print("\n[6/6] حفظ النتائج...")
    if d_traces:
        K25_opt, err_opt, conv = calibrate_K25_from_D_refined(str(d_traces[0]))
        with open("k25_calibration.txt", "w") as f:
            f.write(f"EC-CODE v{VERSION}\n")
            f.write(f"K25_opt = {K25_opt:.6e} m/s\n")
            f.write(f"err_opt = {err_opt:.6f} K\n")
            f.write(f"converged = {conv}\n")
            f.write(f"K25_EFF_DEFAULT = {K25_EFF_DEFAULT:.6e} m/s\n")
            bias_pct = abs(K25_opt - K25_EFF_DEFAULT)/K25_EFF_DEFAULT*100
            f.write(f"bias = {bias_pct:.4f}%\n")
        print(f"[OK] Saved: k25_calibration.txt")
    stress_out = {"version": VERSION, "mode": MODE,
        "summary": {"total": stress["total"], "passed": stress["passed"],
                    "failed": stress["failed"], "check": stress["check"]},
        "results": [{"name": n, "result": s, "note": nt}
                    for n, s, nt in stress["results"]]}
    Path("stress_results.json").write_text(json.dumps(stress_out, indent=2, ensure_ascii=False, default=str))
    print(f"[OK] Saved: stress_results.json")
    print(f"\n[OK] EC-CODE v{VERSION} COMPLETE")
    return result, stress

if __name__ == "__main__":
    result, stress = main()

B1 - imports + labels + TOLERANCE (v2.5)
B2 - physical constants (v2.5)
B3 - grids + columns + accept + segments + required fields
B4 - helpers + thermal_model_energy_balance (v2.5)
B5 - traces + thermal ODE + compute_I_lim
B6 - D curve + K25 calibration
B7 - chemistry + gas + delta + beta
B7b - GUM + MC + elasticities + Outliers + Repro
B8 - Bayesian grid + infer_x1
B9b - verify_files (v2.5)
B10a - run_analysis (part 1) v2.5
B10b - run_analysis (part 2) v2.5
B11 - make_demo_data
B12 - traces + save_demo_ref (v2.5)
B13 - Stress tests (v2.5 مع FIX-K1..K13)
EC-CODE v2.5+FIX-K1..K13 - V50.6.6

[1/6] save_demo_ref()...
[OK] Created runs_demo.csv (18 rows)
[OK] Created 18 traces
[OK] Saved: /content/reference_demo_summary.json
   t 6.6s
[OK] Saved: analysis_demo_summary.json

[2/6] DEMO guard check...
  [OK] DEMO guard يعمل

[3/6] تشخيص K25 bias (FIX-J3):
    K25_EFF_DEFAULT = 1.657300e-03 m/s
    K25_opt (measured) = 1.677606e-03 m/s
    bias = 1.2253%
    [i] bias < 5% (مقبول)

[4/6] تشخي

In [ ]:
import json

# قراءة ملف نتائج الاختبارات
with open('/content/stress_results.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

print("🔍 تفاصيل الاختبارات التي لم تنجح (Fail / Check):\n")
for res in data.get('results', []):
    if res.get('status') in ['FAIL', 'CHECK']:
        print(f"❌ [{res.get('status')}] {res.get('name')}")
        print(f"   السبب: {res.get('detail', res.get('details', 'غير محدد'))}\n")

🔍 تفاصيل الاختبارات التي لم تنجح (Fail / Check):



In [ ]:
import json

with open('/content/stress_results.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

print("🔍 ملخص النتائج:", data.get('summary', {}))

print("\n🔍 تفاصيل الاختبارات التي تحتاج مراجعة (Fail / Check):")
found = False
for res in data.get('results', []):
    # نبحث عن أي اختبار حالته ليست PASS أو True
    status = str(res.get('status', res.get('passed', res.get('state', 'UNKNOWN'))))

    if status.upper() not in ['PASS', 'TRUE', 'OK', '1']:
        print(f"⚠️ الاختبار: {res.get('name', 'غير معروف')}")
        print(f"   الحالة: {status}")
        print(f"   التفاصيل: {res.get('detail', res.get('details', res))}\n")
        found = True

if not found:
    print("لم يتم العثور على اختبارات فاشلة بالبحث المباشر.")
    print("هيكل أول اختبار للتوضيح:", data.get('results', [{}])[0])

🔍 ملخص النتائج: {'total': 69, 'passed': 67, 'failed': 1, 'check': 1}

🔍 تفاصيل الاختبارات التي تحتاج مراجعة (Fail / Check):
⚠️ الاختبار: K25 direct
   الحالة: UNKNOWN
   التفاصيل: {'name': 'K25 direct', 'result': '[OK] recovered 1.657300e-03', 'note': 'rel_err=0.00%'}

⚠️ الاختبار: K25 realistic
   الحالة: UNKNOWN
   التفاصيل: {'name': 'K25 realistic', 'result': '[i] rel_err=73.8% (K_dry+15%)', 'note': 'LIM-6: K25 غير قابل للتعريف'}

⚠️ الاختبار: DEMO guard
   الحالة: UNKNOWN
   التفاصيل: {'name': 'DEMO guard', 'result': '[OK] رفع', 'note': 'correct'}

⚠️ الاختبار: DEMO guard summary
   الحالة: UNKNOWN
   التفاصيل: {'name': 'DEMO guard summary', 'result': '[OK] رفع', 'note': 'correct'}

⚠️ الاختبار: REAL guard
   الحالة: UNKNOWN
   التفاصيل: {'name': 'REAL guard', 'result': '[OK]', 'note': 'correct'}

⚠️ الاختبار: verify_files FIX-D
   الحالة: UNKNOWN
   التفاصيل: {'name': 'verify_files FIX-D', 'result': '[OK] FILL=14', 'note': 'correct'}

⚠️ الاختبار: FIX-D template
   الحالة: UNKNOWN

In [ ]:
# ====================================================================
# Cell PATCH - v2.5.1
# شغّلها بعد Cell 1، قبل Cell 2
# ====================================================================

# ---- التعديل 1: تحديث PROTOCOL_LABEL ----
PROTOCOL_LABEL = "P11.13-Rev7.4.0"

# ---- التعديل 2: اختبار H1/H2 بالبروتوكول الجديد ----
def _test_FIX_H1_H2_labels():
    results = []
    if MODEL_LABEL == "V50.6.6":
        results.append(("FIX-H1 MODEL_LABEL", "[OK] V50.6.6", "correct"))
    else:
        results.append(("FIX-H1 MODEL_LABEL", f"[X] {MODEL_LABEL}", "unexpected"))
    if PROTOCOL_LABEL == "P11.13-Rev7.4.0":
        results.append(("FIX-H2 PROTOCOL_LABEL", "[OK] Rev7.4.0", "correct"))
    else:
        results.append(("FIX-H2 PROTOCOL_LABEL", f"[X] {PROTOCOL_LABEL}", "unexpected"))
    return results

# ---- التعديل 3: اختبار I5 كـ elasticities ----
def _test_FIX_I5_elasticities():
    try:
        s_ = sensitivity_elasticities(0.15)
        e = s_["elasticities"]
        ok = (abs(e["eps"] - 0.15/0.85) < 1e-12 and e["K25"] == 0.0
              and e["dpd"] == 1.0 and e["Q"] == -1.0)
        if ok: return [("FIX-I5 elasticities", f"[OK] eps={e['eps']:.4f}, K25=0", "correct")]
    except Exception:
        pass
    return [("FIX-I5 elasticities", "[X]", "unexpected")]

# ---- التعديل 4: run_stress_tests بالاسم الجديد ----
def run_stress_tests():
    """v2.5.1"""
    all_results = []
    tests = [
        ("K25 direct", _test_K25_direct),
        ("K25 realistic", _test_K25_realistic),
        ("DEMO circular", _test_demo_circular_behavior),
        ("verify_files FIX-D", _test_verify_files_integration),
        ("FIX-D template", _test_FIX_D_template_exclusion),
        ("FIX-D2 template match", _test_FIX_D2_template_check),
        ("compute_I_lim", _test_compute_I_lim_full),
        ("beta_from_rec raises", _test_beta_from_rec_raises),
        ("N-12 per-row diff", _test_N12_per_row_diff_behavioral),
        ("N-12 full match", _test_N12_full_match_raises),
        ("max_abs_err warn DEMO", _test_max_abs_err_demo_warn),
        ("x1 physical bounds", _test_x1_physical),
        ("validate_runs C-9 + R-5", _test_validate_runs_C9_and_R5),
        ("validate_runs R-8", _test_validate_runs_R8),
        ("FIX-H1 H2 labels", _test_FIX_H1_H2_labels),
        ("FIX-H3 K25 unified", _test_FIX_H3_K25_unified),
        ("FIX-H4 A+C traces", _test_FIX_H4_A_C_traces),
        ("FIX-H5 I_lim noise", _test_FIX_H5_I_lim_noise),
        ("FIX-H6 pKa(T)", _test_FIX_H6_pKa_function),
        ("FIX-H7 C_CL", _test_FIX_H7_C_CL),
        ("FIX-H8 A_E", _test_FIX_H8_A_E),
        ("FIX-I1 K25 updated", _test_FIX_I1_k25_updated),
        ("FIX-I2 GUM", _test_FIX_I2_GUM),
        ("FIX-I3 Monte Carlo", _test_FIX_I3_MonteCarlo),
        ("FIX-I5 elasticities", _test_FIX_I5_elasticities),
        ("FIX-I6 Outliers", _test_FIX_I6_Outliers),
        ("FIX-I7 Repro", _test_FIX_I7_Repro),
        ("FIX-J3 K25 bias", _test_FIX_J3_K25_bias),
        ("FIX-J5 GUM extended", _test_FIX_J5_GUM_extended),
        ("FIX-J6 MC extended", _test_FIX_J6_MC_extended),
        ("K1 delta units", _test_K1_delta_units),
        ("K13 C2 consistency", _test_K13_C2_consistency),
        ("K12 load_cal", _test_K12_load_cal_text_rows),
        ("K11 sigma", _test_K11_sigma),
        ("K2 I_lim segments", _test_K2_I_lim_segments),
        ("K3 template", _test_K3_template),
        ("K4 demo thermal", _test_K4_demo_thermal),
        ("K5 thermal REAL", _test_K5_thermal_real_no_abort),
        ("K6 K25 multi-D", _test_K6_K25_multi),
        ("K7 MC data-driven", _test_K7_MC_data_driven),
        ("K7 elasticities", _test_K7_elasticities),
        ("K10 missing trace REAL", _test_K10_missing_trace_real),
    ]
    for test_name, fn in tests:
        try:
            import io, contextlib
            with contextlib.redirect_stdout(io.StringIO()):
                res = fn()
            all_results.extend(res)
        except Exception as e:
            all_results.append((f"{test_name} (EXCEPTION)", "[X]",
                                f"{type(e).__name__}: {str(e)[:80]}"))
    n_total = len(all_results)
    n_pass = sum(1 for _, st, _ in all_results if "[OK]" in st)
    n_fail = sum(1 for _, st, _ in all_results if "[X]" in st)
    n_check = n_total - n_pass - n_fail
    return {"total": n_total, "passed": n_pass, "failed": n_fail,
            "check": n_check, "results": all_results}

# ---- التعديل 5: توثيق K8/K9 ----
FIX_NOTES = "K8, K9 دُمجا ضمن K6 و K7؛ لا يوجدان كاختبارات مستقلة."

# ---- إعادة تشغيل main() ----
print("=" * 70)
print("Cell PATCH v2.5.1 - إعادة تشغيل main()")
print("=" * 70)
print(f"  PROTOCOL_LABEL (جديد) = {PROTOCOL_LABEL}")
print()
result, stress = main()

print()
print("=" * 70)
print("PATCH v2.5.1 مُطبَّق بنجاح")
print(f"  PROTOCOL_LABEL = {PROTOCOL_LABEL}")
print(f"  Stress: {stress['total']} (Pass: {stress['passed']}, Fail: {stress['failed']}, Check: {stress['check']})")
print("=" * 70)

Cell PATCH v2.5.1 - إعادة تشغيل main()
  PROTOCOL_LABEL (جديد) = P11.13-Rev7.4.0

EC-CODE v2.5+FIX-K1..K13 - V50.6.6

[1/6] save_demo_ref()...
[OK] Created runs_demo.csv (18 rows)
[OK] Created 18 traces
[OK] Saved: /content/reference_demo_summary.json
   t 2.1s
[OK] Saved: analysis_demo_summary.json

[2/6] DEMO guard check...
  [OK] DEMO guard يعمل

[3/6] تشخيص K25 bias (FIX-J3):
    K25_EFF_DEFAULT = 1.657300e-03 m/s
    K25_opt (measured) = 1.677606e-03 m/s
    bias = 1.2253%
    [i] bias < 5% (مقبول)

[4/6] تشخيص FIX-A - max_abs_err_K:
    read_environment -> T_air=24.8, RH=51.2
    max_abs_err_K = 0.016599

[5/6] run_stress_tests()...

STRESS TESTS SUMMARY
  Total : 69
  Pass  : 68
  Fail  : 0
  Check : 1

[6/6] حفظ النتائج...
[OK] Saved: k25_calibration.txt
[OK] Saved: stress_results.json

[OK] EC-CODE v2.5+FIX-K1..K13 COMPLETE

PATCH v2.5.1 مُطبَّق بنجاح
  PROTOCOL_LABEL = P11.13-Rev7.4.0
  Stress: 69 (Pass: 68, Fail: 0, Check: 1)


In [ ]:
# ====================================================================
# Cell 2 - التحقق من التحميل (v2.5.1)
# ====================================================================

import inspect
from pathlib import Path

required_functions = [
    "run_analysis", "_run_analysis_part2", "infer_x1", "_infer_x1_grid",
    "beta_from_rec", "eps_prior_from_C_or_C2",
    "calibrate_K25_from_D_refined", "validate_runs",
    "charge_from_trace", "q_gen_from_trace", "compute_I_lim",
    "integrate_thermal_ode", "read_environment",
    "solve_charge_balance", "compute_gas_volumes", "back_calculate_delta",
    "dpd_expected", "kappa25", "E_tn_at", "Q_evap_abs", "Q_evap_delta",
    "P_sat", "_trapz", "_scalar",
    "welch_blocked", "add_systematic_asymmetric",
    "assert_not_demo_for_citation", "assert_mode_is_valid",
    "assert_x1_physical",
    "verify_files", "load_cal", "make_demo_data", "save_demo_ref",
    "_generate_demo_traces", "run_stress_tests", "main",
    "pKa_HOCl_at_T", "thermal_model_energy_balance",
    "gum_uncertainty", "error_propagation_monte_carlo",
    "sensitivity_elasticities", "compute_I_lim_segments", "make_runs_template",
    "missing_required_fields", "calibrate_K25_multi",
    "detect_outliers_grubbs", "detect_outliers_dixon",
    "reproducibility_check",
]

required_constants = [
    "VERSION", "PROTOCOL_LABEL", "MODEL_LABEL", "MODE",
    "F", "M_CL2", "V_L_DEFAULT", "E_TN_25", "E_TN_CL_25",
    "dETN_O2_dT", "dETN_CL_dT", "R_GAS", "M_W_KG", "D_CL", "T_CL",
    "F_EDL", "A_E", "A_E_cm2", "C_CL", "C_WATER", "C_GLASS", "C_TOTAL",
    "A_S", "A_WALL", "U_WALL", "U_CONV", "EPS_IR", "SIGMA_SB",
    "K_DRY_FIXED", "K25_EFF_DEFAULT", "K25_OPT_V22",
    "K1", "K2", "KW", "PKA_HOCL",
    "X1_GRID_MIN", "X1_GRID_MAX", "X1_GRID_N", "EPS_GRID_N",
    "X1_THRESHOLD", "REL_SYST_DEFAULT",
    "X1_PHYSICAL_MIN", "X1_PHYSICAL_MAX",
    "RUNS_COLUMNS_MANDATORY", "ACCEPT", "R5_RANGE",
    "R8_EC_DRIFT_MAX", "R8_TB_RANGE", "DEMO_CITED_ERROR",
    "GRUBBS_ALPHA", "DIXON_ALPHA",
    "E_SEGMENTS", "REQUIRED_BY_COND", "DELTA_RANGE_CM",
]

required_dicts = ["TOLERANCE"]

print("=" * 70)
print("Cell 2 - التحقق من تحميل الكود (v2.5.1)")
print("=" * 70)

print(f"\n[1] الدوال ({len(required_functions)}):")
fn_missing = [f for f in required_functions if not (f in globals() and callable(globals()[f]))]
for f in fn_missing: print(f"  [X] {f}")
print(f"  -> موجودة: {len(required_functions) - len(fn_missing)}/{len(required_functions)}")

print(f"\n[2] الثوابت ({len(required_constants)}):")
const_missing = [c for c in required_constants if c not in globals()]
for c in const_missing: print(f"  [X] {c}")
print(f"  -> موجودة: {len(required_constants) - len(const_missing)}/{len(required_constants)}")

print(f"\n[3] القواميس ({len(required_dicts)}):")
dict_missing = [d for d in required_dicts if not (d in globals() and isinstance(globals()[d], dict))]
for d in dict_missing: print(f"  [X] {d}")
print(f"  -> موجودة: {len(required_dicts) - len(dict_missing)}/{len(required_dicts)}")

print(f"\n[4] فحوص القيم الحرجة (v2.5.1):")
checks = []
if "VERSION" in globals():
    checks.append(("VERSION = 2.5+FIX-K1..K13", VERSION == "2.5+FIX-K1..K13", VERSION))
if "MODEL_LABEL" in globals():
    checks.append(("MODEL_LABEL == V50.6.6", MODEL_LABEL == "V50.6.6", MODEL_LABEL))
if "PROTOCOL_LABEL" in globals():
    checks.append(("PROTOCOL_LABEL == Rev7.4.0", PROTOCOL_LABEL == "P11.13-Rev7.4.0", PROTOCOL_LABEL))
if "K25_EFF_DEFAULT" in globals():
    checks.append(("K25_EFF_DEFAULT == 1.6573e-3", abs(K25_EFF_DEFAULT - 1.6573e-3) < 1e-9, f"{K25_EFF_DEFAULT:.6e}"))
if "pKa_HOCl_at_T" in globals():
    try:
        pka_25 = pKa_HOCl_at_T(25.0); pka_35 = pKa_HOCl_at_T(35.0)
        checks.append(("pKa(T) 25/35", abs(pka_25-7.54)<1e-6 and abs(pka_35-7.04)<1e-6, f"25={pka_25}, 35={pka_35}"))
    except: checks.append(("pKa(T)", False, "exception"))
if "C_CL" in globals():
    checks.append(("C_CL == 5.6 mol/m3", abs(C_CL - 5.6) < 1e-9, f"{C_CL}"))
if "A_E" in globals():
    checks.append(("A_E == 2e-4", abs(A_E - 2.0e-4) < 1e-9, f"{A_E}"))
if "GRUBBS_ALPHA" in globals():
    checks.append(("GRUBBS_ALPHA == 0.05", GRUBBS_ALPHA == 0.05, f"{GRUBBS_ALPHA}"))
if "D_CL" in globals():
    checks.append(("D_CL == 2.03e-9 m2/s", abs(D_CL - 2.03e-9) < 1e-15, f"{D_CL}"))

def _uses(fn, name):
    def walk(co):
        if name in co.co_names: return True
        return any(walk(c) for c in co.co_consts if hasattr(c, "co_code"))
    return walk(fn.__code__)

checks.append(("thermal_model_energy_balance موجودة", callable(thermal_model_energy_balance), "نعم"))
ok_j2 = _uses(_generate_demo_traces, "thermal_model_energy_balance")
checks.append(("FIX-J2 generation unified", ok_j2, "نعم" if ok_j2 else "لا"))
ok_j2b = _uses(_integrate_D_curve, "thermal_model_energy_balance")
checks.append(("FIX-J2 calibration unified", ok_j2b, "نعم" if ok_j2b else "لا"))

for name, ok, val in checks:
    print(f"  {'[OK]' if ok else '[X]'} {name}: {val}")

print(f"\n[5] الملفات المُنتَجة:")
for f in ["runs_demo.csv", "reference_demo_summary.json",
          "analysis_demo_summary.json", "stress_results.json",
          "k25_calibration.txt"]:
    p = Path(f)
    print(f"  {'[OK]' if p.exists() else '[--]'} {f}" + (f" ({p.stat().st_size} bytes)" if p.exists() else ""))

traces_dir = Path("traces_demo")
if traces_dir.exists():
    n = len(list(traces_dir.glob("*.csv")))
    print(f"  [OK] traces_demo/ ({n} ملف)")

print("\n" + "=" * 70)
all_ok = (len(fn_missing) == 0 and len(const_missing) == 0
          and len(dict_missing) == 0 and all(ok for _, ok, _ in checks))
if all_ok:
    print("[OK] الكود v2.5.1 محمّل بالكامل")
else:
    print("[X] مشاكل:")
    if fn_missing: print(f"   - دوال: {fn_missing}")
    if const_missing: print(f"   - ثوابت: {const_missing}")
    failed = [name for name, ok, _ in checks if not ok]
    if failed: print(f"   - فحوص: {failed}")
print("=" * 70)

Cell 2 - التحقق من تحميل الكود (v2.5.1)

[1] الدوال (48):
  -> موجودة: 48/48

[2] الثوابت (54):
  -> موجودة: 54/54

[3] القواميس (1):
  -> موجودة: 1/1

[4] فحوص القيم الحرجة (v2.5.1):
  [OK] VERSION = 2.5+FIX-K1..K13: 2.5+FIX-K1..K13
  [OK] MODEL_LABEL == V50.6.6: V50.6.6
  [OK] PROTOCOL_LABEL == Rev7.4.0: P11.13-Rev7.4.0
  [OK] K25_EFF_DEFAULT == 1.6573e-3: 1.657300e-03
  [OK] pKa(T) 25/35: 25=7.54, 35=7.04
  [OK] C_CL == 5.6 mol/m3: 5.6
  [OK] A_E == 2e-4: 0.0002
  [OK] GRUBBS_ALPHA == 0.05: 0.05
  [OK] D_CL == 2.03e-9 m2/s: 2.03e-09
  [OK] thermal_model_energy_balance موجودة: نعم
  [OK] FIX-J2 generation unified: نعم
  [OK] FIX-J2 calibration unified: نعم

[5] الملفات المُنتَجة:
  [OK] runs_demo.csv (3040 bytes)
  [OK] reference_demo_summary.json (8820 bytes)
  [OK] analysis_demo_summary.json (8820 bytes)
  [OK] stress_results.json (7782 bytes)
  [OK] k25_calibration.txt (140 bytes)
  [OK] traces_demo/ (18 ملف)

[OK] الكود v2.5.1 محمّل بالكامل


In [ ]:
# ====================================================================
# Cell 3 - عرض Stress Tests (v2.5)
# ====================================================================

from pathlib import Path
import json

print("=" * 70)
print("Cell 3 - نتائج اختبارات الإجهاد (v2.5)")
print("=" * 70)

stress_file = Path("stress_results.json")
if not stress_file.exists():
    raise FileNotFoundError("stress_results.json مفقود. شغّل الخلية 1 أولًا.")

data = json.loads(stress_file.read_text(encoding='utf-8'))
summary = data.get('summary', {})

print(f"\nversion  = {data.get('version', '?')}")
print(f"mode     = {data.get('mode', '?')}")
print(f"total    = {summary.get('total', '?')}")
print(f"passed   = {summary.get('passed', '?')}")
print(f"failed   = {summary.get('failed', '?')}")
print(f"check    = {summary.get('check', '?')}")

print(f"\n{'-' * 74}")
print(f"{'#':<4} {'الحالة':<8} {'الاسم':<44} {'نتيجة'}")
print(f"{'-' * 74}")

n_pass = n_warn = n_info = n_fail = 0
for i, r in enumerate(data.get('results', []), 1):
    name = r.get('name', '?')[:42]
    result = r.get('result', '?')
    if '[OK]' in result: mark = '[OK]'; n_pass += 1
    elif '[i]' in result: mark = '[i] '; n_info += 1
    elif '[!]' in result: mark = '[!] '; n_warn += 1
    elif '[X]' in result: mark = '[X] '; n_fail += 1
    else: mark = '[?] '
    print(f"{i:<4} {mark:<8} {name:<44} {result[:25]}")

print(f"{'-' * 74}")
print(f"\nملخّص:")
print(f"  [OK] Pass  : {n_pass}")
print(f"  [i]  Info  : {n_info}")
print(f"  [!]  Warn  : {n_warn}")
print(f"  [X]  Fail  : {n_fail}")

print(f"\n{'=' * 70}")
if n_fail == 0:
    print("[OK] لا فشل - الكود سليم")
else:
    print(f"[X] {n_fail} فشل - يحتاج مراجعة")
print("=" * 70)

print(f"\n[تشخيص 'Check' الوحيد]")
print(f"{'-' * 70}")
checks_found = [r for r in data.get('results', []) if '[i]' in r.get('result', '')]
if checks_found:
    for r in checks_found:
        print(f"  الاسم  : {r.get('name')}")
        print(f"  النتيجة: {r.get('result')}")
        print(f"  ملاحظة : {r.get('note')}")
    print(f"\n  الحالة : 🟠 قيد معلوم (LIM-6)")
else:
    print(f"  [OK] لا Check")

print(f"\n[تحقّق وجود اختبارات FIX-K]")
found = {r.get('name', '') for r in data.get('results', [])}
fix_checks = [
    "K1 delta units", "K2 I_lim segments", "K2 I_lim no plateau", "K2 I_lim V mismatch",
    "K3 template no prefill", "K3 required fields", "K4 demo thermal aligned",
    "K5 thermal REAL no abort", "K6 K25 multi-D", "K7 MC data-driven",
    "K7 elasticities", "K10 missing trace REAL",
    "K11 sigma proportional", "K12 load_cal text rows",
    "K13 C2 consistent", "K13 C2 extra loss flagged", "K13 C2 unavailable",
]
n_ok = 0
for key in fix_checks:
    ok = key in found
    n_ok += ok
    print(f"  {'[OK]' if ok else '[!]'} {key}")
print(f"\n  -> {n_ok}/{len(fix_checks)} اختبار موجود")


In [ ]:
# ====================================================================
# Cell 4 - K25 Calibration (v2.5)
# ====================================================================

from pathlib import Path
import re

print("=" * 70)
print("Cell 4 - K25 Calibration (v2.5)")
print("=" * 70)

cal_file = Path("k25_calibration.txt")
if not cal_file.exists():
    raise FileNotFoundError("k25_calibration.txt مفقود.")

content = cal_file.read_text(encoding='utf-8')
print(f"\n{content}")

K25_opt = None; K25_err = None; converged = None
K25_default = None; bias_pct = None
for line in content.split('\n'):
    if 'K25_opt' in line:
        m = re.search(r'=\s*([\d.eE+-]+)', line)
        if m: K25_opt = float(m.group(1))
    if 'err_opt' in line:
        m = re.search(r'=\s*([\d.eE+-]+)', line)
        if m: K25_err = float(m.group(1))
    if 'converged' in line:
        converged = 'True' in line
    if 'K25_EFF_DEFAULT' in line:
        m = re.search(r'=\s*([\d.eE+-]+)', line)
        if m: K25_default = float(m.group(1))
    if 'bias' in line.lower():
        m = re.search(r'=\s*([\d.eE+-]+)', line)
        if m: bias_pct = float(m.group(1))

print(f"{'-' * 70}")
print(f"[تحليل]")
if K25_opt is not None: print(f"  K25_opt (measured) = {K25_opt:.6e} m/s")
if K25_default is not None: print(f"  K25_EFF_DEFAULT    = {K25_default:.6e} m/s")
if K25_err is not None: print(f"  err_opt            = {K25_err:.6f} K")
print(f"  converged          = {converged}")
if bias_pct is not None:
    print(f"  bias               = {bias_pct:.4f}%")

print(f"\n{'-' * 70}")
print(f"[فحص جودة - FIX-J]")
checks = []
if K25_opt is not None:
    checks.append(("K25 في [1e-4, 1e-2]", 1e-4 <= K25_opt <= 1e-2, f"{K25_opt:.6e}"))
if K25_err is not None:
    checks.append(("err_opt < 0.1 K", K25_err < 0.1, f"{K25_err:.6f}"))
if converged is not None:
    checks.append(("converged == True", converged, str(converged)))
if bias_pct is not None:
    if bias_pct < 1.0:
        checks.append(("FIX-J2 bias < 1%", True, f"{bias_pct:.4f}%"))
    elif bias_pct < 5.0:
        checks.append(("FIX-J2 bias < 5%", True, f"{bias_pct:.4f}% (مقبول)"))
    else:
        checks.append(("FIX-J2 bias < 5%", False, f"{bias_pct:.4f}%"))

for name, ok, val in checks:
    print(f"  {'[OK]' if ok else '[X]'} {name}: {val}")

print(f"\n{'=' * 70}")
if all(ok for _, ok, _ in checks):
    print("[OK] K25 Calibration سليم (v2.5)")
else:
    print("[!] بعض الفحوص فشلت")
print("=" * 70)


In [ ]:
# ====================================================================
# Cell 5 - ملخّص DEMO (v2.5)
# ====================================================================

from pathlib import Path
import json

print("=" * 70)
print("Cell 5 - ملخّص DEMO (v2.5)")
print("=" * 70)

summary_file = Path("reference_demo_summary.json")
if not summary_file.exists():
    raise FileNotFoundError("reference_demo_summary.json مفقود.")

s = json.loads(summary_file.read_text(encoding='utf-8'))

print(f"\n[1] معلومات أساسية:")
print(f"  code_version    = {s.get('code_version')}")
print(f"  model           = {s.get('model')}  {'[OK]' if s.get('model') == MODEL_LABEL else '[X]'}")
print(f"  protocol        = {s.get('protocol')}  {'[OK]' if s.get('protocol') == PROTOCOL_LABEL else '[X]'}")
print(f"  mode            = {s.get('mode')}")
print(f"  reference_type  = {s.get('reference_type')}")

print(f"\n[2] K25 (لا يدخل في x1):")
print(f"  K25_eff_used_m_s = {s.get('K25_eff_used_m_s')}")
print(f"  K25_eff_source   = {s.get('K25_eff_source')}")
print(f"  K25_bias_pct     = {s.get('K25_bias_pct')}")
d = s.get('diagnostics', {})
for f in d.get('K25_fits', []):
    print(f"    {f.get('run')}: K25={f.get('K25')}, err_max={f.get('err_max_K')}, converged={f.get('converged')}")
print(f"  K25_spread_rel   = {d.get('K25_spread_rel')}")

print(f"\n[3] x1:")
x1 = s.get('x1', {})
print(f"  median      = {x1.get('median')}")
print(f"  CI95_bayes  = [{x1.get('CI95_lo')}, {x1.get('CI95_hi')}]")
print(f"  CI95_final  = [{x1.get('CI95_lo_final')}, {x1.get('CI95_hi_final')}]  <- المعتمد")

print(f"\n[3b] فحص اتساق C2 (تحت التيار):")
c2c = s.get('C2_consistency') or {}
if c2c.get('available'):
    for r_ in c2c['runs']:
        print(f"  {r_['run']}: المقاس={r_['obs']:.3f}  المتوقع={r_['pred']:.3f}  النسبة={r_['ratio']:.3f}  z={r_['z']:+.2f}")
    print(f"  mean z = {c2c['mean_z']:+.2f}  -> {'متسق' if c2c['consistent'] else 'غير متسق (فقد/كسب إضافي تحت التيار)'}")
    print(f"  eps_C = {c2c['eps_C']:.3f}   eps_C2(فعّال) = {c2c['eps_C2_effective']:.3f}   x1 البديل = {c2c['x1_if_eps_C2']:.4f} ({c2c['x1_shift_pct']:+.1f}%)")
    print(f"  أصغر عجز قابل للكشف ≈ {c2c['min_detectable_deficit']*100:.0f}% (n={len(c2c['runs'])})")
else:
    print("  [--] غير متاح (يلزم C وC2 بـ dpd_pre مقاس)")

print(f"\n[4] Cross-checks (غير معتمدة للاستشهاد):")
aI = s.get('analysis_I', {})
gum = aI.get('I2_GUM') or {}
if gum and 'error' not in gum:
    print(f"  GUM: u_A={gum.get('u_A'):.4f}, u_B={gum.get('u_B'):.4f}, U={gum.get('U_expanded'):.4f}  CI={gum.get('CI95_GUM')}")
mc = aI.get('I3_MonteCarlo') or {}
if mc and 'error' not in mc:
    print(f"  MC : mean={mc.get('x1_mean'):.4f}, std={mc.get('x1_std'):.4f}, CI=[{mc.get('CI95_lo'):.4f}, {mc.get('CI95_hi'):.4f}]")
sens = aI.get('I5_Sensitivity') or {}
if sens and 'error' not in sens:
    print(f"  مرونات: {sens.get('elasticities')}")

print(f"\n[5] Six Events:")
events = s.get('six_events', {})
e2 = events.get('2_mass_transfer') or {}
print(f"  2_mass_transfer: delta_eff = {e2.get('delta_eff_cm')} cm (مشتق من x1)")
e3 = events.get('3_thermal') or {}
if e3 and 'error' not in e3:
    print(f"  3_thermal: max_abs_err_K = {e3.get('max_abs_err_K'):.6f}")
e7 = events.get('7_I_lim') or {}
if e7 and 'error' not in e7:
    for sg in e7.get('segments', []):
        print(f"    {sg.get('V_nom')} V: I={sg.get('I_mean_mA')} mA (std={sg.get('I_std_mA')}) V_constant={sg.get('V_constant')}")
    print(f"  7_I_lim: plateau={e7.get('plateau')}, I_lim={e7.get('I_lim_mA')} mA, warning={e7.get('warning')}")

print(f"\n[6] Verdict: {s.get('verdict')}")

print(f"\n[7] Diagnostics (كلها تعتمد على البيانات):")
checks = d.get('checks', {})
n_bad = 0
for k, v in checks.items():
    mark = '[OK]' if v is True else ('[--]' if v is None else '[!]')
    if v is False: n_bad += 1
    print(f"  {mark} {k} = {v}")
print(f"\n  limitations_open:")
for lim in d.get('limitations_open', []): print(f"    - {lim}")

print(f"\n{'=' * 70}")
print("[OK] ملخّص DEMO سليم" if n_bad == 0 else f"[!] {n_bad} فحص(وص) فشلت — راجعها")
print("=" * 70)


In [ ]:
# ====================================================================
# Cell 6 - حفظ في Google Drive (v2.5)
# ====================================================================

from pathlib import Path
import shutil

print("=" * 70)
print("Cell 6 - حفظ النتائج في Google Drive (v2.5)")
print("=" * 70)

my_drive = Path("/content/drive/MyDrive")
if not my_drive.exists():
    from google.colab import drive
    drive.mount('/content/drive')
    print("[OK] Drive مثبّت")

dest = my_drive / "EC-CODE_v2.5_FIX-K1..K13"
dest.mkdir(parents=True, exist_ok=True)
print(f"\n[الوجهة]: {dest}")

n_copied = 0
for f in ["runs_demo.csv", "reference_demo_summary.json",
          "analysis_demo_summary.json", "stress_results.json",
          "k25_calibration.txt"]:
    src = Path(f)
    if src.exists():
        shutil.copy(src, dest / f)
        print(f"  [OK] {f} ({src.stat().st_size} bytes)")
        n_copied += 1

src_traces = Path("traces_demo")
if src_traces.exists():
    dst_traces = dest / "traces_demo"
    if dst_traces.exists(): shutil.rmtree(dst_traces)
    shutil.copytree(src_traces, dst_traces)
    n = len(list(dst_traces.glob("*.csv")))
    print(f"  [OK] traces_demo/ ({n} ملف)")
    n_copied += 1

for f in ["runs_template.csv", "calibration_template.csv"]:
    src = Path(f)
    if src.exists():
        shutil.copy(src, dest / f)
        print(f"  [OK] {f}")
        n_copied += 1

src_tt = Path("traces_template")
if src_tt.exists():
    dst_tt = dest / "traces_template"
    if dst_tt.exists(): shutil.rmtree(dst_tt)
    shutil.copytree(src_tt, dst_tt)
    n = len(list(dst_tt.glob("*.csv")))
    print(f"  [OK] traces_template/ ({n} ملف)")
    n_copied += 1

print(f"\n{'=' * 70}")
print(f"[تم نسخ {n_copied} ملف/مجلد]")
print("=" * 70)

Cell 6 - حفظ النتائج في Google Drive (v2.5)


In [ ]:
# ====================================================================
# Cell 7 - قالب runs_template.csv (v2.5)
# ====================================================================

import pandas as pd
from pathlib import Path

print("=" * 70)
print("Cell 7 - قالب runs_template.csv")
print("=" * 70)

template = make_runs_template()          # FIX-K3: بلا قيم مُعبَّأة مسبقًا
out = Path("runs_template.csv")
template.to_csv(out, index=False)
print(f"\n[OK] {out.name} ({len(template)} صف × {template.shape[1]} عمود)")
print("[FIX-K3] dpd_pre وV_L_measured وcharge_C(B/C2) وI_mean_mA وT_b_mean ... فارغة عمدًا.")
print("         كل حقل إلزامي فارغ سيوقف verify_files (الخلية 14) وR-7/C-9 في REAL.")
print("         dilution (A:1, B:3, C:1, C2:5, D:1, E:1) قيم مقترحة — عدّلها إذا خففتَ بغير ذلك.")
print(f"\n{'=' * 70}")

Cell 7 - قالب runs_template.csv

[OK] runs_template.csv (18 صف × 20 عمود)
[FIX-K3] dpd_pre وV_L_measured وcharge_C(B/C2) وI_mean_mA وT_b_mean ... فارغة عمدًا.
         كل حقل إلزامي فارغ سيوقف verify_files (الخلية 14) وR-7/C-9 في REAL.
         dilution (A:1, B:3, C:1, C2:5, D:1, E:1) قيم مقترحة — عدّلها إذا خففتَ بغير ذلك.



In [ ]:
# ====================================================================
# Cell 8 - قالب calibration_template.csv (v2.5)
# ====================================================================

import pandas as pd
from pathlib import Path

print("=" * 70)
print("Cell 8 - قالب calibration_template.csv")
print("=" * 70)

cal = pd.DataFrame({
    "item": ["alpha", "LOD_DPD", "T_RT", "pH_0",
             "V_L_actual", "T_air_avg", "RH_avg", "batch_date"],
    "value": [0.0209, 0.05, 22.0, 7.0, 0.100, 24.8, 51.2, "2026-10-11"],
    "uncertainty": [0.0005, 0.01, 0.5, 0.1, 0.0001, 0.14, 1.8, None],
    "unit": ["per_K", "mg/L", "C", "pH", "L", "C", "%", "YYYY-MM-DD"],
    "note": ["k(T) — استبدلها بقيمتك المقاسة", "LOD — مقاس", "T_ref", "pH البدائي للمحلول (buffer 7)",
             "V_L", "T_air", "RH", "date"],
})

out = Path("calibration_template.csv")
cal.to_csv(out, index=False)
print(f"\n[OK] {out.name} ({len(cal)} بند)")
print("[ملاحظة] K25_eff ليس بندًا هنا: يُعايَر تلقائيًا من تشغيلات D")
for _, row in cal.iterrows():
    print(f"  - {row['item']:15s} = {row['value']}  ±{'—' if pd.isna(row['uncertainty']) else row['uncertainty']}  [{row['unit']}]")
print(f"\n{'=' * 70}")

Cell 8 - قالب calibration_template.csv

[OK] calibration_template.csv (8 بند)
[ملاحظة] K25_eff ليس بندًا هنا: يُعايَر تلقائيًا من تشغيلات D
  - alpha           = 0.0209  ±0.0005  [per_K]
  - LOD_DPD         = 0.05  ±0.01  [mg/L]
  - T_RT            = 22.0  ±0.5  [C]
  - pH_0            = 7.0  ±0.1  [pH]
  - V_L_actual      = 0.1  ±0.0001  [L]
  - T_air_avg       = 24.8  ±0.14  [C]
  - RH_avg          = 51.2  ±1.8  [%]
  - batch_date      = 2026-10-11  ±—  [YYYY-MM-DD]



In [ ]:
# ====================================================================
# Cell 9 - trace templates (v2.5)
# ====================================================================

from pathlib import Path
import numpy as np
import pandas as pd

print("=" * 70)
print("Cell 9 - trace templates (v2.5)")
print("=" * 70)

traces_dir = Path("traces_template")
traces_dir.mkdir(exist_ok=True)
header = "# T_air_actual = 24.8 C\n# RH_actual = 51.2 %\n"

print("\n[1] A_template.csv — baseline")
t_A = np.linspace(-60, 660, 721)
with open(traces_dir / "A_template.csv", "w") as f:
    f.write(header + "# A: baseline.\n")
    pd.DataFrame({"t_s": t_A, "I_mA": np.zeros_like(t_A),
                  "V_V": np.zeros_like(t_A), "T_b_C": np.full_like(t_A, 25.0),
                  "EC_raw_uScm": np.full_like(t_A, 799.6)}).to_csv(f, index=False)
print(f"  [OK] A_template.csv ({len(t_A)} صف)")

print("\n[2] B_template.csv")
t = np.linspace(-60, 660, 721)
I_mA = np.where((t >= 0) & (t <= 600), 50.0, 0.0)
V_V = np.where(I_mA > 0, 12.13, 0.0)
T_b_C = np.where(t < 0, 25.02, np.where(t <= 600, 25.02 + t*0.00087, 25.55))
EC = np.where(I_mA > 0, 799.5 - 0.0047*np.maximum(t, 0), 799.5)
with open(traces_dir / "B_template.csv", "w") as f:
    f.write(header)
    pd.DataFrame({"t_s": t, "I_mA": I_mA, "V_V": V_V,
                  "T_b_C": T_b_C, "EC_raw_uScm": EC}).to_csv(f, index=False)
print(f"  [OK] B_template.csv ({len(t)} صف)")

print("\n[3] C_template.csv — calibration")
t_C = np.linspace(0, 300, 301)
with open(traces_dir / "C_template.csv", "w") as f:
    f.write(header + "# C: calibration.\n")
    pd.DataFrame({"t_s": t_C, "I_mA": np.zeros_like(t_C),
                  "V_V": np.zeros_like(t_C), "T_b_C": np.full_like(t_C, 25.0),
                  "EC_raw_uScm": np.full_like(t_C, 799.6)}).to_csv(f, index=False)
print(f"  [OK] C_template.csv ({len(t_C)} صف)")

print("\n[4] D_template.csv")
t_D = np.linspace(0, 1800, 1801)
T_b_D = 35.0*np.exp(-t_D/1500) + 24.8*(1-np.exp(-t_D/1500))
with open(traces_dir / "D_template.csv", "w") as f:
    f.write(header)
    pd.DataFrame({"t_s": t_D, "I_mA": np.zeros_like(t_D),
                  "V_V": np.zeros_like(t_D), "T_b_C": T_b_D,
                  "EC_raw_uScm": np.full_like(t_D, 799.6)}).to_csv(f, index=False)
print(f"  [OK] D_template.csv ({len(t_D)} صف)")

print("\n[5] E_template.csv — 3 مقاطع")
t_E = np.linspace(-60, 960, 1021)
I_E = np.where((t_E >= 0) & (t_E <= 900), np.nan, 0.0)
V_E = np.where((t_E >= 0) & (t_E < 300), 2.0,
        np.where((t_E >= 300) & (t_E < 600), 3.0,
        np.where((t_E >= 600) & (t_E <= 900), 4.0, 0.0)))
T_E = np.where(t_E < 0, 25.0, 25.0 + np.maximum(t_E, 0)*0.0001)
with open(traces_dir / "E_template.csv", "w") as f:
    f.write(header + "# E: 2V (0-300 s), 3V (300-600 s), 4V (600-900 s). I_lim يُقرأ من آخر 60 ث في كل مقطع (240-300, 540-600, 840-900) + اختبار هضبة.\n")
    pd.DataFrame({"t_s": t_E, "I_mA": I_E, "V_V": V_E, "T_b_C": T_E,
                  "EC_raw_uScm": np.full_like(t_E, 799.6)}).to_csv(f, index=False)
print(f"  [OK] E_template.csv ({len(t_E)} صف)")

print(f"\n{'=' * 70}")
n_files = len(list(traces_dir.glob("*.csv")))
print(f"[{n_files} قوالب في traces_template/]")
print("=" * 70)

Cell 9 - trace templates (v2.5)

[1] A_template.csv — baseline
  [OK] A_template.csv (721 صف)

[2] B_template.csv
  [OK] B_template.csv (721 صف)

[3] C_template.csv — calibration
  [OK] C_template.csv (301 صف)

[4] D_template.csv
  [OK] D_template.csv (1801 صف)

[5] E_template.csv — 3 مقاطع
  [OK] E_template.csv (1021 صف)

[5 قوالب في traces_template/]


In [ ]:
# ====================================================================
# Cell 10 - Checklist قبل المختبر (v2.5.1)
# ====================================================================

from pathlib import Path
import json

print("=" * 70)
print("Cell 10 - Checklist قبل المختبر (v2.5.1)")
print("=" * 70)

print("\n[1] ملفات DEMO:")
files_ok = True
for f in ["runs_demo.csv", "reference_demo_summary.json",
          "analysis_demo_summary.json", "stress_results.json",
          "k25_calibration.txt"]:
    p = Path(f)
    if p.exists(): print(f"  [OK] {f} ({p.stat().st_size} bytes)")
    else: print(f"  [--] {f}"); files_ok = False

traces_demo = Path("traces_demo")
if traces_demo.exists():
    n = len(list(traces_demo.glob("*.csv")))
    print(f"  [OK] traces_demo/ ({n} ملف)")

print("\n[2] قوالب المختبر:")
templates_ok = True
for f in ["runs_template.csv", "calibration_template.csv",
          "traces_template/A_template.csv", "traces_template/B_template.csv",
          "traces_template/C_template.csv", "traces_template/D_template.csv",
          "traces_template/E_template.csv"]:
    p = Path(f)
    if p.exists(): print(f"  [OK] {f}")
    else: print(f"  [--] {f}"); templates_ok = False

print("\n[3] Stress tests:")
stress_ok = False; model_ok = False; protocol_ok = False
stress_file = Path("stress_results.json")
if stress_file.exists():
    data = json.loads(stress_file.read_text(encoding='utf-8'))
    ssum = data.get('summary', {})
    print(f"  Total: {ssum.get('total')}  |  Pass: {ssum.get('passed')}  |  Fail: {ssum.get('failed')}  |  Check: {ssum.get('check')}")
    stress_ok = (ssum.get('failed', 1) == 0)
    print(f"  {'[OK] سليم' if stress_ok else '[X] فيه مشاكل'}")

print("\n[4] DEMO summary:")
ref_file = Path("reference_demo_summary.json")
if ref_file.exists():
    s = json.loads(ref_file.read_text(encoding='utf-8'))
    model_ok = s.get('model') == 'V50.6.6'
    protocol_ok = s.get('protocol') == 'P11.13-Rev7.4.0'
    print(f"  model          = {s.get('model')}  {'[OK]' if model_ok else '[X]'}")
    print(f"  protocol       = {s.get('protocol')}  {'[OK]' if protocol_ok else '[X]'}")
    bias = s.get('K25_bias_pct')
    if bias is not None:
        print(f"  K25_bias_pct   = {bias:.4f}%  {'[OK]' if bias < 1.0 else '[i]'}")

print("\n[5] Diagnostics (تعتمد على البيانات):")
diag_ok = True
if ref_file.exists():
    chk = s.get('diagnostics', {}).get('checks', {})
    for k, v in chk.items():
        mark = '[OK]' if v is True else ('[--]' if v is None else '[!]')
        if v is False: diag_ok = False
        print(f"  {mark} {k} = {v}")

print(f"\n{'=' * 70}")
all_ready = (files_ok and templates_ok and stress_ok and model_ok and protocol_ok and diag_ok)
if all_ready:
    print("[جاهز للمختبر - v2.5.1]")
    print(f"\n   ابدأ بـ A_1 وB_1 وD_1 وE_1 فقط، وتأكد: (1) D يعطي K25 متقاربًا، (2) E يعطي هضبة،")
    print(f"   (3) V_cell وR_cell ضمن المتوقع — ثم أكمل بقية التشغيلات.")
    print(f"   التجارب الكاملة: A_1..A_5، B_1..B_5، C_1..C_2، C2_1..C2_2، D_1..D_3، E_1")
else:
    print("[!] بعض الفحوص فشلت")
print("=" * 70)

Cell 10 - Checklist قبل المختبر (v2.5.1)

[1] ملفات DEMO:
  [OK] runs_demo.csv (3040 bytes)
  [OK] reference_demo_summary.json (8820 bytes)
  [OK] analysis_demo_summary.json (8820 bytes)
  [OK] stress_results.json (7782 bytes)
  [OK] k25_calibration.txt (140 bytes)
  [OK] traces_demo/ (18 ملف)

[2] قوالب المختبر:
  [OK] runs_template.csv
  [OK] calibration_template.csv
  [OK] traces_template/A_template.csv
  [OK] traces_template/B_template.csv
  [OK] traces_template/C_template.csv
  [OK] traces_template/D_template.csv
  [OK] traces_template/E_template.csv

[3] Stress tests:
  Total: 69  |  Pass: 68  |  Fail: 0  |  Check: 1
  [OK] سليم

[4] DEMO summary:
  model          = V50.6.6  [OK]
  protocol       = P11.13-Rev7.4.0  [OK]
  K25_bias_pct   = 0.1439%  [OK]

[5] Diagnostics (تعتمد على البيانات):
  [OK] I_lim_plateau_ok = True
  [OK] K25_converged = True
  [OK] K25_spread_ok = True
  [OK] thermal_check_ok = True
  [OK] recovery_C_present = True
  [OK] all_current_traces_present = True


In [ ]:
# ====================================================================
# Cell 11 - رفع runs.csv
# ====================================================================

from google.colab import files
from pathlib import Path
import pandas as pd
import hashlib

print("=" * 70)
print("Cell 11 - رفع runs.csv")
print("=" * 70)

if not Path("runs_demo.csv").exists():
    raise FileNotFoundError("runs_demo.csv مفقود. شغّل الخلية 1 أولًا.")

print("\n[ارفع runs.csv]:")
uploaded = files.upload()

if "runs.csv" not in uploaded:
    raise FileNotFoundError("لم يتم رفع runs.csv")

df = pd.read_csv("runs.csv")
print(f"\n[OK] runs.csv: {len(df)} صف × {df.shape[1]} عمود")
dist = df["cond"].value_counts().to_dict()
print(f"\nتوزيع الشروط: {dist}")
expected = {"A": 5, "B": 5, "C": 2, "C2": 2, "D": 3, "E": 1}
match = all(dist.get(k, 0) == v for k, v in expected.items())
print(f"\n{'[OK]' if match else '[!]'} التوزيع {'مطابق' if match else 'غير مطابق'}")

h_real = hashlib.md5(Path("runs.csv").read_bytes()).hexdigest()
h_demo = hashlib.md5(Path("runs_demo.csv").read_bytes()).hexdigest()
print(f"\nhash(runs.csv)      = {h_real[:16]}...")
print(f"hash(runs_demo.csv) = {h_demo[:16]}...")
print(f"{'[OK]' if h_real != h_demo else '[X]'} hash مختلف")


In [ ]:
# ====================================================================
# Cell 12 - رفع traces/
# ====================================================================

from google.colab import files
from pathlib import Path

print("=" * 70)
print("Cell 12 - رفع traces/")
print("=" * 70)

traces_dir = Path("traces")
traces_dir.mkdir(exist_ok=True)

print("\n[ارفع ملفات trace]:")
uploaded = files.upload()

for fname, content in uploaded.items():
    (traces_dir / fname).write_bytes(content)

n = len(list(traces_dir.glob("*.csv")))
print(f"\n[OK] {n} ملف في traces/")

print(f"\n[فحص headers]")
n_ok = 0
for tp in sorted(traces_dir.glob("*.csv")):
    content = tp.read_text(encoding='utf-8')[:300]
    ok = "T_air_actual" in content and "RH_actual" in content
    if ok: n_ok += 1
    print(f"  {'[OK]' if ok else '[!]'} {tp.name}")

print(f"\n{'[OK]' if n_ok == n else '[!]'} {n_ok}/{n} ملف له header صحيح")

e_trace = traces_dir / "E_1.csv"
if e_trace.exists():
    print(f"\n[فحص E_1.csv — مقاطع 2/3/4 V]")
    try:
        r = compute_I_lim_segments(str(e_trace))
        if "error" in r:
            print(f"  [X] {r['error']}")
        for sg in r.get("segments", []):
            if "I_mean_mA" in sg:
                print(f"  [i] {sg['V_nom']} V: I = {sg['I_mean_mA']:.3f} mA (std {sg['I_std_mA']:.3f}, n={sg['n']}) V_ok={sg['V_constant']}")
            else:
                print(f"  [!] {sg['V_nom']} V: {sg.get('error')} (n={sg.get('n')})")
        if "plateau" in r:
            print(f"  {'[OK]' if r['plateau'] else '[!]'} plateau = {r['plateau']} (تغيّر آخر مقطعين = {r.get('rel_change_last_two')})")
            print(f"  [i]  I_lim = {r.get('I_lim_mA')} mA   warning = {r.get('warning')}")
    except Exception as e:
        print(f"  [X] خطأ: {e}")


In [ ]:
# ====================================================================
# Cell 13 - رفع calibration.csv
# ====================================================================

from google.colab import files
from pathlib import Path
import pandas as pd

print("=" * 70)
print("Cell 13 - رفع calibration.csv")
print("=" * 70)

print("\n[ارفع calibration.csv]:")
uploaded = files.upload()

if "calibration.csv" not in uploaded:
    raise FileNotFoundError("لم يتم رفع calibration.csv")

df = pd.read_csv("calibration.csv")
print(f"\n[OK] calibration.csv: {len(df)} بند")

required_items = ["alpha", "LOD_DPD", "T_RT"]
missing = [x for x in required_items if x not in df["item"].values]
if missing: print(f"\n[X] بنود مفقودة: {missing}")
else: print(f"\n[OK] كل البنود الإلزامية موجودة")


In [ ]:
# ====================================================================
# خلية مساعدة (بين Cell 13 و Cell 14): تعبئة المتوسطات المحسوبة من الـ trace
# تملأ فقط: I_mean_mA و V_cell_mean و T_b_mean لصف B_1، و T_b_mean لصف A_1.
# لا تلمس أي قياس آخر. V_L_measured و T_air_std و RH_std تبقى من قياسك أنت.
# ====================================================================
import pandas as pd
from pathlib import Path

RUNS = Path("runs.csv")
T_DIRS = [Path("traces"), Path(".")]          # أين رُفعت ملفات الـ traces
WIN = (0.0, 600.0)                            # نافذة مرور التيار في B_1 (ث)

def find_trace(run):
    for d in T_DIRS:
        p = d / f"{run}.csv"
        if p.exists(): return p
    return None

runs = pd.read_csv(RUNS)
for c in ("I_mean_mA", "V_cell_mean", "T_b_mean"):
    runs[c] = pd.to_numeric(runs[c], errors="coerce").astype(float)

def idx_of(run):
    m = runs.index[runs["run"] == run]
    return m[0] if len(m) else None

# ---- B_1: متوسطات داخل نافذة التيار فقط (0..600 ث) ----
p = find_trace("B_1"); i = idx_of("B_1")
if p is None or i is None:
    print("[!] B_1: الـ trace أو الصف غير موجود — لم يتغير شيء")
else:
    d = pd.read_csv(p, comment="#")
    w = d[(d["t_s"] >= WIN[0]) & (d["t_s"] <= WIN[1])]
    if len(w) < 500:
        print(f"[!] B_1: نقاط النافذة {len(w)} فقط (المتوقع ~601) — راجع الـ trace")
    else:
        runs.loc[i, "I_mean_mA"]   = round(float(w["I_mA"].mean()), 4)
        runs.loc[i, "V_cell_mean"] = round(float(w["V_V"].mean()), 4)
        runs.loc[i, "T_b_mean"]    = round(float(w["T_b_C"].mean()), 4)
        print(f"[OK] B_1 (0..600 ث، {len(w)} نقطة): I={runs.loc[i,'I_mean_mA']} mA | "
              f"V={runs.loc[i,'V_cell_mean']} V | T_b={runs.loc[i,'T_b_mean']} C")
        if not 49.5 <= runs.loc[i, "I_mean_mA"] <= 50.5:
            print("    [!] I_mean خارج [49.5, 50.5] — التيار لم يكن ثابتاً عند 50 mA")
        if not 8.0 <= runs.loc[i, "V_cell_mean"] <= 15.0:
            print("    [!] V_cell خارج [8, 15] V — راجع المسافة بين الأقطاب")

# ---- A_1: T_b_mean من كامل الـ trace (لا يوجد تيار) ----
p = find_trace("A_1"); i = idx_of("A_1")
if p is not None and i is not None:
    d = pd.read_csv(p, comment="#")
    runs.loc[i, "T_b_mean"] = round(float(d["T_b_C"].mean()), 4)
    print(f"[OK] A_1: T_b_mean = {runs.loc[i,'T_b_mean']} C ({len(d)} نقطة)")

runs.to_csv(RUNS, index=False)

# ---- تذكير بما يجب أن تكتبه أنت (قياسات لا تُحسب من الـ trace) ----
i = idx_of("B_1")
if i is not None:
    for c in ("V_L_measured", "T_air_std", "RH_std", "ec0", "t0", "ecf", "tf", "dpd", "dilution"):
        v = pd.to_numeric(pd.Series([runs.loc[i, c]]), errors="coerce").iloc[0]
        if pd.isna(v): print(f"[ينقص — اكتبه من قياسك] B_1.{c}")


In [ ]:
# ====================================================================
# Cell 14 - verify_files
# ====================================================================

from pathlib import Path

print("=" * 70)
print("Cell 14 - verify_files")
print("=" * 70)

for name in ["runs.csv", "calibration.csv", "traces"]:
    if not Path(name).exists():
        raise FileNotFoundError(f"{name} مفقود")

res = verify_files(runs_csv="runs.csv", cal_csv="calibration.csv",
                   traces_dir="traces", verbose=True)

print(f"\n{'-' * 70}")
print(f"النتيجة: {res['checks_passed']}/{res['checks_total']}")

if res['issues']:
    print(f"\n[!] مشاكل ({len(res['issues'])}):")
    for issue in res['issues']:
        print(f"  - {issue}")
else:
    print(f"\n[OK] لا مشاكل")

print(f"\n{'=' * 70}")


In [ ]:
# ====================================================================
# Cell 15 - تشغيل REAL
# ====================================================================

from pathlib import Path

print("=" * 70)
print("Cell 15 - تشغيل REAL")
print("=" * 70)

for f in ["runs.csv", "calibration.csv"]:
    if not Path(f).exists(): raise FileNotFoundError(f"{f} مفقود")
if not Path("traces").exists(): raise FileNotFoundError("traces/ مفقود")

MODE = "REAL"
assert_mode_is_valid()
print(f"\n[OK] MODE = {MODE}")
print(f"[!] strict=True")

print(f"\n[تشغيل run_analysis]")
result_real = None
try:
    result_real = run_analysis(
        runs_csv="runs.csv", traces_dir="traces",
        cal_csv="calibration.csv", out="analysis_real_summary.json",
        strict=True, verbose=False,
    )
    print(f"\n[OK] تحليل REAL نجح")
    print(f"  code_version = {result_real.get('code_version')}")
    print(f"  model        = {result_real.get('model')}")
    print(f"  protocol     = {result_real.get('protocol')}")
    print(f"  x1 median    = {result_real.get('x1', {}).get('median')}")
    print(f"  CI95_final   = [{result_real['x1'].get('CI95_lo_final')}, {result_real['x1'].get('CI95_hi_final')}]")
    print(f"  verdict      = {result_real.get('verdict')}")
    print(f"  K25_source   = {result_real.get('K25_eff_source')}")
    print(f"  pH_0_used    = {result_real.get('pH_0_used')}")
    print(f"\n[فحوص]")
    for k, v in result_real['diagnostics'].get('checks', {}).items():
        mark = '[OK]' if v is True else ('[--]' if v is None else '[!]')
        print(f"  {mark} {k} = {v}")
    if result_real['diagnostics'].get('max_abs_err_flag'):
        print(f"\n[!] تحذير حراري (لا يمنع x1): {result_real['diagnostics']['max_abs_err_flag']}")
except ValueError as e:
    msg = str(e)
    print(f"\n[X] فشل التحليل: {msg}")
    if "Acceptance criteria violated" in msg:
        print(f"   STOP: معايير القبول R-1..R-8 انتُهكت — راجع القائمة أعلاه وصحّح القياس أو سجّل السبب")
    elif "traces مفقودة" in msg:
        print(f"   STOP: trace ناقص لتشغيل بتيار — أعد رفعه (لا يُسمح بـ Q اسمي في REAL)")
    elif "x1_median" in msg:
        print(f"   STOP: x1 خارج النطاق الفيزيائي — راجع dpd/Q/التخفيف")
    elif "N-12" in msg or "مطابق" in msg:
        print(f"   STOP: runs.csv مطابق لـ DEMO — تأكد من رفع الملف الحقيقي")
except Exception as e:
    print(f"\n[X] خطأ غير متوقع: {type(e).__name__}: {e}")
    raise


In [ ]:
# ====================================================================
# Cell 16 - تحليل REAL
# ====================================================================

from pathlib import Path
import json

print("=" * 70)
print("Cell 16 - تحليل REAL")
print("=" * 70)

real_file = Path("analysis_real_summary.json")
if not real_file.exists():
    raise FileNotFoundError("analysis_real_summary.json مفقود. شغّل الخلية 15 أولًا.")

s = json.loads(real_file.read_text(encoding='utf-8'))

print(f"\n[1] صلاحية الاستشهاد:")
try:
    assert_not_demo_for_citation(summary=s)
    print(f"  [OK] صالح للاستشهاد (REAL)")
except ValueError as e:
    print(f"  [X] غير صالح: {e}")

print(f"\n[2] معلومات أساسية:")
for k in ["code_version", "model", "protocol", "reference_type", "K25_eff_source", "pH_0_used"]:
    print(f"  {k:<15} = {s.get(k)}")

print(f"\n[3] x1:")
x1 = s.get('x1', {})
print(f"  median      = {x1.get('median')}")
print(f"  CI95_final  = [{x1.get('CI95_lo_final')}, {x1.get('CI95_hi_final')}]  <- المعتمد")
print(f"  chi2_dof    = {x1.get('chi2_dof')}   sigma_x = {x1.get('sigma_x')}")

print(f"\n[3b] فحص اتساق C2 (تحت التيار):")
c2c = s.get('C2_consistency') or {}
if c2c.get('available'):
    for r_ in c2c['runs']:
        print(f"  {r_['run']}: المقاس={r_['obs']:.3f}  المتوقع={r_['pred']:.3f}  النسبة={r_['ratio']:.3f}  z={r_['z']:+.2f}")
    print(f"  mean z = {c2c['mean_z']:+.2f}  -> {'متسق' if c2c['consistent'] else 'غير متسق (فقد/كسب إضافي تحت التيار)'}")
    print(f"  eps_C = {c2c['eps_C']:.3f}   eps_C2(فعّال) = {c2c['eps_C2_effective']:.3f}   x1 البديل = {c2c['x1_if_eps_C2']:.4f} ({c2c['x1_shift_pct']:+.1f}%)")
    print(f"  أصغر عجز قابل للكشف ≈ {c2c['min_detectable_deficit']*100:.0f}% (n={len(c2c['runs'])})")
else:
    print("  [--] غير متاح (يلزم C وC2 بـ dpd_pre مقاس)")

print(f"\n[4] Six Events:")
events = s.get('six_events', {})
e2 = events.get('2_mass_transfer') or {}
print(f"  2_mass_transfer: delta_eff = {e2.get('delta_eff_cm')} cm (مشتق من x1)")
e3 = events.get('3_thermal') or {}
if e3 and 'error' not in e3:
    print(f"  3_thermal: max_abs_err_K = {e3.get('max_abs_err_K')}")
e7 = events.get('7_I_lim') or {}
if e7 and 'error' not in e7:
    for sg in e7.get('segments', []):
        print(f"    {sg.get('V_nom')} V: I = {sg.get('I_mean_mA')} mA")
    print(f"  7_I_lim: plateau = {e7.get('plateau')}, I_lim = {e7.get('I_lim_mA')} mA, warning = {e7.get('warning')}")
elif e7:
    print(f"  7_I_lim: {e7.get('error')}")

print(f"\n[5] Verdict: {s.get('verdict')}")

print(f"\n[6] Diagnostics:")
d = s.get('diagnostics', {})
for k, v in d.get('checks', {}).items():
    mark = '[OK]' if v is True else ('[--]' if v is None else '[!]')
    print(f"  {mark} {k} = {v}")
warns = d.get('validation_warnings', [])
print(f"  validation_warnings: {len(warns)}")
for w in warns[:8]: print(f"    - {w}")

print(f"\n[7] LIMs المفتوحة:")
for lim in d.get('limitations_open', []): print(f"  - {lim}")


In [ ]:
# ====================================================================
# Cell 17 - مقارنة DEMO / REAL (معلومات فقط)
# ====================================================================

from pathlib import Path
import json

print("=" * 70)
print("Cell 17 - مقارنة DEMO / REAL")
print("=" * 70)
print("ملاحظة: DEMO بيانات اصطناعية بـ x1 مُدخَل (0.051). هو للتحقق من الكود، وليس هدفًا للنتيجة.")
print("        لا يُحكم هنا بـ 'تقارب جيد/ضعيف'.")

demo = json.loads(Path("reference_demo_summary.json").read_text(encoding='utf-8'))
real = json.loads(Path("analysis_real_summary.json").read_text(encoding='utf-8'))

print(f"\n{'Field':<28} {'DEMO':<22} {'REAL':<22}")
print(f"{'-' * 72}")
for f in ["code_version", "model", "protocol", "mode", "reference_type",
          "K25_eff_source", "K25_used_in_ode_source", "pH_0_used"]:
    d_val = str(demo.get(f,'?'))[:20]; r_val = str(real.get(f,'?'))[:20]
    print(f"{f:<28} {d_val:<22} {r_val:<22}")

d_x1 = demo.get('x1', {}); r_x1 = real.get('x1', {})
print(f"{'x1_median':<28} {str(d_x1.get('median'))[:20]:<22} {str(r_x1.get('median'))[:20]:<22}")
print(f"{'CI95_final':<28} {str([d_x1.get('CI95_lo_final'), d_x1.get('CI95_hi_final')])[:20]:<22} {str([r_x1.get('CI95_lo_final'), r_x1.get('CI95_hi_final')])[:20]:<22}")
print(f"{'verdict':<28} {str(demo.get('verdict'))[:20]:<22} {str(real.get('verdict'))[:20]:<22}")


In [ ]:
# ====================================================================
# Cell 18 - القرار (v2.5): من verdict البيانات + تحفّظات الجودة
# ====================================================================

from pathlib import Path
import json
import pandas as pd

print("=" * 70)
print("Cell 18 - القرار")
print("=" * 70)

s = json.loads(Path("analysis_real_summary.json").read_text(encoding='utf-8'))
x1 = s['x1']; d = s.get('diagnostics', {}); chk = d.get('checks', {})

try:
    df_runs = pd.read_csv("runs.csv")
    a = s['alpha_per_K']
    A = df_runs[df_runs["cond"] == "A"]
    k0_med = float(pd.Series(A["ec0"].values / (1.0 + a * (A["t0"].values - 25.0))).median()) if len(A) > 0 else None
except Exception:
    k0_med = None

# ---- تحفّظات الجودة (لا تغيّر النتيجة، تُلحَق بها) ----
flags = []
recC = s.get('recovery_C')
eps = (1 - recC['mean']) if recC else None
if eps is None:
    flags.append("لا يوجد استرجاع C: ε من الأولوية الافتراضية")
elif not (0.10 <= eps <= 0.20):
    flags.append(f"ε={eps:.3f} خارج المتوقع [0.10, 0.20]")
drift = s.get('drift_control')
if drift is not None and k0_med:
    if abs(drift) / k0_med >= 0.05:
        flags.append(f"drift_control/k0 = {abs(drift)/k0_med:.3f} >= 5%")
if chk.get('x1_chi2_ok') is False:   flags.append("chi2/dof > 2 (تشتّت زائد بين B)")
c2c = s.get('C2_consistency') or {}
if chk.get('C2_consistent') is False:
    flags.append(f"C2 غير متسق (z={c2c.get('mean_z'):+.1f}): فقد/كسب إضافي تحت التيار؛ x1 البديل بـ ε_C2 = {c2c.get('x1_if_eps_C2'):.4f} ({c2c.get('x1_shift_pct'):+.0f}%)")
elif chk.get('C2_consistent') is None:
    flags.append("C2: لا فحص اتساق (يلزم C وC2 بـ dpd_pre مقاس)")
if chk.get('thermal_check_ok') is False: flags.append("فحص الحرارة فشل (لا يؤثر في x1)")
if chk.get('K25_converged') is False:    flags.append("K25 لم يتقارب في D")
if chk.get('K25_spread_ok') is False:    flags.append("تشتّت K25 بين تشغيلات D > 20%")
if chk.get('I_lim_plateau_ok') is False: flags.append("E: لا هضبة — I_lim غير محدَّد")
if chk.get('I_lim_plateau_ok') is None:  flags.append("E: لا بيانات I_lim")
if chk.get('all_current_traces_present') is False: flags.append("traces ناقصة")
if chk.get('validation_warnings_none') is False:   flags.append("تحذيرات تحقق R-*")

verdict = s.get('verdict')
med = x1.get('median'); lo = x1.get('CI95_lo_final'); hi = x1.get('CI95_hi_final')
if verdict == "pathway_Cl_confirmed":
    DECISION = f"Cl2 مؤكد: x1 = {med:.4f} (CI95 [{lo:.4f}, {hi:.4f}]؛ الحد الأدنى بايزي > 0.005)"
elif verdict == "x1_below_upper95":
    DECISION = "x1 أقل من الحد الأعلى 95% (كل B تحت LOD والاسترجاع جيد)"
elif verdict == "inconclusive_measurement":
    DECISION = "غير حاسم: القياس نفسه غير موثوق (استرجاع C2 منخفض)"
else:
    DECISION = "غير حاسم"
if flags:
    DECISION += f" — مع {len(flags)} تحفّظ"

print(f"\nverdict = {verdict}")
print(f"x1      = {med}  CI95_final = [{lo}, {hi}]")
print(f"\nتحفّظات الجودة ({len(flags)}):")
for f in flags: print(f"  [!] {f}")
if not flags: print("  [OK] لا تحفّظات")

# ---- معلومات فقط (ليست معايير تصديق) ----
print(f"\n[معلومات فقط]")
print(f"  x1 مقابل توقّع مسبق [0.035, 0.065]: {'داخله' if (med is not None and 0.035 <= med <= 0.065) else 'خارجه'} (توقّع وليس معيار)")
e2 = (s.get('six_events', {}) or {}).get('2_mass_transfer') or {}
print(f"  delta_eff = {e2.get('delta_eff_cm')} cm (مشتق من x1، غير مستقل)")
print(f"  K25_eff   = {s.get('K25_eff_used_m_s')} (لا يدخل في x1)")
if c2c.get('available'):
    print(f"  C2: mean z = {c2c['mean_z']:+.2f}؛ x1 البديل (ε من C2) = {c2c['x1_if_eps_C2']:.4f} ({c2c['x1_shift_pct']:+.1f}%)؛ أصغر عجز قابل للكشف ≈ {c2c['min_detectable_deficit']*100:.0f}%")

print(f"\nالقرار النهائي: {DECISION}")


In [ ]:
# ====================================================================
# Cell 19 - CHANGELOG
# ====================================================================

from pathlib import Path
from datetime import datetime

print("=" * 70)
print("Cell 19 - CHANGELOG")
print("=" * 70)

if "DECISION" not in globals():
    raise NameError("DECISION غير محدَّد. شغّل الخلية 18 أولًا.")

changelog = Path("CHANGELOG.md")
if not changelog.exists():
    changelog.write_text("""# CHANGELOG

## شرط رفع التجميد
1. A_1، B_1، D_1 (35°C)، E_1 مكتملة
2. analysis_real_summary.json موجود
3. قرار §7.2 صادر

## السجل
""", encoding='utf-8')

date = datetime.now().strftime("%Y-%m-%d")
with open(changelog, "a", encoding='utf-8') as f:
    f.write(f"\n## [{date}] تشغيل REAL (v2.5)\n\n")
    f.write(f"- [REAL] DECISION = {DECISION}\n")
    f.write(f"- [Code] v{VERSION} (FIX-K1..K13)\n")
    f.write(f"- [Model] {MODEL_LABEL}\n")
    f.write(f"- [Protocol] {PROTOCOL_LABEL}\n")

print(f"\n[OK] تم تسجيل النتائج")


In [ ]:
# ====================================================================
# Cell 20 - ZIP النتائج النهائية
# ====================================================================

from pathlib import Path
import shutil

print("=" * 70)
print("Cell 20 - ZIP النتائج النهائية (v2.5)")
print("=" * 70)

export_dir = Path("EC-CODE_v2.5_FIX-K1..K13_REAL")
if export_dir.exists(): shutil.rmtree(export_dir)
export_dir.mkdir(exist_ok=True)

n_copied = 0
for f in ["runs.csv", "calibration.csv", "analysis_real_summary.json",
          "runs_demo.csv", "reference_demo_summary.json",
          "analysis_demo_summary.json", "stress_results.json",
          "k25_calibration.txt", "runs_template.csv",
          "calibration_template.csv", "CHANGELOG.md"]:
    src = Path(f)
    if src.exists():
        shutil.copy(src, export_dir / f)
        print(f"  [OK] {f}")
        n_copied += 1

for folder in ["traces", "traces_demo", "traces_template"]:
    src_dir = Path(folder)
    if src_dir.exists():
        dst = export_dir / folder
        if dst.exists(): shutil.rmtree(dst)
        shutil.copytree(src_dir, dst)
        n = len(list(dst.glob("*.csv")))
        print(f"  [OK] {folder}/ ({n} ملف)")
        n_copied += 1

zip_path = shutil.make_archive("EC-CODE_v2.5_FIX-K1..K13_REAL", "zip", export_dir)
size_mb = Path(zip_path).stat().st_size / 1024 / 1024
print(f"\n[OK] {zip_path} ({size_mb:.2f} MB)")

try:
    from google.colab import files as cf
    cf.download(zip_path)
    print(f"[OK] تم التنزيل")
except Exception as e:
    print(f"[!] تعذَّر التنزيل: {e}")

print(f"\n{'=' * 70}")
print(f"[اكتمل v2.5]")
print("=" * 70)
